# PyTheraTwin Pipeline — Output Analysis

## Quick Start
1. Set `OUTPUT_TITLE` in the **User Config** cell below and run it.
2. Run **Imports** (cell 1) and **Setup** (cell 2).
3. Run whichever section cells you want. Each visualization cell checks its `SHOW_*` flag — set the flag `True` in Setup to enable that section.

In [ ]:
# ── Edit here, then run this cell ────────────────────────────────────────────
OUTPUT_TITLE = 'paper_dataset'
OUTPUT_ROOT  = '/home/jhubadmin/Theranostic-Virtual-Patient-Pipeline'
CT_DIR       = '/home/jhubadmin/Theranostic-Virtual-Patient-Pipeline/inputs/testing_dataset'

print(f'Target: {OUTPUT_ROOT}/{OUTPUT_TITLE}')


## 1 — Imports and Helpers

In [ ]:
import os, re, json, math, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import SimpleITK as sitk
from scipy.optimize import curve_fit, OptimizeWarning

plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.titleweight': 'bold',
    'axes.labelsize': 16,
    'axes.labelweight': 'bold',
    'figure.titlesize': 20,
    'figure.titleweight': 'bold',
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})


def _strip_comments(text):
    lines = []
    for line in text.splitlines():
        idx = line.find('//')
        lines.append(line[:idx] if idx >= 0 else line)
    return '\n'.join(lines)

def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.loads(_strip_comments(f.read()))

def list_files(directory, suffix):
    if not os.path.isdir(directory):
        return []
    return sorted(os.path.join(directory, f) for f in os.listdir(directory) if f.endswith(suffix))

def stem(path):
    name = os.path.basename(path)
    return name[:-7] if name.endswith('.nii.gz') else os.path.splitext(name)[0]

def fmt_int(v):
    try: return f'{int(v):,}'
    except: return str(v)

def pct(numer, denom, eps=1e-9):
    return (numer - denom) / (abs(denom) + eps) * 100.0

def safe_name(text):
    name = re.sub(r'[^A-Za-z0-9_.-]+', '_', str(text)).strip('_')
    return name or 'plot'

def finish_plot(fig, name, tight=True):
    if tight:
        try: fig.tight_layout()
        except: pass
    _add_title_underlines(fig)
    if globals().get('SAVE_PLOTS', False):
        out_dir = Path(globals().get('PLOT_OUTPUT_DIR', 'notebook_plots'))
        out_dir.mkdir(parents=True, exist_ok=True)
        for ext in globals().get('PLOT_FORMATS', ('png',)):
            out_path = out_dir / f'{safe_name(name)}.{ext}'
            fig.savefig(out_path, dpi=globals().get('PLOT_DPI', 200), bbox_inches='tight')
            if globals().get('PRINT_SAVED_PLOTS', True):
                print('Saved:', out_path)
    if globals().get('SHOW_PLOTS', True):
        plt.show()
    else:
        plt.close(fig)

def plot_spec(section, key, default=None):
    return globals().get('PLOT_SPECS', {}).get(section, {}).get(key, default)

def med_aspect(sp_row_mm, sp_col_mm):
    """Physical aspect ratio for imshow: height-per-pixel / width-per-pixel."""
    return float(sp_row_mm) / float(sp_col_mm)

def optimal_grid(n, max_cols=None):
    """Return (nrows, ncols) minimising empty panels, preferring landscape."""
    if n <= 0: return 1, 1
    ub = min(max_cols, n) if max_cols else n
    best_nc, best_nr, best_score = 1, n, float('inf')
    for nc in range(1, ub + 1):
        nr = math.ceil(n / nc)
        waste = nr * nc - n
        portrait_pen = max(0, nr - nc) * 500  # strongly prefer landscape
        aspect_pen   = abs(nc / nr - 1.0) * 50
        score = waste * 200 + portrait_pen + aspect_pen
        if score < best_score:
            best_score = score
            best_nc, best_nr = nc, nr
    return best_nr, best_nc

def make_grid(n, max_cols=None, pw=5.0, ph=5.0):
    if n <= 0: return None, None, 0, 0
    nr, nc = optimal_grid(n, max_cols=max_cols)
    fig, axes = plt.subplots(nr, nc, figsize=(pw*nc, ph*nr), squeeze=False)
    return fig, axes, nr, nc

def _add_title_underlines(fig):
    """Draw a thin rule beneath every subplot title."""
    try:
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        fw = fig.get_figwidth() * fig.dpi
        fh = fig.get_figheight() * fig.dpi
        for ax in fig.axes:
            t = ax.title
            if not t.get_text(): continue
            bb = t.get_window_extent(renderer=renderer)
            x0, x1 = bb.x0 / fw, bb.x1 / fw
            y = (bb.y0 - 2) / fh
            line = plt.Line2D([x0, x1], [y, y], transform=fig.transFigure,
                              color=t.get_color(), linewidth=0.8, clip_on=False)
            fig.add_artist(line)
    except Exception:
        pass

def fmt_roi_label(roi, patient_idx=None):
    """Display name for an ROI. Synthetic lesions show organ + count from config."""
    if roi.lower().startswith('synthetic_lesion_'):
        suffix = roi.lower()[len('synthetic_lesion_'):]  # e.g. 'tumor1', 'tumorrest'
        confs = globals().get('configs', {})
        if patient_idx is not None and patient_idx in confs:
            specs = confs[patient_idx].get('phase_1', {}).get('synthetic_lesions_stage', {}).get('specs') or {}
            for organ, s in specs.items():
                if s.get('pbpk_label', '').lower() == suffix:
                    n_les = s.get('n_lesions', 1)
                    return f'Synthetic Lesion: Tumor in {organ.title()} × {n_les}'
        num = ''.join(c for c in suffix if c.isdigit())
        return f'Synthetic Lesion: Tumor {num}' if num else 'Synthetic Lesion: Tumor'
    return roi.replace('_', ' ').title()

def fmt_roi_label_with_vol(roi, i, display_label=None):
    """Return 'Label : X.X mL' using stored mask volume if available."""
    base = display_label if display_label is not None else fmt_roi_label(roi, i)
    vol = globals().get('mask_volume_store', {}).get(i, {}).get(
        (roi or '').lower(), {}).get('volume_ml')
    if vol is not None:
        return f'{base} : {vol:.1f} mL'
    return base

def add_colorbar(fig, ax, im, label='', shrink=0.85):
    if globals().get('SHOW_COLORBARS', True):
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, shrink=shrink)
        if label: cb.set_label(label, fontsize=13)
        cb.ax.tick_params(labelsize=12)
        return cb

def select_frame_labels(labels, mode='all', selected=None):
    ls = sorted([str(x) for x in labels], key=lambda k: float(k))
    sel = [] if selected is None else [str(x) for x in selected]
    if not ls: return []
    if mode == 'first': return ls[:1]
    if mode == 'last': return ls[-1:]
    if mode == 'selected':
        want = {str(float(x)).rstrip('0').rstrip('.') for x in sel}
        return [l for l in ls if str(float(l)).rstrip('0').rstrip('.') in want]
    return ls

def normalize_unit(u):
    u = str(u).lower().strip()
    if u not in {'mm','cm'}: raise ValueError(f'Unknown unit {u}')
    return u

def unit_scale_factor(fu, tu):
    fu, tu = normalize_unit(fu), normalize_unit(tu)
    if fu == tu: return 1.0
    if fu == 'mm' and tu == 'cm': return 0.1
    if fu == 'cm' and tu == 'mm': return 10.0
    raise ValueError

def scale_img_units(img, fu, tu):
    f = unit_scale_factor(fu, tu)
    if f == 1.0: return sitk.Image(img)
    out = sitk.Image(img)
    out.SetSpacing((np.array(out.GetSpacing())*f).tolist())
    out.SetOrigin((np.array(out.GetOrigin())*f).tolist())
    return out

def voxel_volume_ml(spacing, unit):
    unit = normalize_unit(unit)
    vol = float(np.prod(np.asarray(spacing, dtype=float)))
    return vol if unit == 'cm' else vol / 1000.0

def comparison_xlim(frame_hr):
    manual = globals().get('COMPARISON_XLIM_HR', 'frame_range')
    if isinstance(manual, (tuple, list)) and len(manual)==2:
        return tuple(float(v) for v in manual)
    if manual == 'full': return None
    fh = np.asarray(frame_hr, dtype=float)
    fh = fh[np.isfinite(fh)]
    if fh.size == 0: return None
    return (0.0, max(1.0, float(fh.max())*1.10))

def nanmean_std_count(values, axis=0):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    count = np.sum(finite, axis=axis)
    summed = np.sum(np.where(finite, arr, 0.0), axis=axis)
    mean = np.divide(summed, count, out=np.full(np.shape(summed), np.nan, dtype=float), where=count>0)
    if axis == 0: centered = arr - mean
    elif axis == 1: centered = arr - mean[:,None]
    else: centered = arr - np.expand_dims(mean, axis=axis)
    ss = np.sum(np.where(finite, centered**2, 0.0), axis=axis)
    std = np.sqrt(np.divide(ss, count-1, out=np.full(np.shape(ss), np.nan, dtype=float), where=count>1))
    return mean, std, count

def nanmedian_iqr(values, axis=0):
    arr = np.asarray(values, dtype=float)
    med = np.nanmedian(arr, axis=axis)
    q1 = np.nanpercentile(arr, 25, axis=axis)
    q3 = np.nanpercentile(arr, 75, axis=axis)
    return med, q3-q1

def plot_mean_std_series(ax, x, mean, std, *, label=None, color=None, marker='o', lw=2.0):
    x, mean, std = np.asarray(x,float), np.asarray(mean,float), np.asarray(std,float)
    mask = np.isfinite(x) & np.isfinite(mean)
    if not np.any(mask): return None
    xp, yp = x[mask], mean[mask]
    sp = np.where(np.isfinite(std[mask]), std[mask], 0.0)
    if xp.size == 1:
        return ax.errorbar(xp, yp, yerr=sp, fmt=marker, ms=7, capsize=4, color=color, label=label)
    (line,) = ax.plot(xp, yp, marker=marker, lw=lw, color=color, label=label)
    bc = color if color is not None else line.get_color()
    ax.fill_between(xp, yp-sp, yp+sp, color=bc, alpha=0.18, linewidth=0)
    return line

def fmt_num(v, width=12, precision=3):
    return f'{v:{width}.{precision}f}' if np.isfinite(v) else f"{'n/a':>{width}s}"

def fmt_pct(v, width=10, precision=1):
    return f'{v:+{width}.{precision}f}%' if np.isfinite(v) else f"{'n/a':>{width}s}"

def fmt_mean_std(mean, std, width=20, precision=3):
    if not np.isfinite(mean): return f"{'n/a':>{width}s}"
    s = f'{std:.{precision}f}' if np.isfinite(std) else 'n/a'
    return f'{mean:.{precision}f}+/-{s}' .rjust(width)

def _resolve_out_folders():
    """Rebuild OUT_FOLDERS from OUTPUT_ROOT/OUTPUT_TITLE if not set."""
    if 'OUT_FOLDERS' in globals(): return globals()['OUT_FOLDERS']
    root = globals().get('OUTPUT_ROOT','')
    title = globals().get('OUTPUT_TITLE','')
    project = os.path.join(root, title)
    def idx(n):
        m = re.search(r'CT_(\d+)', n)
        return int(m.group(1)) if m else 10**9
    return sorted(
        [os.path.join(project, d) for d in os.listdir(project)
         if os.path.isdir(os.path.join(project, d)) and d.startswith('CT_')],
        key=lambda p: idx(os.path.basename(p))
    ) if os.path.isdir(project) else []

def get_slice(arr, axis=1, mode=None):
    """Return a 2D coronal view: 'mid'=centre slice, 'max'=max-intensity projection."""
    if (mode or 'mid').lower() == 'max':
        return np.max(arr, axis=axis)
    return np.take(arr, arr.shape[axis] // 2, axis=axis)

def ct_label(i):
    """Display label for patient i, using actual folder CT number after any patient filter."""
    folders = globals().get('OUT_FOLDERS') or _resolve_out_folders()
    if 0 <= i < len(folders):
        name = os.path.basename(folders[i])
        try: return f"CT {int(name.replace('CT_','',1))}"
        except: return name
    return f'CT {i+1}'

def _iter_display_folders():
    """Yield (i, folder) where i is the index in OUT_FOLDERS — keeps ct_label/configs/stores correct."""
    all_f = _resolve_out_folders()
    ds = globals().get('_display_set', None)
    for i, f in enumerate(all_f):
        if ds is None or ct_index_from_name(os.path.basename(f)) in ds:
            yield i, f

def _resolve_display_folders():
    all_f = _resolve_out_folders()
    ds = globals().get('_display_set', None)
    if ds is None: return all_f
    return [f for f in all_f if ct_index_from_name(os.path.basename(f)) in ds]

def should_display(i):
    ds = globals().get('_display_set', None)
    if ds is None: return True
    all_f = _resolve_out_folders()
    return ct_index_from_name(os.path.basename(all_f[i])) in ds if 0 <= i < len(all_f) else True


def roi_color(roi_key):
    """Consistent hash-based color for an ROI. total_body is always black."""
    import hashlib
    k = str(roi_key).lower()
    if k == 'total_body': return 'black'
    _palette = ['#1f77b4', '#d62728', '#2ca02c', '#9467bd', '#8c564b',
                '#e377c2', '#17becf', '#bcbd22', '#ff7f0e', '#aec7e8']
    h = int(hashlib.md5(k.encode()).hexdigest(), 16)
    return _palette[h % len(_palette)]

print('Imports OK')


## 2 — Setup and Plot Specs
Edit flags here. Set `SHOW_*` to `True` to enable that section's plots.

In [ ]:
# ── Paths (inherit from User Config) ──────────────────────────────────────────
SAVE_PLOTS       = False
PLOT_OUTPUT_DIR  = str(Path(OUTPUT_ROOT) / 'figures' / 'output_analysis' / OUTPUT_TITLE)
PLOT_FORMATS     = ('png',)
PLOT_DPI         = 200
PRINT_SAVED_PLOTS = True
SHOW_PLOTS       = True
SHOW_COLORBARS   = True

PLOT_PATIENTS    = None  # None = all; int or list of CT numbers to include (e.g. 3 or [1,3,5])

# ── Section show/hide flags ────────────────────────────────────────────────────
# inital outputs
SHOW_CT           = True
SHOW_SEGMENTATION = True
SHOW_LESIONS      = True
SHOW_PBPK         = True
SHOW_SIMIND_MAPS  = True
SHOW_SIMIND_PROJ  = True
SHOW_OPENGATE     = True
SHOW_ROI_ACTIVITY = True
SHOW_SPECT        = True
SHOW_DOSE         = True
SHOW_DOSE_SUMMARY = True

#comparison flags
SHOW_MASK_OVERLAY = True
SHOW_COMPARISON   = True
SHOW_FITTING      = True
SHOW_AUC          = True
SHOW_PROFILE      = True
SHOW_RESOLUTION_SCALING = False  # 12d only — relevant when comparing simulation resolutions

# ── Image display ─────────────────────────────────────────────────────────────
CT_WINDOW           = (-200, 200) # HU range for CT display (soft tissue)
SLICE_MODE          = 'mid'       # global: 'mid' = coronal mid-slice | 'max' = max-intensity projection
CT_SLICE_MODE       = None        # per-section override (None → uses SLICE_MODE)
SEG_SLICE_MODE      = None
OPENGATE_SLICE_MODE = None
ROI_ACT_SLICE_MODE  = None
DOSE_SLICE_MODE     = None
SPECT_VIEW_AXIS  = 'coronal'     # 'axial', 'coronal', or 'sagittal' for SPECT recon view
DOSE_SUM_VIEW_AXIS = 'coronal'   # 'axial', 'coronal', or 'sagittal' for 6d dose-sum view
SPECT_SHARED_SCALE = True
SIMIND_ANGLE     = 0              # Projection angle index to display in 6b
SIMIND_FRAME_HR  = None           # Frame time (hr) to display in 6b; None = first available

# ── Colormaps ─────────────────────────────────────────────────────────────────
PLOT_SPECS = {
    'ct':           {'cmap': 'gray'},
    'segmentation': {'cmap': 'tab20'},
    'lesions':      {'cmap': 'hot'},
    'pbpk':         {'colors': ['#1f77b4','#d62728','#2ca02c','#9467bd','#8c564b','#e377c2','#17becf','#bcbd22']},
    'simind':       {'atn_cmap':'gray','activity_cmap':'inferno','projection_cmap':'hot'},
    'opengate':     {'cmap':'plasma'},
    'spect':        {'cmap':'hot','percentile':99.5},
    'activity':     {'cmap':'hot'},
    'dose':         {'cmap':'plasma','percentile':99.0},
    'mask':         {'image_cmap':'hot','contour_color':'cyan'},
    'comparison':   {'pbpk_color':'steelblue','spect_color':'orangered'},
    'profile':      {'cmap':'tab10'},
}

# ── PBPK / SPECT ──────────────────────────────────────────────────────────────
PBPK_XLIM_HR        = 25          # upper x-limit (hr) for ALL PBPK TAC & comparison plots; None = auto
PBPK_Y_SCALE        = 'linear'
SPECT_FRAME_MODE    = 'all'       # 'all', 'first', 'last', or 'selected'
SPECT_SELECTED_FRAMES = []
SPECT_RECON_SPACING_UNIT   = 'mm'
DOSE_UNC_VMAX_PCT          = 10   # uncertainty colour-scale ceiling (%) e.g. 50 → shows 0–50%
SEGMENTATION_SPACING_UNIT  = 'mm'

# ── Comparison controls ───────────────────────────────────────────────────────
COMPARE_ONLY_SIMULATED_ROIS       = True
INCLUDE_REMAINING_BODY_IN_COMPARISON = True
SKIP_ROIS_COMPARISON              = {}
SKIP_LAST_FRAME                   = False
COMPARISON_XLIM_HR                = 'frame_range'
COMPARISON_EXCLUDE_ROIS           = []    # e.g. ['liver'] — exclude from 9a/9b
COMPARISON_EXCLUDE_FRAMES         = []    # e.g. [168] — exclude frame times (hr)
COMPARISON_ERROR_LIMIT_PCT        = 100  # clip y-axis of % error; e.g. 150 = ±150%
FIT_MAX_PBPK_POINTS               = 2000
FITTING_XSCALE                    = 'linear'  # 'log' or 'linear'
FIT_XLIM_FACTOR                   = 1.15  # x-axis = max_spect_frame * this factor

# ── Profiling ─────────────────────────────────────────────────────────────────
PROFILE_PHASE = ''
PARAM_LABEL   = 'Patient'
STAGE_ORDER   = ['segmentation','synthetic_lesions','pbpk_tac','simind','opengate','spect_postprocess','dosemap_postprocess']

print('Setup loaded')
print(f'  OUTPUT_TITLE = {OUTPUT_TITLE}')
print(f'  OUTPUT_ROOT  = {OUTPUT_ROOT}')


## 3 — Discover Output Folders and Stage Inventory

In [ ]:
PROJECT_DIR = os.path.join(OUTPUT_ROOT, OUTPUT_TITLE)

def ct_index_from_name(name):
    try: return int(name.replace('CT_','',1))
    except: return 10**9

def discover_ct_inputs(ct_dir):
    ct_inputs = []
    if os.path.isdir(ct_dir):
        for entry in sorted(os.listdir(ct_dir)):
            p = os.path.join(ct_dir, entry)
            if os.path.isdir(p) or entry.endswith(('.nii','.nii.gz')):
                ct_inputs.append(p)
    return ct_inputs

def recorded_ct_for_folder(folder):
    meta_path = os.path.join(folder, 'pipeline_metadata', 'ct_input.json')
    if not os.path.exists(meta_path): return None
    try:
        meta = load_json(meta_path)
        identity = meta.get('ct_identity', {})
        return identity.get('current_input_path') or identity.get('saved_copy_path')
    except: return None

folder_names = sorted(
    [d for d in (os.listdir(PROJECT_DIR) if os.path.isdir(PROJECT_DIR) else [])
     if os.path.isdir(os.path.join(PROJECT_DIR, d)) and d.startswith('CT_')],
    key=lambda n: (ct_index_from_name(n), n),
)
OUT_FOLDERS = [os.path.join(PROJECT_DIR, d) for d in folder_names]

# ── PLOT_PATIENTS filter (limits per-patient plots; cross-patient summaries always use all) ─
_pt_raw = globals().get('PLOT_PATIENTS', None)
_display_set = None
if _pt_raw is not None:
    _pt_set = {_pt_raw} if isinstance(_pt_raw, int) else set(_pt_raw)
    _all_nums = {ct_index_from_name(os.path.basename(f)) for f in OUT_FOLDERS}
    for p in sorted(_pt_set - _all_nums):
        print(f'PLOT_PATIENTS: CT {p} — N/A (not found in output folder)')
    _display_set = _pt_set & _all_nums  # CT numbers to show in per-patient plots
    print(f'PLOT_PATIENTS filter: showing individual plots for CT(s) {sorted(_display_set)}')

N = len(OUT_FOLDERS)
ct_inputs = discover_ct_inputs(CT_DIR)

if N == 0:
    print(f'No CT_* folders found in {PROJECT_DIR}')
else:
    print(f'Found {N} output folder(s):')
    for i, f in enumerate(OUT_FOLDERS):
        recorded_ct = recorded_ct_for_folder(f)
        fallback_ct = ct_inputs[i] if i < len(ct_inputs) else None
        ct_str = recorded_ct or fallback_ct or '(unknown)'
        source = 'metadata' if recorded_ct else 'input-order fallback'
        print(f'  {ct_label(i)}: {os.path.basename(f)} <- {os.path.basename(str(ct_str))} [{source}]')


In [ ]:
EXPECTED_OUTPUTS = [
    ('Input CT',         ['digital_twin/ct.nii.gz']),
    ('Segmentation',     ['digital_twin/digital_twin.nii.gz']),
    ('Synthetic lesions',['digital_twin/synthetic_lesions_stage/synthetic_lesions_all_lesions_labels.nii.gz']),
    ('PBPK TAC',         ['digital_twin/pbpk_tac_stage/pbpk_tacs.npz']),
    ('SIMIND',           ['simulations/simind_tot_w1.a00']),
    ('OpenGATE',         ['pipeline_metadata/opengate_simulation_stage.json']),
    ('SPECT recon',      ['post_processing/reconstructed_SPECT_*.nii.gz']),
    ('Dose map',         ['post_processing/dosemap_postprocess_total_dose.nii.gz']),
    ('Profiling',        ['profiling_CT_*.json']),
]

def pattern_exists(folder, pattern):
    return bool(list(Path(folder).glob(pattern)))

inventory_store = {}
for i, folder in enumerate(OUT_FOLDERS):
    rows = []
    for stage, patterns in EXPECTED_OUTPUTS:
        hits = [p for p in patterns if pattern_exists(folder, p)]
        rows.append({'stage': stage, 'present': bool(hits), 'missing': [p for p in patterns if p not in hits]})
    inventory_store[i] = rows
    print(f'\n{ct_label(i)}: {os.path.basename(folder)}')
    for row in rows:
        icon = '✓' if row['present'] else '✗'
        print(f'  {icon} {row["stage"]}')


## 4 — Configuration Summary

In [ ]:
configs = {}
for i, folder in enumerate(OUT_FOLDERS):
    cfg_path = os.path.join(folder, 'config.json')
    if os.path.exists(cfg_path):
        configs[i] = load_json(cfg_path)

    print('='*72)
    print(f'  {ct_label(i)}: {os.path.basename(folder)}')
    print('='*72)
    if i not in configs:
        print('  config.json not found'); continue

    cfg = configs[i]
    p1 = cfg.get('phase_1', {}); p2 = cfg.get('phase_2', {}); p3 = cfg.get('phase_3', {})
    seg = p1.get('segmentation_stage',{}); les = p1.get('synthetic_lesions_stage',{})
    pbpk = p1.get('pbpk_tac_stage',{}); sim = p2.get('simind_stage',{})
    og = p2.get('opengate_stage',{}); spect = p3.get('spect_postprocess_stage',{})
    dose = p3.get('dosemap_postprocess_stage',{})

    print('\n  Phase 1: Digital twin')
    print(f'    ROIs: {seg.get("roi_subset",[])}')
    specs = les.get('specs',{})
    if specs:
        for organ, s in specs.items():
            print(f'    Lesions: {organ} — {s.get("n_lesions",0)} lesion(s), radii={s.get("radii_mm")} mm')
    else:
        print('    Lesions: none')
    print(f'    PBPK: model={pbpk.get("model_type","-")}  isotope={pbpk.get("isotope","-")}')

    print('\n  Phase 2: Simulations')
    print(f'    SIMIND: isotope={sim.get("isotope","-")}  collimator={sim.get("Collimator","-")}')
    print(f'      photons={fmt_int(sim.get("NumPhotons","-"))}  projections={sim.get("NumProjections","-")}')
    print(f'    OpenGATE: histories={fmt_int(og.get("gate",{}).get("total_histories","-"))}')

    print('\n  Phase 3: Post-processing')
    ft = sorted(spect.get('FrameStartTimes',[]))
    fd = spect.get('FrameDurations',[])
    ft_hr = [f'{t/60:.1f}h' for t in ft]
    print(f'    SPECT frames: {len(ft)} — {ft_hr}')
    print(f'    Reconstruction: {spect.get("ReconstructionAlgorithm","-")}  iter={spect.get("Iterations","-")}  subsets={spect.get("Subsets","-")}')
print()


## 5 — Phase 1: Digital Twin Outputs

### 5a — Input CT

In [ ]:
if not globals().get('SHOW_CT', False):
    print('5a CT skipped — set SHOW_CT = True in Setup')
else:
    for i, folder in _iter_display_folders():
        ct_path = os.path.join(folder, 'digital_twin', 'ct.nii.gz')
        if not os.path.exists(ct_path):
            print(f'{ct_label(i)}: ct.nii.gz not found'); continue

        img = sitk.ReadImage(ct_path)
        arr = sitk.GetArrayFromImage(img)[::-1]   # z,y,x head-up
        sp  = img.GetSpacing()                     # x,y,z mm
        sp_x, sp_y, sp_z = sp
        z, y, x = arr.shape
        print(f'{ct_label(i)}  shape=({z},{y},{x})  spacing=({sp_x:.2f},{sp_y:.2f},{sp_z:.2f}) mm')

        vmin, vmax_ct = globals().get('CT_WINDOW', (-200, 200))

        _mode = globals().get('CT_SLICE_MODE') or globals().get('SLICE_MODE', 'mid')
        sl = get_slice(arr, axis=1, mode=_mode)
        asp = med_aspect(sp_z, sp_x)
        phys_w = x * sp_x
        phys_h = z * sp_z
        fig_w = 7
        fig_h = max(3.0, fig_w * phys_h / phys_w)

        fig, ax = plt.subplots(figsize=(fig_w, fig_w))
        fig.patch.set_facecolor('black')
        ax.set_facecolor('black')
        ax.imshow(sl, cmap='gray', vmin=vmin, vmax=vmax_ct, interpolation='bilinear', aspect=asp)
        ax.axis('off')
        ax.set_title(f'{ct_label(i)}', color='white', fontsize=20, pad=10)
        finish_plot(fig, f'ct_{i+1}_input_ct', tight=False)


### 5b — PyTheraTwin Segmentation

In [ ]:
if not globals().get('SHOW_SEGMENTATION', False):
    print('5b Segmentation skipped — set SHOW_SEGMENTATION = True in Setup')
else:
    roi_map_path = str(Path(OUTPUT_ROOT) / 'src' / 'data' / 'pipeline_roi_naming_map.json')
    label_to_roi = {}
    if os.path.exists(roi_map_path):
        roi_map = load_json(roi_map_path)
        label_to_roi = {int(k): v.lower() for k,v in roi_map.get('PyTheraTwin_Pipeline',{}).items()}

    for i, folder in _iter_display_folders():
        ct_path  = os.path.join(folder, 'digital_twin', 'ct.nii.gz')
        seg_path = os.path.join(folder, 'digital_twin', 'digital_twin.nii.gz')
        if not os.path.exists(seg_path):
            print(f'{ct_label(i)}: digital_twin.nii.gz not found'); continue

        seg_img = sitk.ReadImage(seg_path)
        seg     = sitk.GetArrayFromImage(seg_img)[::-1]
        sp      = seg_img.GetSpacing()
        sp_x, sp_y, sp_z = sp
        labels  = [int(v) for v in np.unique(seg) if v != 0]
        names   = [label_to_roi.get(l, str(l)) for l in labels]
        print(f'{ct_label(i)}  labels: {dict(zip(labels, names))}')

        z, y, x = seg.shape
        _mode = globals().get('SEG_SLICE_MODE') or globals().get('SLICE_MODE', 'mid')
        sl_seg = get_slice(seg, axis=1, mode=_mode)
        asp = med_aspect(sp_z, sp_x)

        # CT background if available
        ct_sl = None
        if os.path.exists(ct_path):
            ct_img = sitk.ReadImage(ct_path)
            ct_arr = sitk.GetArrayFromImage(ct_img)[::-1]
            ct_arr_r = sitk.GetArrayFromImage(
                sitk.Resample(ct_img, seg_img, sitk.Transform(), sitk.sitkLinear, 0.0)
            )[::-1]
            ct_sl = get_slice(ct_arr_r, axis=1, mode=_mode)

        phys_h = z * sp_z; phys_w = x * sp_x
        fig_w = 10; fig_h = max(3.0, fig_w * phys_h / phys_w)
        ncols = 2 if ct_sl is not None else 1
        fig, axes = plt.subplots(1, ncols, figsize=(5*ncols, 5), squeeze=False)
        fig.patch.set_facecolor('black')
        fig.suptitle(f'{ct_label(i)} — Segmentation', color='white')

        vmin_ct, vmax_ct = globals().get('CT_WINDOW', (-200, 200))
        n_labels = max(20, int(seg.max()))
        cmap_seg = plt.get_cmap(plot_spec('segmentation','cmap','tab20'))

        if ct_sl is not None:
            axes[0][0].set_facecolor('black')
            axes[0][0].imshow(ct_sl, cmap='gray', vmin=vmin_ct, vmax=vmax_ct, interpolation='bilinear', aspect=asp)
            axes[0][0].axis('off')
            axes[0][0].set_title('CT', color='white')
            ax_ov = axes[0][1]
        else:
            ax_ov = axes[0][0]

        if ct_sl is not None:
            ax_ov.imshow(ct_sl, cmap='gray', vmin=vmin_ct, vmax=vmax_ct, interpolation='bilinear', aspect=asp)
        seg_rgba = cmap_seg(sl_seg.astype(float) / n_labels)
        seg_rgba[..., 3] = np.where(sl_seg > 0, 0.65, 0.0)
        ax_ov.imshow(seg_rgba, aspect=asp, interpolation='nearest')
        ax_ov.set_facecolor('black')
        ax_ov.axis('off')
        ax_ov.set_title('CT + Segmentation', color='white')

        fig.tight_layout(rect=[0, 0, 1, 0.90])
        finish_plot(fig, f'ct_{i+1}_segmentation', tight=False)


### 5c — Synthetic Lesions

In [ ]:
if not globals().get('SHOW_LESIONS', False):
    print('5c Lesions skipped — set SHOW_LESIONS = True in Setup')
else:
    for i, folder in _iter_display_folders():
        lesion_path = os.path.join(folder, 'digital_twin', 'synthetic_lesions_stage',
                                   'synthetic_lesions_all_lesions_labels.nii.gz')
        if not os.path.exists(lesion_path):
            print(f'{ct_label(i)}: no synthetic lesions'); continue

        les_img = sitk.ReadImage(lesion_path)
        les_arr = sitk.GetArrayFromImage(les_img)[::-1]
        sp      = les_img.GetSpacing()
        sp_x, sp_y, sp_z = sp
        n_lesions = int(np.max(les_arr))
        z, y, x = les_arr.shape
        print(f'{ct_label(i)}  {n_lesions} lesion(s)  shape=({z},{y},{x})')

        asp = med_aspect(sp_z, sp_x)
        phys_h = z * sp_z; phys_w = x * sp_x
        fig_w = 7; fig_h = max(3.0, fig_w * phys_h / phys_w)
        fig, ax = plt.subplots(figsize=(fig_w, fig_w))
        fig.patch.set_facecolor('black')
        ax.set_facecolor('black')

        # CT mid slice background
        ct_path = os.path.join(folder, 'digital_twin', 'ct.nii.gz')
        if os.path.exists(ct_path):
            ct_img = sitk.ReadImage(ct_path)
            ct_r = sitk.GetArrayFromImage(
                sitk.Resample(ct_img, les_img, sitk.Transform(), sitk.sitkLinear, 0.0)
            )[::-1]
            vmin_ct, vmax_ct = globals().get('CT_WINDOW', (-200, 200))
            ax.imshow(ct_r[:, y//2, :], cmap='gray', vmin=vmin_ct, vmax=vmax_ct,
                      interpolation='bilinear', aspect=asp)

        # Lesion max projection overlay (hot, alpha)
        les_mip = np.max(les_arr, axis=1).astype(float)   # coronal max proj: shape (z, x)
        les_mip_norm = np.where(les_mip > 0, les_mip / max(n_lesions, 1), np.nan)
        cmap_les = plt.get_cmap(plot_spec('lesions', 'cmap', 'hot')).copy()
        cmap_les.set_bad(alpha=0)
        ax.imshow(les_mip_norm, cmap=cmap_les, vmin=0, vmax=5, alpha=0.85,
                  interpolation='nearest', aspect=asp)
        ax.axis('off')
        ax.set_title(f'{ct_label(i)} — Synthetic Lesions ({n_lesions})', color='white', fontsize=18, pad=10)
        finish_plot(fig, f'ct_{i+1}_synthetic_lesions', tight=False)


### 5d — PBPK Time-Activity Curves

In [ ]:
pbpk_store = {}
pbpk_meta_store = {}

if not globals().get('SHOW_PBPK', False):
    print('5d PBPK skipped — set SHOW_PBPK = True in Setup')
else:
    _folders = _resolve_out_folders()
    _cfgs = globals().get('configs', {})
    colors_pbpk = plot_spec('pbpk','colors',
        ['#1f77b4','#d62728','#2ca02c','#9467bd','#8c564b','#e377c2','#17becf','#bcbd22'])

    for i, folder in enumerate(_folders):
        stage_dir = os.path.join(folder, 'digital_twin', 'pbpk_tac_stage')
        npz_path  = os.path.join(stage_dir, 'pbpk_tacs.npz')
        json_path = os.path.join(stage_dir, 'pbpk_tacs.json')
        if not os.path.exists(npz_path):
            print(f'{ct_label(i)}: pbpk_tacs.npz not found'); continue

        data     = np.load(npz_path, allow_pickle=True)
        time_min = data['time'].astype(float)
        time_hr  = time_min / 60.0
        tac_rois = {k.replace('tac_',''):data[k].astype(float) for k in data.files if k.startswith('tac_')}
        pbpk_store[i] = {'time_hr': time_hr, **tac_rois}
        meta = load_json(json_path) if os.path.exists(json_path) else {}
        pbpk_meta_store[i] = meta

        if not should_display(i): continue  # data loaded; skip plot

        _xlim_raw = globals().get('PBPK_XLIM_HR', 25)
        if isinstance(_xlim_raw, (list, tuple)):
            xlim = (float(_xlim_raw[0]), float(_xlim_raw[1]))
        elif _xlim_raw is not None:
            xlim = (0.0, float(_xlim_raw))
        else:
            xlim = None
        mask = (time_hr >= xlim[0]) & (time_hr <= xlim[1]) if xlim else np.ones(len(time_hr), dtype=bool)

        # Filter to ROIs used in SIMIND simulations (+ any synthetic lesions)
        _sim_rois_cfg = set(r.lower() for r in
            _cfgs.get(i, {}).get('phase_2', {}).get('simind_stage', {}).get('roi_subset', []))
        sim_rois = [r for r in tac_rois
                    if r.lower() in _sim_rois_cfg or r.lower().startswith('synthetic_lesion_')]
        if not sim_rois:
            sim_rois = list(tac_rois.keys())  # fallback: all
        # Always include remaining_body (once, sorted last)
        if 'remaining_body' in tac_rois and 'remaining_body' not in [r.lower() for r in sim_rois]:
            sim_rois.append('remaining_body')
        sim_rois = sorted(sim_rois, key=lambda r: (r.lower() == 'remaining_body', r))

        fig, ax = plt.subplots(figsize=(8, 8))
        for j, roi in enumerate(sim_rois):
            col = colors_pbpk[j % len(colors_pbpk)]
            ax.plot(time_hr[mask], tac_rois[roi][mask], lw=5, color=col,
                    label=fmt_roi_label(roi, i))
        if xlim: ax.set_xlim(*xlim)
        ax.set_yscale(globals().get('PBPK_Y_SCALE', 'linear'))
        ax.set_xlabel('Time (hr)')
        ax.set_ylabel('Activity (MBq)')
        ax.set_title(f'{ct_label(i)} — PBPK TAC')
        ax.legend(fontsize=14, framealpha=0.9, loc='upper right', ncol=2, handlelength=2.5)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        finish_plot(fig, f'ct_{i+1}_pbpk_tac')

# Always load PBPK data even if not plotting (needed for comparisons)
if not pbpk_store:
    _folders = _resolve_out_folders()
    for i, folder in enumerate(_folders):
        npz_path = os.path.join(folder, 'digital_twin', 'pbpk_tac_stage', 'pbpk_tacs.npz')
        json_path = os.path.join(folder, 'digital_twin', 'pbpk_tac_stage', 'pbpk_tacs.json')
        if not os.path.exists(npz_path): continue
        data = np.load(npz_path, allow_pickle=True)
        time_hr = data['time'].astype(float) / 60.0
        tac_rois = {k.replace('tac_',''):data[k].astype(float) for k in data.files if k.startswith('tac_')}
        pbpk_store[i] = {'time_hr': time_hr, **tac_rois}
        pbpk_meta_store[i] = load_json(json_path) if os.path.exists(json_path) else {}
    if pbpk_store:
        print(f'PBPK data loaded for {len(pbpk_store)} patient(s) (not plotted)')

# ── Bug 1 fix ────────────────────────────────────────────────────────────────
# pbpk_tacs.npz stores one TAC per PyTheraTwin ROI label, but most labels have no
# dedicated PBPK compartment and simply fall back to a verbatim copy of the 'Rest'
# TAC (== pbpk_store[i]['remaining_body']). Summing every key double/triple counts
# Rest. Detect the aliases programmatically (exact match to remaining_body) instead
# of hard-coding the organ list, so this stays correct if the ROI map changes.
def _detect_pbpk_compartments(store):
    out = {}
    for i, d in store.items():
        rb = d.get('remaining_body')
        if rb is None:
            out[i] = sorted(k for k in d if k != 'time_hr')
            continue
        rb = np.asarray(rb, dtype=float)
        distinct = [k for k, v in d.items()
                    if k not in ('time_hr', 'remaining_body') and not np.allclose(np.asarray(v, dtype=float), rb)]
        distinct.append('remaining_body')
        out[i] = sorted(distinct)
    return out

PBPK_COMPARTMENTS = _detect_pbpk_compartments(pbpk_store)

def pbpk_total(i, t):
    """Prescribed whole-body activity (MBq) at time t (scalar or array-like)."""
    return sum(np.interp(t, pbpk_store[i]['time_hr'], pbpk_store[i][k])
               for k in PBPK_COMPARTMENTS[i])

for i in sorted(pbpk_store.keys()):
    _tot1hr = pbpk_total(i, 1.0)
    assert 3000 <= _tot1hr <= 6000, (
        f'{ct_label(i)}: pbpk_total(1hr) = {_tot1hr:.1f} MBq is outside the expected '
        f'3000-6000 MBq range — investigate before trusting downstream comparisons.')

if pbpk_store:
    _n_raw  = np.mean([len(pbpk_store[i]) - 1 for i in pbpk_store])
    _n_dist = np.mean([len(v) for v in PBPK_COMPARTMENTS.values()])
    print(f'PBPK_COMPARTMENTS: {len(pbpk_store)} patient(s) — '
          f'{_n_raw:.1f} raw keys/patient collapsed to {_n_dist:.1f} distinct compartments/patient (avg)')
    print(f'Mean pbpk_total(1hr) across patients: '
          f'{np.mean([pbpk_total(i, 1.0) for i in pbpk_store]):.1f} MBq')


## 6 — Phase 2: Simulation Outputs

### 6a — SIMIND Attenuation and Activity Maps

In [ ]:
simind_meta_store = {}

if not globals().get('SHOW_SIMIND_MAPS', False):
    print('6a SIMIND maps skipped — set SHOW_SIMIND_MAPS = True in Setup')
else:
    _folders = _resolve_out_folders()
    for i, folder in enumerate(_folders):
        meta_path = os.path.join(folder, 'pipeline_metadata', 'simind_simulation_stage.json')
        if not os.path.exists(meta_path):
            print(f'{ct_label(i)}: SIMIND metadata not found'); continue
        m = load_json(meta_path)
        simind_meta_store[i] = m
        if not should_display(i): continue

        prefix = m.get('file_prefix', 'simind')
        roi_list = m.get('roi_list', [])
        geom = m.get('geometry', {})
        prep_dir = m.get('preprocess_dir', '')
        work_dir = m.get('work_dir', '')
        spacing = m.get('arr_px_spacing_cm', [0,0,0])
        shape_new = [int(v) for v in m.get('arr_shape_new', [0,0,0])]
        z_dim, y_dim, x_dim = shape_new if len(shape_new)==3 else (0,0,0)

        print(f'{ct_label(i)}  prefix={prefix}  ROIs={roi_list}')

        atn_flat = None
        for d in [prep_dir, work_dir]:
            p = os.path.join(d, f'{prefix}_atn_av.bin')
            if os.path.exists(p):
                atn_flat = np.fromfile(p, dtype=np.float32); break

        act_maps = {}
        act_paths = m.get('binary_roi_act_map_paths', {})
        for roi in roi_list:
            p = act_paths.get(roi, os.path.join(prep_dir, f'{prefix}_{roi}_act_av.bin'))
            if os.path.exists(p) and z_dim and y_dim and x_dim:
                flat = np.fromfile(p, dtype=np.float32)
                if flat.size == z_dim*y_dim*x_dim:
                    act_maps[roi] = flat.reshape(z_dim, y_dim, x_dim)[::-1]

        n_plots = (1 if atn_flat is not None else 0) + len(act_maps)
        if n_plots == 0:
            print(f'  {ct_label(i)}: no SIMIND binary maps found'); continue

        fig, axes, nrows, ncols = make_grid(n_plots, max_cols=4, pw=5.0, ph=5.0)
        fig.suptitle(f'{ct_label(i)} — SIMIND Maps (coronal max projection)')
        idx = 0

        if atn_flat is not None and z_dim and y_dim and x_dim and atn_flat.size==z_dim*y_dim*x_dim:
            atn = atn_flat.reshape(z_dim, y_dim, x_dim)[::-1]
            ax = axes[idx//ncols][idx%ncols]
            im = ax.imshow(np.max(atn,axis=1), cmap=plot_spec('simind','atn_cmap','gray'), aspect='auto', interpolation='bilinear')
            ax.set_title('Attenuation'); ax.axis('off')
            add_colorbar(fig, ax, im, 'cm⁻¹'); idx += 1

        for roi, arr in act_maps.items():
            ax = axes[idx//ncols][idx%ncols]
            im = ax.imshow(np.max(arr,axis=1), cmap=plot_spec('simind','activity_cmap','inferno'), aspect='auto', interpolation='bilinear')
            ax.set_title(roi.replace('_',' ')); ax.axis('off')
            add_colorbar(fig, ax, im, 'MBq/mL'); idx += 1

        for j in range(idx, nrows*ncols):
            axes[j//ncols][j%ncols].set_visible(False)
        finish_plot(fig, f'ct_{i+1}_simind_maps')

# Always load SIMIND metadata for comparisons
if not simind_meta_store:
    _folders = _resolve_out_folders()
    for i, folder in enumerate(_folders):
        p = os.path.join(folder, 'pipeline_metadata', 'simind_simulation_stage.json')
        if os.path.exists(p):
            simind_meta_store[i] = load_json(p)
    if simind_meta_store:
        print(f'SIMIND metadata loaded for {len(simind_meta_store)} patient(s)')


### 6b — SIMIND Projection Data

In [ ]:
if not globals().get('SHOW_SIMIND_PROJ', False):
    print('6b SIMIND projections skipped — set SHOW_SIMIND_PROJ = True in Setup')
else:
    import glob as _glob

    _angle   = int(globals().get('SIMIND_ANGLE', 0))
    _want_hr = globals().get('SIMIND_FRAME_HR', None)

    def _find_proj_niftis(folder):
        """Return {frame_hr: {w: path}} dict from spect_postprocess niftis."""
        pp_dir = os.path.join(folder, 'post_processing', 'spect_postprocess')
        frames = {}
        for window in ('w1', 'w2', 'w3'):
            for p in _glob.glob(os.path.join(pp_dir, f'*_tot_{window}.nii.gz')):
                base = os.path.basename(p)
                # name pattern: prefix_<frame_hr>_tot_w?.nii.gz
                parts = base.replace(f'_tot_{window}.nii.gz','').rsplit('_', 1)
                try:
                    hr = float(parts[-1])
                except ValueError:
                    continue
                frames.setdefault(hr, {})[window] = p
        return frames

    for i, folder in _iter_display_folders():
        frame_map = _find_proj_niftis(folder)
        if not frame_map:
            print(f'{ct_label(i)}: no per-frame projection niftis found in spect_postprocess'); continue

        available_hrs = sorted(frame_map.keys())
        if _want_hr is not None and _want_hr in frame_map:
            show_hr = _want_hr
        else:
            show_hr = available_hrs[0]

        windows_avail = [w for w in ('w1','w2','w3') if w in frame_map.get(show_hr,{})]
        if not windows_avail:
            print(f'{ct_label(i)}: no w1/w2/w3 at frame {show_hr}h'); continue

        print(f'{ct_label(i)}: showing frame={show_hr}h  angle_idx={_angle}  windows={windows_avail}')

        fig, axes = plt.subplots(1, len(windows_avail), figsize=(6*len(windows_avail), 6), squeeze=False)
        fig.patch.set_facecolor('black')
        fig.suptitle(f'{ct_label(i)} — SIMIND Projections  (frame {show_hr} h, angle {_angle})',
                     color='white', fontsize=20, fontweight='bold')

        window_labels = {'w1':'W1 — Lower Scatter','w2':'W2 — Photopeak','w3':'W3 — Upper Scatter'}
        _all_imgs = []
        for w in windows_avail:
            arr3d = sitk.GetArrayFromImage(sitk.ReadImage(frame_map[show_hr][w]))
            # shape: (n_angles, h, w)
            n_ang = arr3d.shape[0]
            ang_idx = min(_angle, n_ang-1)
            img2d = np.rot90(arr3d[ang_idx])
            _all_imgs.append(img2d)

        # shared scale across windows
        _pos_vals = np.concatenate([im[im>0].ravel() for im in _all_imgs if np.any(im>0)])
        _vmax = float(np.percentile(_pos_vals, 99.5)) if _pos_vals.size else 1.0
        _norm = mcolors.PowerNorm(gamma=0.5, vmin=0, vmax=_vmax)

        for ax, w, img2d in zip(axes[0], windows_avail, _all_imgs):
            ax.set_facecolor('black')
            im = ax.imshow(img2d, cmap=plot_spec('simind','projection_cmap','hot'),
                           norm=_norm, interpolation='bilinear', aspect='auto')
            ax.axis('off')
            ax.set_title(window_labels.get(w, w), color='white', fontsize=16, pad=10)

        # shared colorbar
        cbar = fig.colorbar(im, ax=axes[0].tolist(), fraction=0.025, pad=0.02, shrink=0.85)
        cbar.set_label('Counts', color='white', fontsize=14)
        cbar.ax.tick_params(colors='white', labelsize=12)
        cbar.outline.set_edgecolor('grey')

        finish_plot(fig, f'ct_{i+1}_simind_proj_frame{show_hr}', tight=False)


### 6c — OpenGATE Dose Maps

In [ ]:
if not globals().get('SHOW_OPENGATE', False):
    print('6c OpenGATE skipped — set SHOW_OPENGATE = True in Setup')
else:
    _cfgs = globals().get('configs', {})
    for i, folder in _iter_display_folders():
        og_cfg = _cfgs.get(i,{}).get('phase_2',{}).get('opengate_stage',{})
        prefix = og_cfg.get('file_prefix','opengate_simulation')
        og_dir = os.path.join(folder, 'simulations', og_cfg.get('sub_dir_name','opengate_simulation'))
        og_work = os.path.join(og_dir, 'work_dir')

        tag = prefix + '_dose_'
        dose_roi_files = [f for f in list_files(og_dir,'.nii.gz') if os.path.basename(f).startswith(tag)]
        if not dose_roi_files:
            dose_roi_files = [f for f in list_files(og_work,'.nii.gz') if os.path.basename(f).startswith(tag)]

        dose_sum_path = None
        for candidate in [
            os.path.join(folder,'simulations', prefix+'_dose_sum.nii.gz'),
            os.path.join(og_dir, prefix+'_dose_sum.nii.gz'),
        ]:
            if os.path.exists(candidate):
                dose_sum_path = candidate; break

        all_dose = [(stem(p)[len(tag):] if stem(p).startswith(tag) else stem(p), p)
                    for p in dose_roi_files if not stem(p).endswith('_dose_sum')]
        if dose_sum_path:
            all_dose.append(('dose sum', dose_sum_path))

        if not all_dose:
            print(f'{ct_label(i)}: no OpenGATE dose maps found'); continue

        fig, axes, nrows, ncols = make_grid(len(all_dose), max_cols=4, pw=5.3, ph=5.3)
        fig.suptitle(f'{ct_label(i)} — OpenGATE Dose Maps (Gy/decay)')
        cmap = plot_spec('opengate','cmap','plasma')

        for j, (label, path) in enumerate(all_dose):
            ax = axes[j//ncols][j%ncols]
            img = sitk.ReadImage(path)
            arr = sitk.GetArrayFromImage(img)[::-1]
            sp  = img.GetSpacing()
            z, y, x = arr.shape
            _mode = globals().get('OPENGATE_SLICE_MODE') or globals().get('SLICE_MODE', 'mid')
            sl = get_slice(arr, axis=1, mode=_mode)
            im = ax.imshow(sl, cmap=cmap, aspect=med_aspect(sp[2],sp[0]), interpolation='bilinear')
            ax.set_title(label.replace('_',' ')); ax.axis('off')
            add_colorbar(fig, ax, im, 'Gy/decay')

        for j in range(len(all_dose), nrows*ncols):
            axes[j//ncols][j%ncols].set_visible(False)
        finish_plot(fig, f'ct_{i+1}_opengate_dose_maps')


### 6d — Voxel S-value Map (OpenGATE · Gy/decay)

In [ ]:
if not globals().get('SHOW_OPENGATE', False):
    print('6d OpenGATE dose sum skipped — set SHOW_OPENGATE = True in Setup')
else:
    _cfgs     = globals().get('configs', {})
    cmap_ds   = plt.get_cmap(plot_spec('opengate', 'cmap', 'plasma')).copy()
    cmap_ds.set_under('black')
    _view     = globals().get('DOSE_SUM_VIEW_AXIS', 'coronal')
    _axis_map = {'axial': 0, 'coronal': 1, 'sagittal': 2}
    _axis     = _axis_map.get(_view, 1)
    _asp_axes = {0: (1, 0), 1: (2, 0), 2: (2, 1)}  # (sp idx numerator, denominator)

    for i, folder in _iter_display_folders():
        og_cfg = _cfgs.get(i, {}).get('phase_2', {}).get('opengate_stage', {})
        prefix = og_cfg.get('file_prefix', 'opengate_simulation')

        dose_sum_path = None
        for candidate in [
            os.path.join(folder, 'simulations', prefix + '_dose_sum.nii.gz'),
            os.path.join(folder, 'simulations',
                         og_cfg.get('sub_dir_name', 'opengate_simulation'),
                         prefix + '_dose_sum.nii.gz'),
        ]:
            if os.path.exists(candidate):
                dose_sum_path = candidate; break

        if dose_sum_path is None:
            print(f'{ct_label(i)}: opengate dose sum not found'); continue

        img = sitk.ReadImage(dose_sum_path)
        arr = sitk.GetArrayFromImage(img)[::-1]
        sp  = img.GetSpacing()          # (sp_x, sp_y, sp_z)
        z, y, x = arr.shape
        pos = arr[arr > 0]
        vm  = float(np.percentile(pos, 99.5)) if pos.size else float(arr.max() + 1e-9)
        print(f'{ct_label(i)}  dose_sum  shape=({z},{y},{x})  vmax={vm:.4g} Gy/decay')

        _mode = globals().get('DOSE_SLICE_MODE') or globals().get('SLICE_MODE', 'mid')
        sl    = get_slice(arr, axis=_axis, mode=_mode)
        ni, nj = _asp_axes[_axis]
        asp   = med_aspect(sp[ni], sp[nj])

        fig, ax = plt.subplots(figsize=(8, 8))
        fig.patch.set_facecolor('black')
        ax.set_facecolor('black')
        im = ax.imshow(sl, cmap=cmap_ds, vmin=1e-30, vmax=vm,
                       aspect=asp, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(f'{ct_label(i)} — Voxel S-value Map  [{_view.capitalize()}]  (Gy/decay)',
                     color='white', fontsize=13, pad=8)
        cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03, shrink=0.85)
        cb.set_label('Gy / decay', color='white', fontsize=11)
        cb.ax.tick_params(colors='white', labelsize=9)
        cb.outline.set_edgecolor('grey')
        finish_plot(fig, f'ct_{i+1}_voxel_svalue_{_view}')


## 7 — Phase 3: Post-processing Outputs

### 7b — Per-ROI Activity Maps
*Shown before 7a since these use only PBPK + segmentation (no SPECT reconstruction needed).*

In [ ]:
if not globals().get('SHOW_ROI_ACTIVITY', False):
    print('7b ROI activity maps skipped — set SHOW_ROI_ACTIVITY = True in Setup')
else:
    _cfgs = globals().get('configs', {})
    for i, folder in _iter_display_folders():
        work_dir = os.path.join(folder, 'post_processing', 'spect_postprocess', 'work_dir')
        act_files = list_files(work_dir, '_act_av.nii.gz')
        if not act_files:
            print(f'{ct_label(i)}: no per-ROI activity maps in {work_dir}'); continue

        sp_prefix = _cfgs.get(i,{}).get('phase_3',{}).get('spect_postprocess_stage',{}).get('file_prefix','spect_postprocess')
        labels = []
        for p in act_files:
            s = stem(p).replace('_act_av','')
            labels.append(s[len(sp_prefix)+1:] if s.startswith(sp_prefix+'_') else s)

        fig, axes, nrows, ncols = make_grid(len(act_files), max_cols=4, pw=5.3, ph=5.3)
        fig.suptitle(f'{ct_label(i)} — Per-ROI Activity Maps')
        cmap = plot_spec('activity','cmap','hot')

        for j, (path, label) in enumerate(zip(act_files, labels)):
            ax = axes[j//ncols][j%ncols]
            img = sitk.ReadImage(path)
            arr = sitk.GetArrayFromImage(img)[::-1]
            sp  = img.GetSpacing()
            z, y, x = arr.shape

            _mode = globals().get('ROI_ACT_SLICE_MODE') or globals().get('SLICE_MODE', 'mid')
            if _mode == 'max':
                sl = get_slice(arr, axis=1, mode='max')
            else:
                nz_yx = np.where(np.max(arr, axis=2) > 0)
                y_mid = int(np.round(np.median(nz_yx[1]))) if nz_yx[0].size > 0 else y // 2
                sl = arr[:, y_mid, :]
            pos = sl[sl > 0]
            vmax_roi = float(np.percentile(pos, 99)) if pos.size else float(sl.max()+1e-9)
            im = ax.imshow(sl, cmap=cmap, vmin=0, vmax=vmax_roi,
                           aspect=med_aspect(sp[2], sp[0]), interpolation='bilinear')
            ax.set_title(fmt_roi_label_with_vol(label, i, label.replace('_', ' '))); ax.axis('off')
            add_colorbar(fig, ax, im, 'MBq/mL')

        for j in range(len(act_files), nrows*ncols):
            axes[j//ncols][j%ncols].set_visible(False)
        finish_plot(fig, f'ct_{i+1}_roi_activity_maps')


### 7a — Reconstructed SPECT

In [ ]:
recon_paths_store = {}

if not globals().get('SHOW_SPECT', False):
    print('7a SPECT skipped — set SHOW_SPECT = True in Setup')
else:
    _folders = _resolve_out_folders()
    _view    = globals().get('SPECT_VIEW_AXIS', 'sagittal')   # 'axial','coronal','sagittal'

    for i, folder in enumerate(_folders):
        pp_dir = os.path.join(folder, 'post_processing')
        recons = sorted([os.path.join(pp_dir, f) for f in os.listdir(pp_dir)
                         if f.startswith('reconstructed_SPECT_') and f.endswith('.nii.gz')
                        ], key=lambda p: float(os.path.basename(p).replace('reconstructed_SPECT_','').replace('.nii.gz','')))                  if os.path.isdir(pp_dir) else []
        if not recons:
            print(f'{ct_label(i)}: no reconstructed SPECT'); continue

        frame_map = {os.path.basename(p).replace('reconstructed_SPECT_','').replace('.nii.gz',''):p for p in recons}
        recon_paths_store[i] = frame_map
        if not should_display(i): continue

        frame_labels = select_frame_labels(frame_map.keys(), globals().get('SPECT_FRAME_MODE','all'),
                                           globals().get('SPECT_SELECTED_FRAMES',[]))
        if not frame_labels:
            print(f'{ct_label(i)}: no frames selected'); continue

        # Load arrays + compute shared vmax
        arrs, vmax = {}, 0.0
        for lbl in frame_labels:
            img = sitk.ReadImage(frame_map[lbl])
            arr = sitk.GetArrayFromImage(img)[::-1]
            sp  = img.GetSpacing()
            arrs[lbl] = (arr, sp)
            pos = arr[arr > 0]
            if pos.size:
                vmax = max(vmax, float(np.percentile(pos, plot_spec('spect','percentile',99.5))))
        shared_vmax = vmax if globals().get('SPECT_SHARED_SCALE', True) and vmax > 0 else None

        n = len(frame_labels)
        nrows, ncols_per_row = optimal_grid(n, max_cols=7)

        # Use first image for reference dimensions
        ref_arr, ref_sp = arrs[frame_labels[0]]
        z, y, x = ref_arr.shape
        sp_x, sp_y, sp_z = ref_sp

        if _view == 'axial':
            asp = med_aspect(sp_y, sp_x); _ph = 4.5
        elif _view == 'coronal':
            asp = med_aspect(sp_z, sp_x); _ph = 5.5
        else:  # sagittal
            asp = med_aspect(sp_z, sp_y); _ph = 5.5

        fig, axes = plt.subplots(nrows, ncols_per_row,
                                  figsize=(_ph*ncols_per_row, _ph*nrows), squeeze=False)
        fig.patch.set_facecolor('black')
        fig.suptitle(f'{ct_label(i)} — Reconstructed SPECT ({_view})', color='white', fontsize=20)

        cmap = plot_spec('spect','cmap','hot')
        norm = mcolors.PowerNorm(gamma=0.5, vmin=0, vmax=shared_vmax) if shared_vmax else None
        last_im = None

        for idx, lbl in enumerate(frame_labels):
            arr, sp = arrs[lbl]
            z, y, x = arr.shape
            if _view == 'axial':     sl = arr[z//2, :, :]
            elif _view == 'coronal': sl = np.max(arr, axis=1)
            else:                    sl = arr[:, :, x//2]

            row, col = divmod(idx, ncols_per_row)
            ax = axes[row][col]
            ax.set_facecolor('black')
            kwargs = dict(cmap=cmap, aspect=asp, interpolation='bilinear')
            if norm: kwargs['norm'] = norm
            last_im = ax.imshow(sl, **kwargs)
            ax.axis('off')
            ax.set_title(f'{lbl} h', color='white', fontsize=14)

        for idx in range(n, nrows*ncols_per_row):
            r, c = divmod(idx, ncols_per_row)
            axes[r][c].set_visible(False)

        # Shared colorbar on far right
        if last_im is not None:
            cbar_ax = fig.add_axes([0.93, 0.12, 0.015, 0.70])
            cb = fig.colorbar(last_im, cax=cbar_ax)
            cb.set_label('MBq/mL', color='white', fontsize=14)
            cb.ax.tick_params(colors='white', labelsize=12)
            cb.outline.set_edgecolor('grey')

        finish_plot(fig, f'ct_{i+1}_reconstructed_spect', tight=False)

# Always build recon_paths_store for comparison setup
if not recon_paths_store:
    _folders = _resolve_out_folders()
    for i, folder in enumerate(_folders):
        pp_dir = os.path.join(folder, 'post_processing')
        if not os.path.isdir(pp_dir): continue
        frame_map = {os.path.basename(p).replace('reconstructed_SPECT_','').replace('.nii.gz',''):
                     os.path.join(pp_dir, p)
                     for p in os.listdir(pp_dir)
                     if p.startswith('reconstructed_SPECT_') and p.endswith('.nii.gz')}
        if frame_map:
            recon_paths_store[i] = frame_map
    if recon_paths_store:
        print(f'Recon paths loaded for {len(recon_paths_store)} patient(s) (not plotted)')



### 7c — Total Dose Map

In [ ]:
if not globals().get('SHOW_DOSE', False):
    print('7c Dose map skipped — set SHOW_DOSE = True in Setup')
else:
    _unc_vmax = globals().get('DOSE_UNC_VMAX_PCT', 50) / 100.0  # e.g. 50 % → 0.5

    for i, folder in _iter_display_folders():
        dose_path = os.path.join(folder, 'post_processing', 'dosemap_postprocess_total_dose.nii.gz')
        unc_path  = os.path.join(folder, 'simulations', 'opengate_simulation_dose_sum_unc.nii.gz')
        if not os.path.exists(dose_path):
            print(f'{ct_label(i)}: total dose map not found'); continue

        img  = sitk.ReadImage(dose_path)
        arr  = sitk.GetArrayFromImage(img)[::-1]
        sp   = img.GetSpacing()
        z, y, x = arr.shape
        body_mask = arr > 0
        pos  = arr[body_mask]
        vm   = float(np.percentile(pos, plot_spec('dose','percentile',99.0))) if pos.size else float(arr.max()+1e-9)
        print(f'{ct_label(i)}  dose shape=({z},{y},{x})  max={arr.max():.4g} Gy')

        _mode   = globals().get('DOSE_SLICE_MODE') or globals().get('SLICE_MODE', 'mid')
        sl_dose = get_slice(arr, axis=1, mode=_mode)
        mask_sl = get_slice(body_mask.astype(np.uint8), axis=1, mode=_mode) > 0
        asp     = med_aspect(sp[2], sp[0])
        phys_h  = z * sp[2]; phys_w = x * sp[0]

        has_unc = os.path.exists(unc_path)
        ncols   = 2 if has_unc else 1
        fig_w   = 7
        fig_h   = max(3.0, fig_w * phys_h / phys_w)
        fig, axes = plt.subplots(1, ncols, figsize=(fig_w * ncols, fig_w), squeeze=False)
        axes = axes[0]
        fig.patch.set_facecolor('black')

        # ── Left: total dose map ─────────────────────────────────────────────
        cmap_dose = plt.get_cmap(plot_spec('dose','cmap','plasma')).copy()
        cmap_dose.set_bad('black')
        disp_dose = np.where(mask_sl, sl_dose, np.nan)
        im_dose = axes[0].imshow(disp_dose, cmap=cmap_dose, vmin=0, vmax=vm,
                                  aspect=asp, interpolation='bilinear')
        axes[0].axis('off')
        axes[0].set_facecolor('black')
        axes[0].set_title('Total Dose Map', color='white')
        cb_d = fig.colorbar(im_dose, ax=axes[0], fraction=0.04, pad=0.03, shrink=0.85)
        cb_d.set_label('Dose (Gy)', color='white', fontsize=13)
        cb_d.ax.tick_params(colors='white', labelsize=11)
        cb_d.outline.set_edgecolor('grey')

        # ── Right: uncertainty map ────────────────────────────────────────────
        if has_unc:
            unc_arr  = sitk.GetArrayFromImage(sitk.ReadImage(unc_path))[::-1]
            sl_unc   = unc_arr[:, y//2, :]
            disp_unc = np.where(mask_sl, sl_unc, np.nan)  # zero outside body → black

            cmap_unc = plt.get_cmap('RdYlGn_r').copy()
            cmap_unc.set_bad('black')
            axes[1].set_facecolor('black')
            im_unc = axes[1].imshow(disp_unc, cmap=cmap_unc, vmin=0, vmax=_unc_vmax,
                                     aspect=asp, interpolation='bilinear')
            axes[1].axis('off')
            axes[1].set_title(f'Uncertainty Map  (0 – {int(_unc_vmax*100)} %)', color='white')
            cb_u = fig.colorbar(im_unc, ax=axes[1], fraction=0.04, pad=0.03, shrink=0.85)
            cb_u.set_label('Relative Uncertainty', color='white', fontsize=13)
            cb_u.ax.yaxis.set_major_formatter(
                plt.matplotlib.ticker.FuncFormatter(lambda v, _: f'{v*100:.0f}%'))
            cb_u.ax.tick_params(colors='white', labelsize=11)
            cb_u.outline.set_edgecolor('grey')

        fig.suptitle(f'{ct_label(i)} — Total Absorbed Dose', color='white', fontsize=18)
        fig.tight_layout(rect=[0, 0, 1, 0.93])
        finish_plot(fig, f'ct_{i+1}_total_dose', tight=False)


### 7d — Dose Integration Summary

In [ ]:
dose_summary_store = {}

if not globals().get('SHOW_DOSE_SUMMARY', False):
    print('7d Dose summary skipped — set SHOW_DOSE_SUMMARY = True in Setup')
else:
    _folders = _resolve_out_folders()
    def pbpk_decays_for_roi(patient_idx, roi):
        if patient_idx not in pbpk_store or roi not in pbpk_store[patient_idx]: return np.nan
        t_min = np.asarray(pbpk_store[patient_idx]['time_hr'], float) * 60.0
        y_mbq = np.asarray(pbpk_store[patient_idx][roi], float)
        return float(np.trapezoid(y_mbq, t_min) * 60.0 * 1e6)

    for i, folder in enumerate(_folders):
        meta_path = os.path.join(folder, 'pipeline_metadata', 'dosemap_postprocess_stage.json')
        if not os.path.exists(meta_path):
            print(f'{ct_label(i)}: dose postprocess metadata not found'); continue
        m = load_json(meta_path)
        dose_summary_store[i] = m
        decays = m.get('cumulated_activities_per_roi_decays', {})
        range_hr = m.get('integration_range_hr', [])
        print(f'\n{ct_label(i)}: integration range = {range_hr} hr')
        if not decays:
            print('  No cumulated activity entries'); continue
        print(f"  {'ROI':20s} {'dose-stage CA (MBq·h)':>22s} {'PBPK CA (MBq·h)':>17s}")
        for roi, val in sorted(decays.items()):
            direct = pbpk_decays_for_roi(i, roi)
            print(f'  {roi:20s} {val/3.6e9:22.1f} {direct/3.6e9:17.1f}')


## 8 — SPECT Comparison Setup

### 8a — Build ROI Masks in SPECT Space

In [ ]:
def physical_center(img):
    size = np.array(img.GetSize(), dtype=float)
    sp   = np.array(img.GetSpacing(), dtype=float)
    orig = np.array(img.GetOrigin(), dtype=float)
    direction = np.array(img.GetDirection(), dtype=float).reshape(3,3)
    return orig + direction @ ((size-1.0)/2.0*sp)

def set_center(img, target):
    out = sitk.Image(img)
    size = np.array(out.GetSize(), dtype=float)
    sp   = np.array(out.GetSpacing(), dtype=float)
    direction = np.array(out.GetDirection(), dtype=float).reshape(3,3)
    out.SetOrigin((target - direction @ ((size-1.0)/2.0*sp)).tolist())
    return out

def resample_nn(moving, reference):
    rs = sitk.ResampleImageFilter()
    rs.SetReferenceImage(reference)
    rs.SetInterpolator(sitk.sitkNearestNeighbor)
    rs.SetDefaultPixelValue(0)
    return rs.Execute(moving)

masks_recon       = {}
recon_imgs_sitk   = {}
recon_vox_ml      = {}
recon_arr_ref     = {}
mask_volume_store = {}

_folders = _resolve_out_folders()
roi_map_path = str(Path(OUTPUT_ROOT) / 'src' / 'data' / 'pipeline_roi_naming_map.json')
label_to_roi = {}
if os.path.exists(roi_map_path):
    roi_map = load_json(roi_map_path)
    label_to_roi = {int(k): v.lower() for k,v in roi_map.get('PyTheraTwin_Pipeline',{}).items()}

_rps = recon_paths_store if recon_paths_store else {}

for i, folder in enumerate(_folders):
    if i not in _rps:
        print(f'{ct_label(i)}: no recon paths — run 7a first or set SHOW_SPECT=True and re-run'); continue

    seg_path = os.path.join(folder, 'digital_twin', 'digital_twin.nii.gz')
    if not os.path.exists(seg_path):
        print(f'{ct_label(i)}: digital_twin.nii.gz not found'); continue

    frame_map = _rps[i]
    ref_label = sorted(frame_map.keys(), key=lambda k: float(k))[-1]
    ref_img   = sitk.ReadImage(frame_map[ref_label])
    sp_recon  = np.array(ref_img.GetSpacing(), dtype=float)
    recon_unit = normalize_unit(globals().get('SPECT_RECON_SPACING_UNIT','mm'))
    seg_unit   = normalize_unit(globals().get('SEGMENTATION_SPACING_UNIT','mm'))

    recon_imgs_sitk[i] = {lbl: sitk.ReadImage(p) for lbl, p in frame_map.items()}
    recon_vox_ml[i]    = voxel_volume_ml(sp_recon, recon_unit)
    recon_arr_ref[i]   = (sitk.GetArrayFromImage(ref_img)[::-1], sp_recon)

    seg_img = sitk.ReadImage(seg_path)
    seg_img_ru = scale_img_units(seg_img, seg_unit, recon_unit)
    seg_centered = set_center(seg_img_ru, physical_center(ref_img))
    seg_in_recon = resample_nn(seg_centered, ref_img)
    seg_arr = sitk.GetArrayFromImage(seg_in_recon).astype(np.int16)

    masks_recon[i]       = {}
    mask_volume_store[i] = {}
    for lbl in np.unique(seg_arr):
        if lbl == 0: continue
        roi_name = label_to_roi.get(int(lbl), f'label_{lbl}')
        mask = (seg_arr == lbl)
        masks_recon[i][roi_name] = mask
        mask_volume_store[i][roi_name] = {'voxels': int(mask.sum()), 'volume_ml': float(mask.sum()*recon_vox_ml[i])}

    # ── Bug 2a fix ─────────────────────────────────────────────────────────
    # digital_twin.nii.gz's remaining_body (label 1) was computed at phase 1 against
    # ALL ~40 sub-organ labels. SIMIND recomputes remaining_body per-stage as
    # body_outline - union(roi_subset) (see label_utils.filter_roi_seg_to_subset), and
    # its roi_subset is only the handful of organs it actually simulates directly — so
    # the SIMIND remaining_body source covers everything else (bone, gi_tract, muscle,
    # etc.). Rebuild the mask here to match what SIMIND actually filled, instead of the
    # much smaller phase-1 definition.
    body_outline = np.zeros_like(seg_arr, dtype=bool)
    for roi_name, m in masks_recon[i].items():
        if roi_name != 'background':
            body_outline |= m

    _simind_rois = [str(r).lower() for r in
                     globals().get('simind_meta_store', {}).get(i, {}).get('roi_list', [])]
    if not _simind_rois:
        _simind_rois = [str(r).lower() for r in
                         configs.get(i, {}).get('phase_2', {}).get('simind_stage', {}).get('roi_subset', [])]
    _simind_subtract_rois = [r for r in _simind_rois if r != 'remaining_body']

    _subtract_mask = np.zeros_like(seg_arr, dtype=bool)
    for roi_name in _simind_subtract_rois:
        m = masks_recon[i].get(roi_name)
        if m is not None:
            _subtract_mask |= m

    rb_mask = body_outline & ~_subtract_mask
    masks_recon[i]['remaining_body'] = rb_mask
    mask_volume_store[i]['remaining_body'] = {
        'voxels': int(rb_mask.sum()), 'volume_ml': float(rb_mask.sum() * recon_vox_ml[i])}

    print(f'{ct_label(i)}: {len(masks_recon[i])} ROI masks built  recon_vox={recon_vox_ml[i]:.4f} mL')
    for roi, v in mask_volume_store[i].items():
        print(f'  {roi:25s} {v["voxels"]:6d} vox  {v["volume_ml"]:.2f} mL')


### 8b — Mask Overlay Sanity Check

In [ ]:
if not globals().get('SHOW_MASK_OVERLAY', False):
    print('8b mask overlay skipped — set SHOW_MASK_OVERLAY = True in Setup')
elif not masks_recon:
    print('8b: run 8a first')
else:
    for i in masks_recon:
        if not should_display(i): continue
        arr_ref, sp_recon = recon_arr_ref[i]
        sp_x, sp_y, sp_z  = sp_recon
        rois = [r for r in masks_recon[i] if r != 'background']
        if not rois:
            print(f'{ct_label(i)}: no masks'); continue

        fig, axes, nrows, ncols = make_grid(len(rois), max_cols=5, pw=4.5, ph=4.5)
        fig.suptitle(f'{ct_label(i)} — ROI Masks in SPECT Space (latest frame)')
        image_cmap   = plot_spec('mask','image_cmap','hot')
        contour_color = plot_spec('mask','contour_color','cyan')

        for j, roi in enumerate(rois):
            ax = axes[j//ncols][j%ncols]
            mask = masks_recon[i][roi][::-1]   # display orientation
            pts  = np.argwhere(mask)
            if pts.size == 0:
                ax.set_visible(False); continue

            # Find coronal slice through the non-zero centre of the ROI
            y_mid = int(np.round(np.median(pts[:, 1])))
            sl_recon = arr_ref[:, y_mid, :]    # shape (z, x)
            sl_mask  = mask[:, y_mid, :]

            pos = sl_recon[np.isfinite(sl_recon) & (sl_recon > 0)]
            vmax_r = float(np.percentile(pos, 99)) if pos.size else 1.0

            asp = med_aspect(sp_z, sp_x)       # physical aspect ratio
            ax.imshow(sl_recon, cmap=image_cmap, vmin=0, vmax=vmax_r,
                      aspect=asp, interpolation='bilinear')
            ax.contour(sl_mask.astype(np.uint8), levels=[0.5],
                       colors=contour_color, linewidths=1.5)
            ax.set_title(fmt_roi_label_with_vol(roi, i, roi.replace('_', ' ')), fontsize=13)
            ax.axis('off')

        for j in range(len(rois), nrows*ncols):
            axes[j//ncols][j%ncols].set_visible(False)
        finish_plot(fig, f'ct_{i+1}_mask_overlay')


### 8c — Extract SPECT TACs

In [ ]:
def roi_stats(arr_zyx, mask_zyx, vox_vol_ml):
    n = int(mask_zyx.sum())
    if n == 0: return 0.0, 0.0, 0
    mean_mbq_ml = float(np.mean(arr_zyx[mask_zyx]))
    total_mbq   = float(np.sum(arr_zyx[mask_zyx]) * vox_vol_ml)
    return mean_mbq_ml, total_mbq, n

spect_tacs = {}

for i in masks_recon:
    frame_map   = recon_imgs_sitk[i]
    frame_labels = sorted(frame_map.keys(), key=lambda k: float(k))
    rois   = [r for r in masks_recon[i] if r != 'background']
    vox_ml = recon_vox_ml[i]
    spect_tacs[i] = {roi: {'frame_hr':[],'sum_mbq':[],'mean_mbq_ml':[],'n_voxels':[]} for roi in rois}

    for lbl in frame_labels:
        arr = sitk.GetArrayFromImage(frame_map[lbl]).astype(np.float32)
        for roi in rois:
            mask = masks_recon[i][roi]
            if arr.shape != mask.shape:
                print(f'{ct_label(i)} {roi}: shape mismatch'); continue
            mean_c, total, n_vox = roi_stats(arr, mask, vox_ml)
            spect_tacs[i][roi]['frame_hr'].append(float(lbl))
            spect_tacs[i][roi]['sum_mbq'].append(total)
            spect_tacs[i][roi]['mean_mbq_ml'].append(mean_c)
            spect_tacs[i][roi]['n_voxels'].append(n_vox)

    print(f'{ct_label(i)}: SPECT TACs extracted  frames={frame_labels}  ROIs={rois}')


## 9 — Comparison 1: PBPK vs Reconstructed SPECT TACs

**9a — PBPK vs SPECT TAC comparison.**  
For each ROI, PBPK activity (interpolated to SPECT frame times) is compared against measured SPECT counts.  
Diff = SPECT − PBPK. % Error = (SPECT − PBPK) / |PBPK| × 100.

In [ ]:
# ── Per-run comparison filters (edit here) ───────────────────────────────────
_EXCLUDE_ROIS   = globals().get('COMPARISON_EXCLUDE_ROIS', [])    # e.g. ['liver']
_EXCLUDE_FRAMES = globals().get('COMPARISON_EXCLUDE_FRAMES', [])  # e.g. [168]
_ERR_LIMIT      = globals().get('COMPARISON_ERROR_LIMIT_PCT', None)  # e.g. 150

def simulated_rois_for_patient(i):
    rois = set()
    meta = simind_meta_store.get(i, {})
    rois.update(str(r).lower() for r in meta.get('roi_list',[]) if r)
    if not rois:
        cfg_rois = configs.get(i,{}).get('phase_2',{}).get('simind_stage',{}).get('roi_subset',[])
        rois.update(str(r).lower() for r in cfg_rois if r)
    if not globals().get('INCLUDE_REMAINING_BODY_IN_COMPARISON', False):
        rois.discard('remaining_body')
    return rois

def comparison_rois_for_patient(i, available):
    simulated = simulated_rois_for_patient(i) if globals().get('COMPARE_ONLY_SIMULATED_ROIS',True) else None
    skip = globals().get('SKIP_ROIS_COMPARISON', {'remaining_body','background'})
    ex   = {r.lower() for r in _EXCLUDE_ROIS}
    out  = []
    for roi in available:
        k = roi.lower()
        if k in skip or k in ex: continue
        if not globals().get('INCLUDE_REMAINING_BODY_IN_COMPARISON',False) and k=='remaining_body': continue
        if simulated is not None and simulated and k not in simulated: continue
        out.append(roi)
    return sorted(out)

def array_after_frame_filter(t, *ys):
    t = np.asarray(t, float); ys = [np.asarray(y,float) for y in ys]
    order = np.argsort(t); t = t[order]; ys = [y[order] for y in ys]
    ex_set = {float(f) for f in _EXCLUDE_FRAMES}
    keep = np.array([not any(abs(tv-f)<0.01 for f in ex_set) for tv in t])
    if globals().get('SKIP_LAST_FRAME',False) and keep.sum()>1:
        last = np.where(keep)[0][-1]; keep[last] = False
    t = t[keep]; ys = [y[keep] for y in ys]
    return (t, *ys)

# ── Bug 2b fix ─────────────────────────────────────────────────────────────
# pbpk_store[i]['remaining_body'] is only the 'Rest' TAC — the SIMIND remaining_body
# SOURCE (see the Bug 2a mask rebuild in 8a) covers everything SIMIND didn't simulate
# directly, so its prescribed reference must be the whole-body total minus the
# organs SIMIND actually simulated, not the Rest TAC alone.
def remaining_body_prescribed(i, t_hr):
    """Prescribed remaining_body activity (MBq) at t_hr: pbpk_total minus the
    SIMIND-simulated organs' PBPK TACs (mirrors the Bug 2a remaining_body mask)."""
    sim_rois = simind_meta_store.get(i, {}).get('roi_list') or \
               configs.get(i, {}).get('phase_2', {}).get('simind_stage', {}).get('roi_subset', [])
    organs = [str(r).lower() for r in sim_rois if str(r).lower() != 'remaining_body']
    out = np.asarray(pbpk_total(i, t_hr), dtype=float)
    for r in organs:
        if r in pbpk_store[i]:
            out = out - np.interp(t_hr, pbpk_store[i]['time_hr'], pbpk_store[i][r])
    return out

if not globals().get('SHOW_COMPARISON', False):
    print('9a Comparison skipped — set SHOW_COMPARISON = True in Setup')
elif not spect_tacs:
    print('9a: run 8c first')
else:
    diff_store = {}
    pbpk_color = plot_spec('comparison','pbpk_color','steelblue')
    spect_color = plot_spec('comparison','spect_color','orangered')
    colors_pbpk = plot_spec('pbpk','colors',['#1f77b4','#d62728','#2ca02c','#9467bd','#8c564b','#e377c2','#17becf','#bcbd22'])

    for i in spect_tacs:
        if i not in pbpk_store: continue
        diff_store[i] = {}
        pbpk_t = pbpk_store[i]['time_hr']
        available = [r for r in spect_tacs[i] if spect_tacs[i][r]['sum_mbq']]
        plot_rois = comparison_rois_for_patient(i, available)
        if not plot_rois:
            print(f'{ct_label(i)}: no ROIs after filters'); continue

        for roi in plot_rois:
            pbpk_key = roi.lower()
            if pbpk_key not in pbpk_store[i]: continue
            t_hr = np.array(spect_tacs[i][roi]['frame_hr'])
            y_s  = np.array(spect_tacs[i][roi]['sum_mbq'])
            if pbpk_key == 'remaining_body':
                y_p = remaining_body_prescribed(i, t_hr)
            else:
                y_p = np.interp(t_hr, pbpk_t, pbpk_store[i][pbpk_key])
            t_hr, y_s, y_p = array_after_frame_filter(t_hr, y_s, y_p)
            diff_store[i][roi] = {
                'frame_hr': t_hr, 'diff_mbq': y_s-y_p,
                'pct_err': pct(y_s, y_p), 'y_spect': y_s, 'y_pbpk': y_p,
            }

    for i, diffs in diff_store.items():
        plot_rois = sorted([r for r in diffs if r != 'total_body'],
                           key=lambda r: (r.lower() == 'remaining_body', r))
        if not plot_rois: continue

        # ── Bug 3 fix — total body: entire reconstructed volume (no mask) vs the
        # whole-body prescribed total (pbpk_total), instead of summing a handful of
        # ROI masks (which is not a whole-body measurement at all).
        _tb_vox_ml = recon_vox_ml[i]
        _tb_frame_labels = sorted(recon_imgs_sitk[i].keys(), key=lambda k: float(k))
        _tb_frames = np.array([float(lbl) for lbl in _tb_frame_labels])
        _tb_spect = np.array([
            float(np.sum(sitk.GetArrayFromImage(recon_imgs_sitk[i][lbl])) * _tb_vox_ml)
            for lbl in _tb_frame_labels
        ])
        _tb_pbpk_interp = np.asarray(pbpk_total(i, _tb_frames), dtype=float)
        _tb_diff = _tb_spect - _tb_pbpk_interp
        _tb_pct  = pct(_tb_spect, _tb_pbpk_interp)
        diff_store[i]['total_body'] = {
            'frame_hr': _tb_frames, 'diff_mbq': _tb_diff,
            'pct_err': _tb_pct, 'y_spect': _tb_spect, 'y_pbpk': _tb_pbpk_interp,
        }
        _tb_pbpk_full = pbpk_total(i, pbpk_store[i]['time_hr'])  # dense curve, for the plot below

        if not should_display(i): continue  # total_body stored; skip per-patient plots

        pbpk_t = pbpk_store[i]['time_hr']
        all_frames = np.concatenate([diffs[r]['frame_hr'] for r in plot_rois if len(diffs[r]['frame_hr'])])
        xlim = comparison_xlim(all_frames)

        # ── Resolve shared xlim for all plots this patient ───────────────────
        _comp_xlim_raw = globals().get('PBPK_XLIM_HR', 25)
        if isinstance(_comp_xlim_raw, (list,tuple)): _comp_xlim_hr = float(_comp_xlim_raw[1])
        elif _comp_xlim_raw is not None: _comp_xlim_hr = float(_comp_xlim_raw)
        else: _comp_xlim_hr = None

        # ── TAC overlay per ROI + total body as last subplot ─────────────────
        _n_tac = len(plot_rois) + 1
        fig, axes, nrows, ncols = make_grid(_n_tac, max_cols=4, pw=5.8, ph=5.8)
        fig.suptitle(f'{ct_label(i)} — PBPK vs SPECT TAC')

        for j, roi in enumerate(plot_rois):
            ax = axes[j//ncols][j%ncols]
            d  = diffs[roi]
            y_pbpk_dense = (remaining_body_prescribed(i, pbpk_t) if roi.lower() == 'remaining_body'
                             else pbpk_store[i][roi.lower()])
            ax.plot(pbpk_t, y_pbpk_dense, lw=4, color=pbpk_color, label='PBPK')
            ax.scatter(d['frame_hr'], d['y_spect'], s=75, color=spect_color, zorder=5, label='SPECT')
            ax.plot(d['frame_hr'], d['y_spect'], '--', color=spect_color, lw=1, alpha=0.7)
            if _comp_xlim_hr is not None: ax.set_xlim(0, _comp_xlim_hr)
            elif xlim: ax.set_xlim(*xlim)
            ax.set_title(fmt_roi_label_with_vol(roi, i))
            ax.set_xlabel('Time (hr)'); ax.set_ylabel('Activity (MBq)')
            ax.legend(fontsize=13); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

        # Total body subplot (last)
        _j_tb = len(plot_rois)
        ax_tb = axes[_j_tb//ncols][_j_tb%ncols]
        _step = max(1, len(pbpk_t) // 1000)
        ax_tb.plot(pbpk_t[::_step], _tb_pbpk_full[::_step], lw=4, color=pbpk_color, label='PBPK')
        ax_tb.scatter(_tb_frames, _tb_spect, s=75, color=spect_color, zorder=5, label='SPECT')
        ax_tb.plot(_tb_frames, _tb_spect, '--', color=spect_color, lw=1, alpha=0.7)
        if _comp_xlim_hr is not None: ax_tb.set_xlim(0, _comp_xlim_hr)
        elif xlim: ax_tb.set_xlim(*xlim)
        ax_tb.set_title('Total Body (entire recon volume)', fontsize=16)
        ax_tb.set_xlabel('Time (hr)'); ax_tb.set_ylabel('Activity (MBq)')
        ax_tb.legend(fontsize=13); ax_tb.spines['top'].set_visible(False); ax_tb.spines['right'].set_visible(False)

        for j in range(_n_tac, nrows*ncols):
            axes[j//ncols][j%ncols].set_visible(False)
        finish_plot(fig, f'ct_{i+1}_comparison1_tac')

        # ── Error trends: diff_mbq and pct_err per ROI + total body ─────────
        fig_err, axes_err = plt.subplots(1, 2, figsize=(15, 6))
        fig_err.suptitle(f'{ct_label(i)} — Error Trends by Frame', fontsize=14)
        for roi in plot_rois:
            col = roi_color(roi)
            d = diffs[roi]
            label = fmt_roi_label(roi, i)
            axes_err[0].plot(d['frame_hr'], d['diff_mbq'], marker='o', lw=1.8, color=col, label=label)
            axes_err[1].plot(d['frame_hr'], d['pct_err'], marker='o', lw=1.8, color=col, label=label)
        # Total body line — black, dashed, thicker, diamond marker
        axes_err[0].plot(_tb_frames, _tb_diff, marker='D', ms=6, lw=2.5, color='k', ls='--', label='Total Body', zorder=10)
        axes_err[1].plot(_tb_frames, _tb_pct,  marker='D', ms=6, lw=2.5, color='k', ls='--', label='Total Body', zorder=10)
        _err_lim = globals().get('COMPARISON_ERROR_LIMIT_PCT', None)
        for ax_e, ylabel_e, title_e, _is_pct in zip(axes_err, ['SPECT − PBPK (MBq)', '% Error'], ['Difference SPECT vs PBPK', 'Error % of SPECT vs PBPK'], [False, True]):
            ax_e.axhline(0, color='k', lw=1, ls='--', alpha=0.55)
            ax_e.set_title(title_e, fontsize=12); ax_e.set_xlabel('Frame time (hr)'); ax_e.set_ylabel(ylabel_e)
            ax_e.legend(fontsize=9)
            ax_e.spines['top'].set_visible(False); ax_e.spines['right'].set_visible(False)
            if xlim: ax_e.set_xlim(*xlim)
            if _is_pct and _err_lim is not None: ax_e.set_ylim(-float(_err_lim), float(_err_lim))
        finish_plot(fig_err, f'ct_{i+1}_comparison1_error_trends')

        # ── Error table: SPECT, PBPK, diff, % error per ROI per frame ──────
        import pandas as pd
        from IPython.display import display
        _rows = []
        for roi in plot_rois:
            d = diffs[roi]
            for fr, ys, yp, dm, pe in zip(d['frame_hr'], d['y_spect'], d['y_pbpk'], d['diff_mbq'], d['pct_err']):
                _rows.append({'ROI': roi.replace('_', ' '), 'Frame (hr)': round(float(fr), 1),
                              'SPECT (MBq)': round(float(ys), 3), 'PBPK (MBq)': round(float(yp), 3),
                              'Diff (MBq)': round(float(dm), 3), '% Error': round(float(pe), 1)})
        # Total body rows
        for fr, ys, yp, dm, pe in zip(_tb_frames, _tb_spect, _tb_pbpk_interp, _tb_diff, _tb_pct):
            _rows.append({'ROI': 'Total Body', 'Frame (hr)': round(float(fr), 1),
                          'SPECT (MBq)': round(float(ys), 3), 'PBPK (MBq)': round(float(yp), 3),
                          'Diff (MBq)': round(float(dm), 3), '% Error': round(float(pe), 1)})
        _df = pd.DataFrame(_rows)
        display(_df.style.set_caption(f'{ct_label(i)} — Comparison 1: Values & Error'))


### 9b — Comparison 1: Average Across Patients

In [ ]:
comparison1_summary_store = {}

if not globals().get('SHOW_COMPARISON', False):
    print('9b skipped')
elif 'diff_store' not in globals() or len(diff_store) < 2:
    print('9b: need diff_store from 9a with >=2 patients')
else:
    all_rois_base = sorted(set().union(*[set(d.keys()) for d in diff_store.values() if d]),
                           key=lambda r: (r == 'total_body', r))
    all_frames    = sorted({float(f) for d in diff_store.values() for r in d.values() for f in r['frame_hr']})
    patient_idxs  = sorted(diff_store.keys())

    # ── Liver subgroup detection: two groups only ────────────────────────────
    def _liver_has_lesion(pidx):
        cfg   = configs.get(pidx, {})
        specs = (cfg.get('phase_1', {}).get('synthetic_lesions_stage', {}) or {}).get('specs') or {}
        return any(organ.lower() == 'liver' for organ in specs)

    def _roi_display_label_9b(roi):
        r = roi.lower()
        if r == 'liver_no_lesion':   return 'Liver (no lesion)'
        if r == 'liver_with_lesion': return 'Liver (with lesion)'
        if r.startswith('synthetic_lesion_'):
            suffix = r[len('synthetic_lesion_'):]
            if suffix in ('tumorrest', 'tumor_rest'):
                return 'Synthetic Lesion: Tumor 3'
            num2 = ''.join(c for c in suffix if c.isdigit())
            return f'Synthetic Lesion: Tumor {num2}' if num2 else 'Synthetic Lesion: Tumor'
        return roi.replace('_', ' ').title()

    # Build exactly two liver subgroups
    _liver_groups = {}
    for _pidx in patient_idxs:
        if 'liver' in diff_store.get(_pidx, {}):
            _key = 'liver_with_lesion' if _liver_has_lesion(_pidx) else 'liver_no_lesion'
            _liver_groups.setdefault(_key, []).append(_pidx)

    print('\nLiver subgroups (Comparison 1):')
    for _lk in ['liver_no_lesion', 'liver_with_lesion']:
        _lpts = _liver_groups.get(_lk, [])
        print(f'  {_lk}: n={len(_lpts)}  patients={[ct_label(p) for p in sorted(_lpts)]}')

    all_rois_expanded = []
    for _r in all_rois_base:
        if _r == 'liver':
            all_rois_expanded.extend(sorted(_liver_groups.keys()))
        else:
            all_rois_expanded.append(_r)

    if all_rois_expanded and all_frames:
        frame_arr = np.asarray(all_frames, float)

        for roi in all_rois_expanded:
            # Liver subgroups pull from 'liver' diff data, filtered to their patient subset
            is_liver_sub = (roi.startswith('liver_') and
                            roi not in diff_store.get(patient_idxs[0] if patient_idxs else -1, {}))
            contributing = _liver_groups.get(roi, []) if is_liver_sub else patient_idxs
            source_roi   = 'liver' if is_liver_sub else roi

            diff_stack = np.full((len(contributing), len(all_frames)), np.nan)
            err_stack  = np.full((len(contributing), len(all_frames)), np.nan)
            for row_idx, pidx in enumerate(contributing):
                if source_roi not in diff_store.get(pidx, {}): continue
                d  = diff_store[pidx][source_roi]
                pf = np.asarray(d['frame_hr'], float)
                for col_idx, frame in enumerate(all_frames):
                    m = np.where(np.isclose(pf, frame, atol=0.01))[0]
                    if m.size:
                        diff_stack[row_idx, col_idx] = d['diff_mbq'][m[0]]
                        err_stack[row_idx, col_idx]  = d['pct_err'][m[0]]
            mean_d, std_d, n_d = nanmean_std_count(diff_stack, axis=0)
            mean_e, std_e, n_e = nanmean_std_count(err_stack, axis=0)
            comparison1_summary_store[roi] = {
                'frame_hr': frame_arr, 'mean_diff_mbq': mean_d, 'std_diff_mbq': std_d,
                'mean_pct_err': mean_e, 'std_pct_err': std_e, 'n': n_d,
                'contributing_patients': contributing,
            }

        import pandas as pd
        from IPython.display import display

        # ── Summary table ────────────────────────────────────────────────────
        _rows = []
        for roi in all_rois_expanded:
            s   = comparison1_summary_store[roi]
            lbl = _roi_display_label_9b(roi)
            n_pts = len(s.get('contributing_patients', patient_idxs))
            for fr, md_v, sd_v, me_v, se_v in zip(
                    s['frame_hr'], s['mean_diff_mbq'], s['std_diff_mbq'],
                    s['mean_pct_err'], s['std_pct_err']):
                _rows.append({
                    'ROI': f'{lbl} (n={n_pts})', 'Frame (hr)': round(float(fr), 1),
                    'Mean Diff (MBq)': round(float(md_v), 3),
                    'Std Diff (MBq)':  round(float(sd_v), 3),
                    'Mean % Err':      round(float(me_v), 1),
                    'Std % Err':       round(float(se_v), 1),
                })
        _df = pd.DataFrame(_rows)
        display(_df.style.set_caption(f'Comparison 1 — Average Across {len(patient_idxs)} Patients'))

        def _cp_legend_label(roi):
            s     = comparison1_summary_store.get(roi)
            n_pts = len(s.get('contributing_patients', patient_idxs)) if s else 0
            return f'{_roi_display_label_9b(roi)} (n={n_pts})'

        # ── Main paper figure ────────────────────────────────────────────────
        if comparison1_summary_store:
            _sum_rois = sorted(comparison1_summary_store.keys(),
                               key=lambda r: (r == 'total_body', r))
            _n_rois   = len(_sum_rois)
            _err_lim_cp = globals().get('COMPARISON_ERROR_LIMIT_PCT', 100)

            fig_main, axes_main = plt.subplots(1, 2, figsize=(26, 10))
            ax_diff, ax_err = axes_main
            fig_main.suptitle(
                f'{len(patient_idxs)} Patient SPECT vs PBPK TAC Values',
                fontsize=26, fontweight='bold', y=1.02)

            for _roi in _sum_rois:
                s    = comparison1_summary_store[_roi]
                _t   = s['frame_hr']
                _lbl = _cp_legend_label(_roi)
                _col = roi_color(_roi)
                ax_diff.plot(_t, s['mean_diff_mbq'], lw=3.5, color=_col, marker='o', ms=7, label=_lbl)
                ax_diff.fill_between(_t,
                    s['mean_diff_mbq'] - s['std_diff_mbq'],
                    s['mean_diff_mbq'] + s['std_diff_mbq'],
                    color=_col, alpha=0.18)
                ax_err.plot(_t, s['mean_pct_err'], lw=3.5, color=_col, marker='o', ms=7, label=_lbl)
                ax_err.fill_between(_t,
                    s['mean_pct_err'] - s['std_pct_err'],
                    s['mean_pct_err'] + s['std_pct_err'],
                    color=_col, alpha=0.18)

            for _ax, _yl, _ttl in [
                    (ax_diff, 'SPECT - PBPK [MBq]',             'Mean Difference'),
                    (ax_err,  '(SPECT - PBPK)/PBPK * 100 [%]',  'Mean Error %')]:
                _ax.axhline(0, color='k', lw=2.5, ls='--', alpha=0.65)
                _ax.set_xlabel('Time (hr)', fontsize=22)
                _ax.set_ylabel(_yl, fontsize=22)
                _ax.set_title(_ttl, fontsize=24, fontweight='bold')
                _ax.spines['top'].set_visible(False)
                _ax.spines['right'].set_visible(False)
                _ax.tick_params(axis='both', labelsize=18)

            if _err_lim_cp is not None:
                ax_err.set_ylim(-float(_err_lim_cp), float(_err_lim_cp))

            _handles, _labels = ax_diff.get_legend_handles_labels()
            fig_main.legend(_handles, _labels,
                            loc='lower center', ncol=min(4, _n_rois),
                            fontsize=16, framealpha=0.92,
                            handlelength=2.5, handleheight=1.4,
                            borderpad=0.8, labelspacing=0.6,
                            bbox_to_anchor=(0.5, -0.02),
                            bbox_transform=fig_main.transFigure)
            fig_main.tight_layout(rect=[0, 0.15, 1, 1.0])
            finish_plot(fig_main, 'comparison1_paper_main', tight=False)

            # Per-ROI breakdown grids
            _n_rois_g = len(_sum_rois)
            fig_gd, axes_gd, _nrows_g, _ncols_g = make_grid(_n_rois_g, max_cols=4, pw=4.5, ph=3.8)
            fig_gd.suptitle(f'Cross-Patient Difference [SPECT - PBPK] per ROI — {len(patient_idxs)} patients', fontsize=13)
            fig_ge, axes_ge, _, _ = make_grid(_n_rois_g, max_cols=4, pw=4.5, ph=3.8)
            fig_ge.suptitle(f'Cross-Patient % Error [(SPECT - PBPK)/PBPK * 100] per ROI — {len(patient_idxs)} patients', fontsize=13)
            _err_lim_g = globals().get('COMPARISON_ERROR_LIMIT_PCT', None)
            for _j, _roi in enumerate(_sum_rois):
                s        = comparison1_summary_store[_roi]
                _t       = s['frame_hr']
                _lbl_r   = _roi_display_label_9b(_roi)
                _n_pts_g = len(s.get('contributing_patients', patient_idxs))
                _col_g   = roi_color(_roi)
                ax_d = axes_gd[_j//_ncols_g][_j%_ncols_g]
                ax_e = axes_ge[_j//_ncols_g][_j%_ncols_g]
                ax_d.plot(_t, s['mean_diff_mbq'], lw=2, color=_col_g, label='Mean')
                ax_d.fill_between(_t,
                    s['mean_diff_mbq'] - s['std_diff_mbq'],
                    s['mean_diff_mbq'] + s['std_diff_mbq'],
                    alpha=0.25, color=_col_g, label='±1 SD')
                ax_d.axhline(0, color='k', lw=0.8, ls='--', alpha=0.55)
                ax_d.set_title(f'{_lbl_r} (n={_n_pts_g})', fontsize=10)
                ax_d.set_xlabel('Time (hr)'); ax_d.set_ylabel('Diff (MBq)')
                ax_d.legend(fontsize=8)
                ax_d.spines['top'].set_visible(False); ax_d.spines['right'].set_visible(False)
                ax_e.plot(_t, s['mean_pct_err'], lw=2, color=_col_g, label='Mean')
                ax_e.fill_between(_t,
                    s['mean_pct_err'] - s['std_pct_err'],
                    s['mean_pct_err'] + s['std_pct_err'],
                    alpha=0.25, color=_col_g, label='±1 SD')
                ax_e.axhline(0, color='k', lw=0.8, ls='--', alpha=0.55)
                ax_e.set_title(f'{_lbl_r} (n={_n_pts_g})', fontsize=10)
                ax_e.set_xlabel('Time (hr)'); ax_e.set_ylabel('% Error')
                ax_e.legend(fontsize=8)
                ax_e.spines['top'].set_visible(False); ax_e.spines['right'].set_visible(False)
                if _err_lim_g is not None: ax_e.set_ylim(-float(_err_lim_g), float(_err_lim_g))
            for _j in range(_n_rois_g, _nrows_g * _ncols_g):
                axes_gd[_j//_ncols_g][_j%_ncols_g].set_visible(False)
                axes_ge[_j//_ncols_g][_j%_ncols_g].set_visible(False)
            finish_plot(fig_gd, 'comparison1_cross_patient_diff_per_roi')
            finish_plot(fig_ge, 'comparison1_cross_patient_pct_err_per_roi')

## 10 — Comparison 2: Bi-exponential Kinetics

**10a — Bi-exponential (Bateman) kinetic fit.**  
Each TAC is fitted to:

$$y(t) = A \left(e^{-k_{\text{slow}}\,t} - e^{-(k_{\text{slow}}+\sigma)\,t}\right)$$

where $y(0)=0$, $k_{\text{fast}} = k_{\text{slow}} + \sigma$ (σ > 0 enforced), and $A$ scales the peak.  
Fitted separately to PBPK data (dense) and SPECT frames.

In [ ]:
def biexp(t, A, k_slow, sigma):
    """y(t) = A*(exp(-k_slow*t) - exp(-(k_slow+sigma)*t))
    y(0)=0 always.  k_fast = k_slow + sigma  (sigma > 0 enforced by bounds)."""
    return A * (np.exp(-k_slow * t) - np.exp(-(k_slow + sigma) * t))

def downsample_xy(t, y, max_pts):
    t, y = np.asarray(t, float), np.asarray(y, float)
    if t.size <= max_pts: return t, y
    idx = np.unique(np.linspace(0, t.size - 1, max_pts).astype(int))
    return t[idx], y[idx]

def fit_biexp(t, y):
    """Fit biexp curve. Returns ([A, k_slow, sigma], RMSE).
    sigma = k_fast - k_slow > 0 is enforced by the lower bound."""
    t, y = np.asarray(t, float), np.asarray(y, float)
    mask = np.isfinite(t) & np.isfinite(y) & (y >= 0)
    t, y = t[mask], y[mask]
    if t.size < 3:
        raise ValueError(f"Need ≥3 points, got {t.size}")
    p0 = [max(float(np.max(y)) * 2.0, 1e-6), 0.05, 1.0]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", OptimizeWarning)
        popt, _ = curve_fit(biexp, t, y, p0=p0,
                            bounds=([0, 1e-6, 1e-6], [np.inf, np.inf, np.inf]),
                            maxfev=100000)
    rmse = float(np.sqrt(np.mean((y - biexp(t, *popt)) ** 2)))
    return popt, rmse   # popt = [A, k_slow, sigma]

fit_store      = {}
fit_skip_store = {}

if not globals().get("SHOW_FITTING", False):
    print("10a Fitting skipped — set SHOW_FITTING = True in Setup")
elif not spect_tacs:
    print("10a: run 8c first")
else:
    pbpk_color = plot_spec("comparison", "pbpk_color", "steelblue")
    spect_color = plot_spec("comparison", "spect_color", "orangered")
    _xlim_raw  = globals().get("PBPK_XLIM_HR", 25)
    _xlim_hr   = float(_xlim_raw[1]) if isinstance(_xlim_raw,(list,tuple)) else (float(_xlim_raw) if _xlim_raw else 25)

    for i in spect_tacs:
        if i not in pbpk_store: continue
        fit_store[i]      = {}
        fit_skip_store[i] = {}
        pbpk_t = pbpk_store[i]["time_hr"]
        t_full = float(pbpk_t.max())
        # fit curve drawn over full PBPK time; xlim clips display to _xlim_hr
        _xscale  = globals().get('FITTING_XSCALE', 'log')
        _t_start = 1e-3 if _xscale == 'log' else 0.0
        t_grid   = (np.geomspace(_t_start, t_full, 800) if _xscale == 'log'
                    else np.linspace(0.0, t_full, 800))

        available = [r for r in spect_tacs[i]
                     if r.lower() in pbpk_store[i] and spect_tacs[i][r]["sum_mbq"]]
        plot_rois = comparison_rois_for_patient(i, available)
        if not plot_rois: continue

        # ── Phase 1: fit ALL patients and store results (no display gate) ────
        for roi in plot_rois:
            y_pbpk = pbpk_store[i][roi.lower()]
            t_s = np.array(spect_tacs[i][roi]["frame_hr"])
            y_s = np.array(spect_tacs[i][roi]["sum_mbq"])
            t_s, y_s = array_after_frame_filter(t_s, y_s)
            try:
                t_fit_p, y_fit_p = downsample_xy(
                    pbpk_t, y_pbpk, globals().get("FIT_MAX_PBPK_POINTS", 2000))
                popt_p, rmse_p = fit_biexp(t_fit_p, y_fit_p)
                fit_store[i].setdefault(roi, {})["pbpk"] = popt_p
                fit_store[i][roi]["pbpk_rmse"] = rmse_p
            except Exception as e:
                fit_skip_store[i].setdefault(roi, {})["pbpk"] = str(e)
            try:
                popt_s, rmse_s = fit_biexp(t_s, y_s)
                fit_store[i].setdefault(roi, {})["spect"] = popt_s
                fit_store[i][roi]["spect_rmse"] = rmse_s
            except Exception as e:
                fit_skip_store[i].setdefault(roi, {})["spect"] = str(e)

        # ── Phase 2: plot only for should_display patients ────────────────────
        if not should_display(i): continue

        fig, axes, nrows, ncols = make_grid(len(plot_rois), max_cols=4, pw=5.8, ph=6.5)
        fig.suptitle(f"{ct_label(i)} — Bi-Exponential Fit of PBPK vs SPECT TAC")

        # ── Parameter table ───────────────────────────────────────────────────
        print(f"\n{ct_label(i)} — Bi-Exponential Fit Parameters"
              f"  [y = A·(e^(-k_slow·t) - e^(-(k_slow+sigma)·t))]")
        print(f"  {'ROI':22s} {'Src':5s} {'A (MBq)':>12s} {'k_slow (1/hr)':>14s}"
              f" {'sigma (1/hr)':>13s} {'k_fast (1/hr)':>14s} {'RMSE (MBq)':>11s}")
        print("  " + "-" * 96)

        for j, roi in enumerate(plot_rois):
            ax     = axes[j // ncols][j % ncols]
            y_pbpk = pbpk_store[i][roi.lower()]

            ax.set_title(fmt_roi_label_with_vol(roi, i), fontsize=16)
            ax.set_xlabel("Time (hr)")
            ax.set_ylabel("Activity (MBq)")
            _t_lo = 1e-2 if _xscale == "log" else 0.0
            ax.set_xlim(_t_lo, _xlim_hr)
            ax.set_xscale(_xscale)
            ax.axhline(0, color="gray", lw=1.2, ls=":", alpha=0.5)

            # ── PBPK data line ────────────────────────────────────────────────
            if _xscale == "log":
                _pos = np.where(pbpk_t > 0)[0]
                _idx = np.unique(np.geomspace(_pos[0], len(pbpk_t)-1, 800).astype(int))
                t_pp, y_pp = pbpk_t[_idx], y_pbpk[_idx]
            else:
                t_pp, y_pp = downsample_xy(pbpk_t, y_pbpk, 800)
            ax.plot(t_pp, y_pp, ls=":", lw=2.5, color=pbpk_color,
                    alpha=0.75, label="PBPK data")

            # ── PBPK fit (reuse stored params) ────────────────────────────────
            fd = fit_store[i].get(roi, {})
            popt_p = fd.get("pbpk"); rmse_p = fd.get("pbpk_rmse", float("nan"))
            if popt_p is not None:
                ax.plot(t_grid, biexp(t_grid, *popt_p),
                        lw=3, color=pbpk_color, label="PBPK fit")
                kf_p = popt_p[1] + popt_p[2]
                print(f"  {roi:22s} {'PBPK':5s} {popt_p[0]:12.3f} {popt_p[1]:14.5f}"
                      f" {popt_p[2]:13.5f} {kf_p:14.5f} {rmse_p:11.4f}")
            else:
                print(f"  {roi:22s} {'PBPK':5s} fit failed: "
                      f"{fit_skip_store[i].get(roi,{}).get('pbpk','')[:55]}")

            # ── SPECT data ────────────────────────────────────────────────────
            t_s = np.array(spect_tacs[i][roi]["frame_hr"])
            y_s = np.array(spect_tacs[i][roi]["sum_mbq"])
            t_s, y_s = array_after_frame_filter(t_s, y_s)
            ax.scatter(t_s, y_s, s=120, color=spect_color, zorder=5,
                       edgecolors="white", linewidths=0.5, label="SPECT data")

            # ── SPECT fit (reuse stored params) ───────────────────────────────
            popt_s = fd.get("spect"); rmse_s = fd.get("spect_rmse", float("nan"))
            if popt_s is not None:
                ax.plot(t_grid, biexp(t_grid, *popt_s),
                        lw=3, ls="--", color=spect_color, label="SPECT fit")
                kf_s = popt_s[1] + popt_s[2]
                print(f"  {roi:22s} {'SPECT':5s} {popt_s[0]:12.3f} {popt_s[1]:14.5f}"
                      f" {popt_s[2]:13.5f} {kf_s:14.5f} {rmse_s:11.4f}")
            else:
                print(f"  {roi:22s} {'SPECT':5s} fit failed: "
                      f"{fit_skip_store[i].get(roi,{}).get('spect','')[:55]}")

            ax.legend(fontsize=11, loc="upper right", ncol=1,
                      framealpha=0.9, handlelength=2.2, borderpad=0.6,
                      handletextpad=0.5, labelspacing=0.4)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        for j in range(len(plot_rois), nrows * ncols):
            axes[j // ncols][j % ncols].set_visible(False)
        finish_plot(fig, f"ct_{i+1}_biexp_fit")


### 10b — Average Kinetic Parameters (Bland-Altman)

In [ ]:
if not globals().get('SHOW_FITTING', False):
    print('10b skipped')
elif len(fit_store) < 2:
    print('10b: no fitted ROIs available')
else:
    all_rois     = sorted(set().union(*[set(v.keys()) for v in fit_store.values() if v]))
    patient_idxs = sorted(fit_store.keys())

    import pandas as pd
    from IPython.display import display

    _BIEXP_ORDER = ['kidney', 'liver', 'lung', 'salivary',
                    'synthetic_lesion_tumor1', 'synthetic_lesion_tumor2',
                    'synthetic_lesion_tumorrest', 'synthetic_lesion_tumor_rest']
    def _biexp_order_key(roi):
        r = roi.lower()
        for i, k in enumerate(_BIEXP_ORDER):
            if r == k or r.startswith(k): return i
        return 100 + ord(r[0]) if r else 999
    _sorted_rois = sorted(all_rois, key=_biexp_order_key)

    # ── Liver subgroup detection ─────────────────────────────────────────────
    def _liver_has_lesion(pidx):
        cfg   = configs.get(pidx, {})
        specs = (cfg.get('phase_1', {}).get('synthetic_lesions_stage', {}) or {}).get('specs') or {}
        return any(organ.lower() == 'liver' for organ in specs)

    _liver_no_pts  = [p for p in patient_idxs if not _liver_has_lesion(p)]
    _liver_yes_pts = [p for p in patient_idxs if _liver_has_lesion(p)]
    print(f'\nLiver subgroups (Comparison 2):')
    print(f'  Liver (no lesion):   n={len(_liver_no_pts)}')
    print(f'  Liver (with lesion): n={len(_liver_yes_pts)}')

    def _biexp_roi_label(roi):
        r = roi.lower()
        if r == 'liver_no_lesion':   return 'Liver (no lesion)'
        if r == 'liver_with_lesion': return 'Liver (with lesion)'
        if r.startswith('synthetic_lesion_'):
            suffix = r[len('synthetic_lesion_'):]
            if suffix in ('tumorrest', 'tumor_rest'):
                return 'Synthetic Lesion: Tumor 3'
            num = ''.join(c for c in suffix if c.isdigit())
            return f'Synthetic Lesion: Tumor {num}' if num else 'Synthetic Lesion: Tumor'
        return roi.replace('_', ' ').title()

    def _nice_table(df, caption):
        return (df.style
                .set_caption(caption)
                .set_table_styles([
                    {'selector': 'caption',
                     'props': [('font-size','15px'),('font-weight','bold'),('padding-bottom','8px')]},
                    {'selector': 'th',
                     'props': [('text-align','center'),('font-weight','bold'),
                               ('border','1px solid #bbb'),('padding','6px 12px'),
                               ('background-color','#ebebeb')]},
                    {'selector': 'td',
                     'props': [('text-align','center'),('border','1px solid #bbb'),
                               ('padding','5px 12px')]},
                    {'selector': 'tr:nth-child(even)',
                     'props': [('background-color','#f9f9f9')]},
                ])
                .hide(axis='index'))

    def _fmt_ms(m, s, prec=3):
        if not np.isfinite(m): return 'N/A'
        if np.isfinite(s): return f'{m:.{prec}f} ± {s:.{prec}f}'
        return f'{m:.{prec}f}'

    # ── Per-patient parameter differences (PBPK - SPECT), liver split ────────
    def _build_biexp_diff_row(lbl, source_roi, pts):
        dA_list, dks_list, dsig_list = [], [], []
        for pidx in pts:
            fd = fit_store.get(pidx, {}).get(source_roi)
            if not fd: continue
            pp = fd.get('pbpk'); ps = fd.get('spect')
            if pp is None or ps is None: continue
            dA_list.append(float(pp[0]) - float(ps[0]))
            dks_list.append(float(pp[1]) - float(ps[1]))
            dsig_list.append(float(pp[2]) - float(ps[2]))
        n = len(dA_list)
        if n == 0: return None
        def _ms(arr):
            arr = np.array(arr, float)
            m   = float(np.nanmean(arr))
            s   = float(np.nanstd(arr, ddof=1)) if len(arr) > 1 else np.nan
            return m, s
        mA, sA   = _ms(dA_list)
        mks, sks = _ms(dks_list)
        msi, ssi = _ms(dsig_list)
        return {
            'ROI': f'{lbl} (n={n})', 'N': n,
            'Mean Δ A [MBq]':       _fmt_ms(mA,  sA,  prec=2),
            'Mean Δ k_slow [1/hr]': _fmt_ms(mks, sks, prec=5),
            'Mean Δ σ [1/hr]': _fmt_ms(msi, ssi, prec=5),
        }

    _diff_rows = []
    for roi in _sorted_rois:
        if roi == 'liver':
            for _lpts, _llbl in [(_liver_no_pts,  'Liver (no lesion)'),
                                  (_liver_yes_pts, 'Liver (with lesion)')]:
                row = _build_biexp_diff_row(_llbl, 'liver', _lpts)
                if row: _diff_rows.append(row)
            continue
        row = _build_biexp_diff_row(_biexp_roi_label(roi), roi, patient_idxs)
        if row: _diff_rows.append(row)

    display(_nice_table(pd.DataFrame(_diff_rows),
        f'Comparison 2 — Bi-Exponential Parameter Differences (PBPK − SPECT, Mean ± SD)  [{len(patient_idxs)} patients]'))

    # ── Bland-Altman for A, K_Slow, K_Fast ───────────────────────────────────
    # remaining_body excluded; liver split into 2 groups; n<10 -> bias only.
    _ba_params = [
        ('A',      lambda pp, ps: (float(pp[0]),       float(ps[0])),       'MBq'),
        ('K_Slow', lambda pp, ps: (float(pp[1]),       float(ps[1])),       '1/hr'),
        ('K_Fast', lambda pp, ps: (float(pp[1]+pp[2]), float(ps[1]+ps[2])), '1/hr'),
    ]
    _BA_BIEXP_SKIP = {'remaining_body', 'total_body', 'background'}
    _BIEXP_BA_ORDER_EXP = ['kidney', 'liver_no_lesion', 'liver_with_lesion', 'lung', 'salivary',
                           'synthetic_lesion_tumor1', 'synthetic_lesion_tumor2',
                           'synthetic_lesion_tumorrest', 'synthetic_lesion_tumor_rest']
    def _biexp_ba_order_key(roi):
        r = roi.lower()
        for _bi, _bk in enumerate(_BIEXP_BA_ORDER_EXP):
            if r == _bk or r.startswith(_bk): return _bi
        return 100

    for pname, getter, unit in _ba_params:
        rows_ba = []
        for pidx in patient_idxs:
            _liver_grp = 'liver_with_lesion' if _liver_has_lesion(pidx) else 'liver_no_lesion'
            for roi in _sorted_rois:
                if roi.lower() in _BA_BIEXP_SKIP: continue
                if roi not in fit_store.get(pidx, {}): continue
                fd = fit_store[pidx][roi]
                pp = fd.get('pbpk'); ps = fd.get('spect')
                if pp is None or ps is None: continue
                vp, vs = getter(pp, ps)
                if not (np.isfinite(vp) and np.isfinite(vs)): continue
                _ba_roi = _liver_grp if roi == 'liver' else roi
                rows_ba.append({'roi': _ba_roi, 'patient': pidx,
                                'pbpk': vp, 'spect': vs,
                                'mean_xy': (vp+vs)/2, 'diff': vs-vp})

        if not rows_ba: print(f'{pname}: no paired data'); continue

        _ba_rois = sorted(set(row['roi'] for row in rows_ba), key=_biexp_ba_order_key)
        fig_ba, axes_ba, _nr, _nc = make_grid(len(_ba_rois), max_cols=4, pw=4.5, ph=4.0)
        fig_ba.suptitle(
            f'Comparison 2 — {pname} Bland-Altman ({unit})  [{len(patient_idxs)} patients]\n'
            f'Dashed limits shown only for n≥10; bias line dotted for n<10',
            fontsize=12)
        for _j, _roi in enumerate(_ba_rois):
            _rr   = [r for r in rows_ba if r['roi']==_roi]
            _mxy  = np.array([r['mean_xy'] for r in _rr])
            _diff = np.array([r['diff']    for r in _rr])
            _n_ba = len(_rr)
            _bias = float(np.mean(_diff))
            _sd   = float(np.std(_diff, ddof=1) if _n_ba > 1 else 0.0)
            _lo   = _bias - 1.96*_sd; _hi = _bias + 1.96*_sd
            ax_b  = axes_ba[_j//_nc][_j%_nc]
            ax_b.scatter(_mxy, _diff, s=55, color=roi_color(_roi), alpha=0.75, zorder=4)
            ax_b.annotate(f'n = {_n_ba}', xy=(0.05, 0.95), xycoords='axes fraction',
                          fontsize=9, va='top', ha='left')
            if _n_ba >= 10:
                ax_b.axhline(_bias, color='k',      lw=1.5, label=f'Bias {_bias:.3g}')
                ax_b.axhline(_hi,   color='tomato', lw=1,   ls='--', label=f'+1.96SD {_hi:.3g}')
                ax_b.axhline(_lo,   color='tomato', lw=1,   ls='--', label=f'−1.96SD {_lo:.3g}')
            else:
                ax_b.axhline(_bias, color='k', lw=1.5, ls=':', label=f'Bias {_bias:.3g}')
            ax_b.axhline(0, color='gray', lw=0.7, ls=':')
            ax_b.set_title(_biexp_roi_label(_roi), fontsize=10)
            ax_b.set_xlabel(f'Mean {pname} ({unit})')
            ax_b.set_ylabel(f'SPECT − PBPK ({unit})')
            ax_b.legend(fontsize=7)
            ax_b.spines['top'].set_visible(False); ax_b.spines['right'].set_visible(False)
        for _j in range(len(_ba_rois), _nr*_nc):
            axes_ba[_j//_nc][_j%_nc].set_visible(False)
        finish_plot(fig_ba, f'comparison2_{pname.lower().replace(" ","_")}_bland_altman')

## 11 — Comparison 3: Area Under Curve

**11a — Area Under the Curve (AUC) in MBq·hr.**  
PBPK AUC: numerical trapezoid over the full TAC up to the last SPECT frame time.  
SPECT AUC: analytic integral of the Bateman fit if converged, otherwise trapezoid over measured frames.  
% Error = (SPECT AUC − PBPK AUC) / |PBPK AUC| × 100.

In [ ]:
def spect_auc_analytic(popt, t_end_hr):
    """Bi-exponential fit AUC — kept for reference/display only. NOT used as the
    reported recovery coefficient: a single-decay biexp model misrepresents
    multi-phase organ clearance (e.g. liver has a fast + slow phase, so the fit
    collapses toward zero well before the true tail — see matched_trapezoid_auc)."""
    A, ks, sig_v = [float(v) for v in popt]
    kf = ks + sig_v
    if not all(np.isfinite([A,ks,kf])) or ks<=0 or kf<=0: return np.nan
    T = float(t_end_hr)
    return A * ((1-np.exp(-ks*T))/ks - (1-np.exp(-kf*T))/kf)

def matched_trapezoid_auc(t_hr, y_mbq):
    """Trapezoidal AUC over exactly the given (sparse, acquired) time points — no
    zero-padding, no resampling onto a denser grid. Used identically for SPECT and
    PBPK (both read off diff_store at the SAME frame times) so that whatever
    discretization error trapezoidal integration introduces is identical on both
    sides and cancels in the ratio. This is also what a real dosimetry workflow
    does: in a patient you only ever have the acquired frames."""
    t, y = np.asarray(t_hr, float), np.asarray(y_mbq, float)
    mask = np.isfinite(t) & np.isfinite(y)
    t, y = t[mask], y[mask]
    order = np.argsort(t); t, y = t[order], y[order]
    if t.size < 2: return np.nan
    return float(np.trapezoid(y, t))

auc_store = {}

if not globals().get('SHOW_AUC', False):
    print('11a AUC skipped — set SHOW_AUC = True in Setup')
elif not spect_tacs:
    print('11a: run 8c first')
elif 'diff_store' not in globals() or not diff_store:
    print('11a: run 9a first (diff_store required)')
else:
    pbpk_color  = plot_spec('comparison','pbpk_color','steelblue')
    spect_color = plot_spec('comparison','spect_color','orangered')

    for i in spect_tacs:
        if i not in pbpk_store or i not in diff_store: continue
        auc_store[i] = {}
        # diff_store[i][roi] already carries SPECT and PBPK on the identical,
        # frame-filtered time axis (and already has the Bug 2/3 corrected
        # remaining_body / total_body references) — integrate both from there.
        for roi, d in diff_store[i].items():
            t  = np.asarray(d['frame_hr'], float)
            ys = np.asarray(d['y_spect'],  float)
            yp = np.asarray(d['y_pbpk'],   float)
            if t.size < 2: continue
            auc_p = matched_trapezoid_auc(t, yp)
            auc_s = matched_trapezoid_auc(t, ys)
            if not (np.isfinite(auc_p) and auc_p > 0): continue

            fits = fit_store.get(i, {}).get(roi)
            auc_s_fit = spect_auc_analytic(fits['spect'], float(t[-1])) if fits and 'spect' in fits else np.nan

            auc_store[i][roi] = {
                'pbpk_auc': auc_p, 'spect_auc_trap': auc_s, 'spect_auc_fit': auc_s_fit,
                'spect_auc_primary': auc_s, 'spect_auc_primary_label': 'matched_trapezoid',
                't_end_hr': float(t[-1]), 'n_spect_points': int(t.size),
            }

    # Per-patient Bland-Altman for AUC
    for i, aucs in auc_store.items():
        if not aucs: continue
        rois = list(aucs.keys())
        n_r  = len(rois)
        pbpk_vals = np.array([aucs[r]['pbpk_auc'] for r in rois])
        prim_vals = np.array([aucs[r]['spect_auc_primary'] for r in rois])

        mean_xy = (pbpk_vals + prim_vals) / 2
        diff    = prim_vals - pbpk_vals

        import pandas as pd
        from IPython.display import display
        _rows = []
        for roi in rois:
            a = aucs[roi]
            e = pct(a['spect_auc_primary'], a['pbpk_auc'])
            _rows.append({
                'ROI': roi.replace('_', ' '),
                'PBPK AUC (MBq·hr)': round(float(a['pbpk_auc']), 3),
                'SPECT AUC, matched trapezoid (MBq·hr)': round(float(a['spect_auc_primary']), 3),
                'SPECT AUC, biexp fit — reference only (MBq·hr)': round(float(a['spect_auc_fit']), 3) if np.isfinite(a['spect_auc_fit']) else np.nan,
                'Diff (MBq·hr)': round(float(a['spect_auc_primary'] - a['pbpk_auc']), 3),
                '% Error': round(float(e), 1),
                't_end (hr)': round(float(a['t_end_hr']), 1),
                'N SPECT pts': int(a['n_spect_points']),
            })
        _df = pd.DataFrame(_rows)
        display(_df.style.set_caption(f'{ct_label(i)} — Comparison 3: AUC Summary (matched-trapezoid = reported RC)'))

        if not should_display(i): continue  # auc_store populated; skip plot

        # ── Per-patient AUC plot: values / diff / % error ───────────────────
        x_pos = np.arange(n_r)
        fit_vals     = np.asarray([aucs[r]['spect_auc_fit']     for r in rois], float)
        primary_vals = np.asarray([aucs[r]['spect_auc_primary'] for r in rois], float)
        primary_pct  = pct(primary_vals, pbpk_vals)

        fig_a, axes_a = plt.subplots(1, 3, figsize=(17, 5.2))
        fig_a.suptitle(f'{ct_label(i)} — Comparison 3: AUC (PBPK vs Recon SPECT, matched-trapezoid)', fontsize=13)

        # Panel 1: raw AUC values (biexp fit shown faded, for reference only — not primary)
        axes_a[0].scatter(x_pos - 0.15, pbpk_vals, s=85, color='steelblue', label='PBPK', zorder=4)
        axes_a[0].scatter(x_pos + 0.15, primary_vals, s=85, color='goldenrod', label='SPECT (matched trap.)', zorder=4)
        if np.any(np.isfinite(fit_vals)):
            axes_a[0].scatter(x_pos, fit_vals, s=45, color='orangered', alpha=0.5, marker='x',
                               label='SPECT biexp fit (reference only)', zorder=3)
        for xi, vp, vt in zip(x_pos, pbpk_vals, primary_vals):
            axes_a[0].plot([xi - 0.15, xi + 0.15], [vp, vt], color='0.65', lw=1.2, alpha=0.8)
        axes_a[0].set_ylabel('AUC (MBq·hr)'); axes_a[0].set_title('AUC values'); axes_a[0].legend(fontsize=9)

        # Panel 2: absolute diff
        diff_vals = primary_vals - pbpk_vals
        axes_a[1].scatter(x_pos, diff_vals, s=85,
            color=['tomato' if v > 0 else 'steelblue' for v in diff_vals], zorder=4)
        axes_a[1].axhline(0, color='k', lw=0.8, ls='--')
        axes_a[1].set_ylabel('SPECT − PBPK (MBq·hr)'); axes_a[1].set_title('AUC difference')

        # Panel 3: % error
        axes_a[2].scatter(x_pos, primary_pct, s=85,
            color=['tomato' if v > 0 else 'steelblue' for v in primary_pct], zorder=4)
        axes_a[2].axhline(0, color='k', lw=0.8, ls='--')
        axes_a[2].set_ylabel('% Error'); axes_a[2].set_title('AUC % error')

        tick_labels = [r.replace('_', ' ') for r in rois]
        for ax_a in axes_a:
            ax_a.set_xticks(x_pos); ax_a.set_xticklabels(tick_labels, rotation=30, ha='right')
            ax_a.spines['top'].set_visible(False); ax_a.spines['right'].set_visible(False)
        finish_plot(fig_a, f'ct_{i+1}_comparison3_auc')


### 11b — Average AUC Across Patients (Bland-Altman)

In [ ]:
if not globals().get('SHOW_AUC', False):
    print('11b skipped')
elif len(auc_store) < 2:
    print('11b: need >=2 patients')
else:
    all_rois     = sorted(set().union(*[set(v.keys()) for v in auc_store.values() if v]))
    patient_idxs = sorted(auc_store.keys())

    import pandas as pd
    from IPython.display import display

    _AUC_ORDER = ['kidney', 'liver', 'lung', 'salivary',
                  'synthetic_lesion_tumor1', 'synthetic_lesion_tumor2',
                  'synthetic_lesion_tumorrest', 'synthetic_lesion_tumor_rest']
    def _auc_order_key(roi):
        r = roi.lower()
        for i, k in enumerate(_AUC_ORDER):
            if r == k or r.startswith(k): return i
        return 100 + ord(r[0]) if r else 999
    _sorted_auc_rois = sorted(all_rois, key=_auc_order_key)

    def _auc_roi_label(roi):
        r = roi.lower()
        if r == 'liver_no_lesion':   return 'Liver (no lesion)'
        if r == 'liver_with_lesion': return 'Liver (with lesion)'
        if r.startswith('synthetic_lesion_'):
            suffix = r[len('synthetic_lesion_'):]
            if suffix in ('tumorrest', 'tumor_rest'):
                return 'Synthetic Lesion: Tumor 3'
            num = ''.join(c for c in suffix if c.isdigit())
            return f'Synthetic Lesion: Tumor {num}' if num else 'Synthetic Lesion: Tumor'
        return roi.replace('_', ' ').title()

    def _nice_auc_table(df, caption):
        return (df.style
                .set_caption(caption)
                .set_table_styles([
                    {'selector': 'caption',
                     'props': [('font-size','15px'),('font-weight','bold'),('padding-bottom','8px')]},
                    {'selector': 'th',
                     'props': [('text-align','center'),('font-weight','bold'),
                               ('border','1px solid #bbb'),('padding','6px 12px'),
                               ('background-color','#ebebeb')]},
                    {'selector': 'td',
                     'props': [('text-align','center'),('border','1px solid #bbb'),
                               ('padding','5px 12px')]},
                    {'selector': 'tr:nth-child(even)',
                     'props': [('background-color','#f9f9f9')]},
                ])
                .hide(axis='index'))

    # ── Liver subgroup detection ─────────────────────────────────────────────
    def _liver_has_lesion(pidx):
        cfg   = configs.get(pidx, {})
        specs = (cfg.get('phase_1', {}).get('synthetic_lesions_stage', {}) or {}).get('specs') or {}
        return any(organ.lower() == 'liver' for organ in specs)

    _liver_no_pts  = [p for p in patient_idxs if not _liver_has_lesion(p)]
    _liver_yes_pts = [p for p in patient_idxs if _liver_has_lesion(p)]
    print(f'\nLiver subgroups (Comparison 3):')
    print(f'  Liver (no lesion):   n={len(_liver_no_pts)}  → full BA limits shown')
    print(f'  Liver (with lesion): n={len(_liver_yes_pts)}  → bias only (n<10)')

    # Expand 'liver' into two subgroups for tables; other ROIs unchanged
    _sorted_auc_rois_exp = []
    for _r in _sorted_auc_rois:
        if _r == 'liver':
            _sorted_auc_rois_exp.extend(['liver_no_lesion', 'liver_with_lesion'])
        else:
            _sorted_auc_rois_exp.append(_r)

    # ── Per-patient AUC differences (keeps remaining_body if present) ─────────
    def _fms(m, s, prec=2):
        if not np.isfinite(m): return 'N/A'
        if np.isfinite(s): return f'{m:.{prec}f} ± {s:.{prec}f}'
        return f'{m:.{prec}f}'

    _auc_rows = []
    for roi in _sorted_auc_rois_exp:
        is_liver_sub = roi in ('liver_no_lesion', 'liver_with_lesion')
        source_roi   = 'liver' if is_liver_sub else roi
        contributing = (_liver_no_pts  if roi == 'liver_no_lesion'
                        else _liver_yes_pts if roi == 'liver_with_lesion'
                        else patient_idxs)
        lbl = _auc_roi_label(roi)
        diff_list, pct_list = [], []
        for pidx in contributing:
            if source_roi not in auc_store.get(pidx, {}): continue
            a  = auc_store[pidx][source_roi]
            vp = a['pbpk_auc']; vs = a['spect_auc_primary']
            if np.isfinite(vp) and np.isfinite(vs):
                diff_list.append(vs - vp)
                pct_list.append(pct(vs, vp))
        n = len(diff_list)
        if n == 0: continue
        mean_d = float(np.mean(diff_list))
        std_d  = float(np.std(diff_list, ddof=1)) if n > 1 else np.nan
        mean_p = float(np.mean(pct_list))
        std_p  = float(np.std(pct_list, ddof=1)) if n > 1 else np.nan
        _auc_rows.append({
            'ROI': f'{lbl} (n={n})',
            'Mean Δ AUC [MBq·hr]': _fms(mean_d, std_d),
            'Mean Δ% Error':             _fms(mean_p, std_p, prec=1),
        })
    display(_nice_auc_table(pd.DataFrame(_auc_rows),
        f'Comparison 3 — AUC Differences (SPECT − PBPK, Mean ± SD)  [{len(patient_idxs)} patients]'))

    # ── Bland-Altman for AUC ─────────────────────────────────────────────────
    # Exclude remaining_body; split liver; n<10 -> bias line only (dotted).
    _BA_AUC_SKIP = {'remaining_body', 'total_body', 'background'}
    _AUC_BA_ORDER_EXP = ['kidney', 'liver_no_lesion', 'liver_with_lesion', 'lung', 'salivary',
                         'synthetic_lesion_tumor1', 'synthetic_lesion_tumor2',
                         'synthetic_lesion_tumorrest', 'synthetic_lesion_tumor_rest']
    def _auc_ba_order_key(roi):
        r = roi.lower()
        for _bi, _bk in enumerate(_AUC_BA_ORDER_EXP):
            if r == _bk or r.startswith(_bk): return _bi
        return 100

    rows_ba = []
    for pidx in patient_idxs:
        _liver_grp = 'liver_with_lesion' if _liver_has_lesion(pidx) else 'liver_no_lesion'
        for roi in _sorted_auc_rois:
            if roi.lower() in _BA_AUC_SKIP: continue
            if roi not in auc_store.get(pidx, {}): continue
            a = auc_store[pidx][roi]
            vp = a['pbpk_auc']; vs = a['spect_auc_primary']
            if not (np.isfinite(vp) and np.isfinite(vs)): continue
            _ba_roi = _liver_grp if roi == 'liver' else roi
            rows_ba.append({'roi': _ba_roi, 'patient': pidx, 'pbpk': vp, 'spect': vs,
                            'mean_xy': (vp+vs)/2, 'diff': vs-vp})

    _ba_auc_rois = sorted(set(row['roi'] for row in rows_ba), key=_auc_ba_order_key)
    fig_auc_ba, axes_auc_ba, _nr_ba, _nc_ba = make_grid(len(_ba_auc_rois), max_cols=4, pw=4.5, ph=4.0)
    fig_auc_ba.suptitle(
        f'Comparison 3 — AUC Bland-Altman (MBq·hr)  [{len(patient_idxs)} patients]',
        fontsize=20, fontweight='bold')
    fig_auc_ba.text(0.5, 0.97,
        'Dashed limits shown only for n≥10; bias line dotted for n<10',
        ha='center', va='top', fontsize=12, color='0.45',
        transform=fig_auc_ba.transFigure)
    for _j, _roi in enumerate(_ba_auc_rois):
        _rr   = [r for r in rows_ba if r['roi']==_roi]
        _mxy  = np.array([r['mean_xy'] for r in _rr])
        _diff = np.array([r['diff']    for r in _rr])
        _n_ba = len(_rr)
        _bias = float(np.mean(_diff))
        _sd   = float(np.std(_diff, ddof=1) if _n_ba > 1 else 0.0)
        _lo   = _bias - 1.96*_sd; _hi = _bias + 1.96*_sd
        ax_b  = axes_auc_ba[_j//_nc_ba][_j%_nc_ba]
        ax_b.scatter(_mxy, _diff, s=55, color=roi_color(_roi), alpha=0.75, zorder=4)
        ax_b.annotate(f'n = {_n_ba}', xy=(0.05, 0.95), xycoords='axes fraction',
                      fontsize=9, va='top', ha='left')
        if _n_ba >= 10:
            ax_b.axhline(_bias, color='k',      lw=1.5, label=f'Bias {_bias:.3g}')
            ax_b.axhline(_hi,   color='tomato', lw=1,   ls='--', label=f'+1.96SD {_hi:.3g}')
            ax_b.axhline(_lo,   color='tomato', lw=1,   ls='--', label=f'−1.96SD {_lo:.3g}')
        else:
            ax_b.axhline(_bias, color='k', lw=1.5, ls=':', label=f'Bias {_bias:.3g}')
        ax_b.axhline(0, color='gray', lw=0.7, ls=':')
        ax_b.set_title(_auc_roi_label(_roi), fontsize=14, fontweight='bold')
        ax_b.set_xlabel('Mean AUC (MBq·hr)')
        ax_b.set_ylabel('SPECT − PBPK (MBq·hr)')
        ax_b.legend(fontsize=7)
        ax_b.spines['top'].set_visible(False); ax_b.spines['right'].set_visible(False)
    for _j in range(len(_ba_auc_rois), _nr_ba*_nc_ba):
        axes_auc_ba[_j//_nc_ba][_j%_nc_ba].set_visible(False)
    finish_plot(fig_auc_ba, 'comparison3_auc_bland_altman')

    # ── Recovery coefficient: RC_i = SPECT_AUC_i / PBPK_AUC_i ─────────────────
    # Per-patient ratio; mean ± SD(ddof=1) across patients.
    # n<5 → not reported; 5≤n<10 → reported with low-n flag.
    print('\n=== Comparison 3 — Recovery Coefficient (RC = SPECT_AUC / PBPK_AUC) ===')
    _rc_rows = []
    for roi in _sorted_auc_rois_exp:
        is_liver_sub = roi in ('liver_no_lesion', 'liver_with_lesion')
        source_roi   = 'liver' if is_liver_sub else roi
        contributing = (_liver_no_pts  if roi == 'liver_no_lesion'
                        else _liver_yes_pts if roi == 'liver_with_lesion'
                        else patient_idxs)
        lbl = _auc_roi_label(roi)
        rc_list = []
        for pidx in contributing:
            if source_roi not in auc_store.get(pidx, {}): continue
            a = auc_store[pidx][source_roi]
            vp, vs = a['pbpk_auc'], a['spect_auc_primary']
            if np.isfinite(vp) and vp > 0 and np.isfinite(vs):
                rc_list.append(float(vs) / float(vp))
        n = len(rc_list)
        if n < 5:
            _rc_rows.append({'ROI': lbl, 'n': n,
                             'Mean RC': '—', 'SD RC': '—', 'Note': 'n<5, not reported'})
            continue
        rc_arr  = np.array(rc_list)
        mean_rc = float(np.mean(rc_arr))
        sd_rc   = float(np.std(rc_arr, ddof=1)) if n > 1 else np.nan
        note    = 'low-n (n<10)' if n < 10 else ''
        _rc_rows.append({
            'ROI': lbl, 'n': n,
            'Mean RC': f'{mean_rc:.3f}',
            'SD RC':   f'{sd_rc:.3f}' if np.isfinite(sd_rc) else '—',
            'Note': note,
        })
    _rc_df = pd.DataFrame(_rc_rows)
    print(_rc_df.to_string(index=False))
    display(_nice_auc_table(_rc_df,
        'Comparison 3 — Recovery Coefficient per ROI  (RC = SPECT AUC / PBPK AUC,  mean ± SD across patients)'))

    # ── Table 7: Combined AUC summary (RC + Bland-Altman stats) ─────────────
    # Covers same ROIs as the BA figure (remaining_body excluded).
    # Bias and LoA rounded to nearest integer (MBq·hr).
    _table7_rows = []
    for roi in _ba_auc_rois:
        lbl    = _auc_roi_label(roi)
        _rr_t7 = [r for r in rows_ba if r['roi'] == roi]
        n_t7   = len(_rr_t7)
        if n_t7 == 0: continue
        _diff_t7 = np.array([r['diff'] for r in _rr_t7])
        _bias_t7 = float(np.mean(_diff_t7))
        _sd_t7   = float(np.std(_diff_t7, ddof=1)) if n_t7 > 1 else np.nan
        _lo_t7   = _bias_t7 - 1.96*_sd_t7
        _hi_t7   = _bias_t7 + 1.96*_sd_t7
        _rc = next((r for r in _rc_rows if r['ROI'] == lbl), {})
        _mean_rc = _rc.get('Mean RC', '—')
        _sd_rc   = _rc.get('SD RC',   '—')
        _rc_str  = (f'{_mean_rc} ± {_sd_rc}'
                    if _mean_rc != '—' and _sd_rc != '—' else _mean_rc)
        _table7_rows.append({
            'ROI':                        f'{lbl} (n={n_t7})',
            'Recovery Coeff (mean ± SD)': _rc_str,
            'Bias (MBq·hr)':        f'{round(_bias_t7)}',
            '±1.96 SD LoA (MBq·hr)': (
                f'[{round(_lo_t7)}, {round(_hi_t7)}]' if n_t7 >= 10
                else '(n<10)'),
        })
    print('\n=== Table 7 — AUC Comparison Summary ===')
    display(_nice_auc_table(pd.DataFrame(_table7_rows),
        f'Table 7 — AUC Comparison Summary  [{len(patient_idxs)} patients total]'))

    # ── Cross-patient AUC scatter: PBPK vs SPECT per ROI ────────────────────
    # Primary series is the matched-trapezoid AUC (== the reported RC); the biexp
    # fit is shown faded, for reference only (see auc_integration_methodology).
    all_rois_cp = sorted(set().union(*[set(v.keys()) for v in auc_store.values() if v]))
    roi_acc = {r: {'pbpk': [], 'primary': [], 'fit_ref': []}
               for r in all_rois_cp}
    for _pidx in sorted(auc_store.keys()):
        for _r in all_rois_cp:
            if _r not in auc_store.get(_pidx, {}):
                for k in roi_acc[_r]: roi_acc[_r][k].append(np.nan)
                continue
            _a = auc_store[_pidx][_r]
            roi_acc[_r]['pbpk'].append(_a['pbpk_auc'])
            roi_acc[_r]['primary'].append(_a['spect_auc_primary'])
            roi_acc[_r]['fit_ref'].append(_a['spect_auc_fit'])

    fig_auc, axes_auc, _nrows_auc, _ncols_auc = make_grid(len(all_rois_cp), max_cols=4, pw=4.5, ph=4.0)
    fig_auc.suptitle(f'Comparison 3 — PBPK vs SPECT AUC (matched-trapezoid)  [{len(auc_store)} patients]', fontsize=13)
    for _j, _roi in enumerate(all_rois_cp):
        _pbpk_v = np.array(roi_acc[_roi]['pbpk'],    float)
        _prim_v = np.array(roi_acc[_roi]['primary'], float)
        _fit_v  = np.array(roi_acc[_roi]['fit_ref'], float)
        ax_s    = axes_auc[_j//_ncols_auc][_j%_ncols_auc]
        _valid  = np.isfinite(_pbpk_v) & np.isfinite(_prim_v)
        if _valid.any():
            ax_s.scatter(_pbpk_v[_valid], _prim_v[_valid], s=60, color='goldenrod',
                         alpha=0.8, zorder=4, label='SPECT (matched trap.)')
        _valid_f = np.isfinite(_pbpk_v) & np.isfinite(_fit_v)
        if _valid_f.any():
            ax_s.scatter(_pbpk_v[_valid_f], _fit_v[_valid_f], s=35, color='orangered',
                         alpha=0.45, marker='x', zorder=3, label='SPECT biexp fit (reference only)')
        _all_v = np.concatenate([_pbpk_v[np.isfinite(_pbpk_v)], _prim_v[np.isfinite(_prim_v)]])
        if _all_v.size:
            _lim = [float(_all_v.min()), float(_all_v.max())]
            ax_s.plot(_lim, _lim, 'k--', lw=0.8, alpha=0.5, label='y = x')
        ax_s.set_title(_roi.replace('_', ' '), fontsize=10)
        ax_s.set_xlabel('PBPK AUC (MBq·hr)'); ax_s.set_ylabel('SPECT AUC (MBq·hr)')
        ax_s.legend(fontsize=7)
        ax_s.spines['top'].set_visible(False); ax_s.spines['right'].set_visible(False)
    for _j in range(len(all_rois_cp), _nrows_auc*_ncols_auc):
        axes_auc[_j//_ncols_auc][_j%_ncols_auc].set_visible(False)
    finish_plot(fig_auc, 'comparison3_auc_scatter')


### 11c — Cohort ROI Volumes vs Recovery Coefficient

In [ ]:
# ── Cohort ROI volumes alongside the matched-trapezoid recovery coefficients ────
# Reuses _sorted_auc_rois_exp / _liver_no_pts / _liver_yes_pts / _auc_roi_label /
# _liver_has_lesion / _rc_rows from cell 11b so the ROI set, liver split, and RC
# values here are IDENTICAL to what's already displayed there (no recomputation
# of RC — pulled straight from _rc_rows).
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import display

if not globals().get('SHOW_AUC', False) or '_rc_rows' not in globals():
    print('ROI volumes: run 11b first (needs _sorted_auc_rois_exp / _rc_rows)')
else:
    _rc_by_label = {r['ROI']: r for r in _rc_rows}

    def _body_outline_vol(pidx):
        """Whole-body contour volume: union of ALL non-background masks (not the
        raw image array — see Bug 3 in output_analysis_bugfixes)."""
        m = masks_recon.get(pidx, {})
        if not m: return np.nan
        outline = None
        for roi_name, mask in m.items():
            if roi_name == 'background': continue
            outline = mask.copy() if outline is None else (outline | mask)
        if outline is None: return np.nan
        return float(outline.sum()) * recon_vox_ml[pidx]

    _vol_rows = []
    for roi in _sorted_auc_rois_exp:
        is_liver_sub = roi in ('liver_no_lesion', 'liver_with_lesion')
        source_roi   = 'liver' if is_liver_sub else roi
        contributing = (_liver_no_pts  if roi == 'liver_no_lesion'
                        else _liver_yes_pts if roi == 'liver_with_lesion'
                        else patient_idxs)
        lbl = _auc_roi_label(roi)

        vols = []
        for pidx in contributing:
            # Same gating as the 11b RC loop: only patients where this ROI actually
            # entered auc_store (i.e. was simulated / had SPECT+PBPK data).
            if source_roi not in auc_store.get(pidx, {}): continue
            if source_roi == 'total_body':
                v = _body_outline_vol(pidx)
            else:
                mask = masks_recon.get(pidx, {}).get(source_roi)
                if mask is None: continue
                v = int(mask.sum()) * recon_vox_ml[pidx]
            if np.isfinite(v):
                vols.append(v)

        n = len(vols)
        rc_info = _rc_by_label.get(lbl, {})
        _vol_rows.append({
            'ROI': lbl, 'n': n,
            'Mean Vol (mL)': float(np.mean(vols)) if n else np.nan,
            'SD Vol (mL)':   float(np.std(vols, ddof=1)) if n > 1 else np.nan,
            'Min Vol (mL)':  float(np.min(vols)) if n else np.nan,
            'Max Vol (mL)':  float(np.max(vols)) if n else np.nan,
            'Mean RC': rc_info.get('Mean RC', '—'),
            'SD RC':   rc_info.get('SD RC', '—'),
            'RC Note': rc_info.get('Note', ''),
        })

    _vol_df = pd.DataFrame(_vol_rows).sort_values('Mean Vol (mL)', ascending=True).reset_index(drop=True)
    _fmt_df = _vol_df.copy()
    for c in ['Mean Vol (mL)', 'SD Vol (mL)', 'Min Vol (mL)', 'Max Vol (mL)']:
        _fmt_df[c] = _fmt_df[c].map(lambda v: f'{v:.1f}' if np.isfinite(v) else 'N/A')
    display(_fmt_df.style.set_caption(
        'Cohort ROI volumes (recon-grid masks) and matched-trapezoid RC, sorted by mean volume'))

    # ── Spearman: mean volume vs mean RC across ROIs ────────────────────────────
    # Excludes remaining_body/total_body (not organs), any ROI with n<5 (RC not
    # numerically reported in cell 11b for those), AND Liver (with lesion):
    # (1) its volume (1608.1 mL) ties Liver (no lesion) (1608.2 mL) — a near-tie on
    #     the x-axis paired with wildly different y-values (1.314 vs 0.880) distorts
    #     rank correlation; (2) its elevated RC is spill-in from an adjacent lesion,
    #     a different physical mechanism from partial-volume-vs-size — mixing it in
    #     conflates two effects into one number. See the spill-in investigation below.
    _SPEARMAN_EXCLUDE = {'Remaining Body', 'Total Body', 'Liver (with lesion)'}
    _sp_vol, _sp_rc, _sp_labels = [], [], []
    for row in _vol_rows:
        if row['ROI'] in _SPEARMAN_EXCLUDE: continue
        try:
            rc_val = float(row['Mean RC'])
        except (TypeError, ValueError):
            continue  # '—' / n<5, not reported
        if not np.isfinite(row['Mean Vol (mL)']): continue
        _sp_vol.append(row['Mean Vol (mL)']); _sp_rc.append(rc_val); _sp_labels.append(row['ROI'])

    print(f'\nSpearman correlation (mean ROI volume vs mean RC), n={len(_sp_vol)} ROIs: {_sp_labels}')
    if len(_sp_vol) >= 3:
        rho, p = spearmanr(_sp_vol, _sp_rc)
        print(f'  rho = {rho:.3f}, p = {p:.4f}  (n={len(_sp_vol)} region TYPES, not independent samples —')
        print('   the perfect rank ordering is the evidence; treat p as descriptive, not a clinical endpoint)')
    else:
        print('  too few ROIs with a reported RC to compute a rank correlation')

    # ── Sanity check 1: physiological plausibility ──────────────────────────────
    print('\n=== Sanity check: physiological plausibility of cohort mean volumes ===')
    _plaus_ranges = {
        'Kidney':          (150, 400),
        'Liver (no lesion)':   (1000, 2000),
        'Liver (with lesion)': (1000, 2000),
        'Lungs':           (1000, 3000),
        'Salivary Glands': (20, 60),
    }
    for roi_lbl, (lo, hi) in _plaus_ranges.items():
        row = next((r for r in _vol_rows if r['ROI'] == roi_lbl), None)
        if row is None or not np.isfinite(row['Mean Vol (mL)']):
            print(f'  {roi_lbl:22s} — not found'); continue
        v = row['Mean Vol (mL)']
        ok = lo <= v <= hi
        print(f'  {roi_lbl:22s} mean={v:8.1f} mL   expected [{lo},{hi}] mL   {"PASS" if ok else "FAIL"}')

    # ── Consistency check (NOT independent validation): ROI masks reconstruct the
    # whole-body contour by construction ────────────────────────────────────────
    # remaining_body is DEFINED (cell 8a) as body_outline - union(SIMIND roi_subset
    # organs + lesions), on the same integer-label array, so remaining_body + those
    # organs/lesions summing back to the body contour is arithmetic identity, not a
    # finding — it would be ~0% even if the masks were wrong in some other way. This
    # only confirms the mask bookkeeping is internally consistent (no double-count/
    # gap in cell 8a). It is NOT the same as, and does not substitute for, an
    # independent check that actually involves the reconstructed counts (e.g. an
    # AUC balance: organ deficits + remaining-body surplus vs the total-body
    # difference) — do not cite this residual in the paper as validating anything.
    print('\n=== Mask bookkeeping check (construction identity, not independent evidence) ===')
    _resid_rows = []
    for pidx in sorted(masks_recon.keys()):
        if pidx not in recon_vox_ml: continue
        vox_ml = recon_vox_ml[pidx]
        body_vol = _body_outline_vol(pidx)
        if not np.isfinite(body_vol) or body_vol <= 0: continue

        sim_rois = simind_meta_store.get(pidx, {}).get('roi_list') or \
                   configs.get(pidx, {}).get('phase_2', {}).get('simind_stage', {}).get('roi_subset', [])
        organ_rois = [str(r).lower() for r in sim_rois if str(r).lower() != 'remaining_body']

        parts_vol = 0.0
        for r in organ_rois:
            m = masks_recon[pidx].get(r)
            if m is not None:
                parts_vol += int(m.sum()) * vox_ml
        rb = masks_recon[pidx].get('remaining_body')
        if rb is not None:
            parts_vol += int(rb.sum()) * vox_ml

        resid_pct = (parts_vol - body_vol) / body_vol * 100.0
        _resid_rows.append({'patient': ct_label(pidx), 'body_vol_mL': round(body_vol, 1),
                            'parts_vol_mL': round(parts_vol, 1), 'residual_pct': round(resid_pct, 3)})

    _resid_df = pd.DataFrame(_resid_rows)
    display(_resid_df)
    _max_abs_resid = float(_resid_df.residual_pct.abs().max()) if len(_resid_df) else np.nan
    print(f'Max |residual| across {len(_resid_df)} patients: {_max_abs_resid:.3f}%')
    assert np.isfinite(_max_abs_resid) and _max_abs_resid <= 1.0, (
        f'ROI masks do not partition the whole-body contour to within 1% '
        f'(max residual {_max_abs_resid:.3f}%) — stopping rather than adjusting anything; '
        f'investigate mask construction in cell 8a before trusting the volume table above.')
    print('Bookkeeping OK (expected to be ~exact by construction — see note above; this is a')
    print('consistency check on cell 8a, not evidence about the imaging chain).')


### 11d — Liver-Lesion Spill-in Investigation

In [ ]:
# ── Liver-lesion spill-in investigation ─────────────────────────────────────
# Liver (with lesion) RC (1.314 ± 0.504, n=7) is driven by a subset of the 7
# liver-lesion patients, not all of them. Lesion volume is ruled out (8-12 mL
# across all 7, no size trend). This checks: (1) which patients actually show the
# effect, (2) whether PRESCRIBED lesion activity distinguishes them, (3) whether
# the liver's activity excess is comparable in magnitude to the lesion's own
# recovery deficit (a spill-in signature), and (4) lesion-liver mask adjacency.
import numpy as np
import pandas as pd
from scipy import ndimage
from IPython.display import display

_TUMOR_LABEL_TO_KEY = {
    'tumor1': 'synthetic_lesion_tumor1',
    'tumor2': 'synthetic_lesion_tumor2',
    'tumorrest': 'synthetic_lesion_tumorrest',
}

_rows = []
for pidx in sorted(globals().get('_liver_yes_pts', [])):
    specs = configs.get(pidx, {}).get('phase_1', {}).get('synthetic_lesions_stage', {}).get('specs', {})
    liver_spec = specs.get('liver', {})
    pbpk_label = str(liver_spec.get('pbpk_label', '')).lower()
    lesion_key = _TUMOR_LABEL_TO_KEY.get(pbpk_label)

    liver_a = auc_store.get(pidx, {}).get('liver')
    if liver_a is None or liver_a['pbpk_auc'] <= 0:
        continue
    liver_rc = liver_a['spect_auc_primary'] / liver_a['pbpk_auc']
    liver_excess_auc = liver_a['spect_auc_primary'] - liver_a['pbpk_auc']  # MBq*hr, SPECT - PBPK

    lesion_a = auc_store.get(pidx, {}).get(lesion_key) if lesion_key else None
    lesion_deficit_auc = (lesion_a['pbpk_auc'] - lesion_a['spect_auc_primary']) if lesion_a else np.nan

    lesion_mask = masks_recon.get(pidx, {}).get(lesion_key) if lesion_key else None
    liver_mask  = masks_recon.get(pidx, {}).get('liver')
    lesion_vol_mL = int(lesion_mask.sum()) * recon_vox_ml[pidx] if lesion_mask is not None else np.nan

    # Prescribed lesion activity: total cumulated (AUC) and instantaneous at 1 hr.
    lesion_pbpk_auc = lesion_a['pbpk_auc'] if lesion_a else np.nan
    if lesion_key and lesion_key in pbpk_store.get(pidx, {}):
        lesion_act_1hr = float(np.interp(1.0, pbpk_store[pidx]['time_hr'], pbpk_store[pidx][lesion_key]))
    else:
        lesion_act_1hr = np.nan
    lesion_conc_1hr = lesion_act_1hr / lesion_vol_mL if (np.isfinite(lesion_act_1hr) and lesion_vol_mL) else np.nan

    # Mask adjacency: dilate the lesion mask by 1 voxel (~1 recon voxel edge) and
    # check overlap with the liver mask (0 = not touching, since raw masks are
    # already mutually exclusive labels by construction).
    if lesion_mask is not None and liver_mask is not None:
        dilated = ndimage.binary_dilation(lesion_mask, iterations=1)
        touching_voxels = int((dilated & liver_mask).sum())
        lesion_surface_voxels = int((dilated & ~lesion_mask).sum())  # dilation shell
        adjacency_frac = touching_voxels / lesion_surface_voxels if lesion_surface_voxels else np.nan
    else:
        touching_voxels, adjacency_frac = np.nan, np.nan

    _rows.append({
        'Patient': ct_label(pidx), 'PBPK class': liver_spec.get('pbpk_label', '?'),
        'n_lesions': liver_spec.get('n_lesions'), 'radii_mm': liver_spec.get('radii_mm'),
        'Lesion Vol (mL)': round(lesion_vol_mL, 1) if np.isfinite(lesion_vol_mL) else np.nan,
        'Liver RC (matched)': round(liver_rc, 3),
        'Spill-in (RC>1)': 'yes' if liver_rc > 1.0 else 'no',
        'Liver excess AUC (MBq·hr)': round(liver_excess_auc, 1),
        'Lesion deficit AUC (MBq·hr)': round(lesion_deficit_auc, 1) if np.isfinite(lesion_deficit_auc) else np.nan,
        'Lesion PBPK AUC (MBq·hr)': round(lesion_pbpk_auc, 1) if np.isfinite(lesion_pbpk_auc) else np.nan,
        'Lesion PBPK @1h (MBq)': round(lesion_act_1hr, 2) if np.isfinite(lesion_act_1hr) else np.nan,
        'Lesion PBPK conc @1h (MBq/mL)': round(lesion_conc_1hr, 3) if np.isfinite(lesion_conc_1hr) else np.nan,
        'Lesion voxels touching liver (dilated 1vox)': touching_voxels,
        'Adjacency frac (touching/shell)': round(adjacency_frac, 3) if np.isfinite(adjacency_frac) else np.nan,
    })

_df = pd.DataFrame(_rows).sort_values('Liver RC (matched)', ascending=False).reset_index(drop=True)
display(_df)

print('\n=== Spill-in (RC>1) vs clean liver-lesion patients: prescribed lesion activity ===')
_g = _df.groupby('Spill-in (RC>1)').agg(
    n=('Patient', 'count'),
    mean_lesion_vol_mL=('Lesion Vol (mL)', 'mean'),
    mean_lesion_pbpk_auc=('Lesion PBPK AUC (MBq·hr)', 'mean'),
    mean_lesion_pbpk_1hr=('Lesion PBPK @1h (MBq)', 'mean'),
    mean_lesion_conc_1hr=('Lesion PBPK conc @1h (MBq/mL)', 'mean'),
    mean_adjacency_frac=('Adjacency frac (touching/shell)', 'mean'),
).round(3)
print(_g)

print('\n=== Liver excess AUC vs lesion deficit AUC (spill-in patients only) ===')
_spill = _df[_df['Spill-in (RC>1)'] == 'yes']
for _, r in _spill.iterrows():
    exc, defc = r['Liver excess AUC (MBq·hr)'], r['Lesion deficit AUC (MBq·hr)']
    ratio = exc / defc if (np.isfinite(defc) and defc != 0) else np.nan
    print(f"  {r['Patient']}: liver excess = {exc:+.1f} MBq·hr, lesion deficit = {defc:+.1f} MBq·hr, "
          f"excess/deficit ratio = {ratio:.2f}" if np.isfinite(ratio) else
          f"  {r['Patient']}: liver excess = {exc:+.1f} MBq·hr, lesion deficit = N/A (lesion not in auc_store)")


## 12 — CPU and RAM Profiling

In [ ]:
profiles = {}
profile_summary_store = {}

def summarize_profile(prof):
    rows = []
    cpu_count = prof.get('logical_cpu_count', 1)
    for s in prof.get('stages', []):
        # Clamp raw cpu_pct to the physical machine ceiling (cpu_count * 100) before
        # computing any stats. Values above this limit are a psutil sampling artefact
        # (sub-jiffy inflation when many short-lived processes are discovered); they
        # are not real CPU usage. See profiler.py _iter_process_tree for the fix
        # applied to future runs.
        cpu = np.clip(np.asarray(s.get('cpu_pct_samples',[]), float), 0.0, cpu_count * 100.0)
        ram = np.asarray(s.get('ram_mb_samples',[]), float)
        rows.append({
            'name': s.get('name','unknown'),
            'elapsed_s': float(s.get('elapsed_s',0.0)),
            'samples': int(cpu.size),
            'mean_cpu_capacity_pct': float(np.nanmean(cpu/cpu_count)) if cpu.size else np.nan,
            'max_cpu_capacity_pct':  float(np.nanmax(cpu/cpu_count))  if cpu.size else np.nan,
            'mean_cpu_cores': float(np.nanmean(cpu/100.0)) if cpu.size else np.nan,
            'max_cpu_cores':  float(np.nanmax(cpu/100.0))  if cpu.size else np.nan,
            'peak_ram_gb': float(np.nanmax(ram)/1024.0) if ram.size else np.nan,
            'mean_ram_gb': float(np.nanmean(ram)/1024.0) if ram.size else np.nan,
        })
    return rows


_STAGE_DISPLAY = {
    'simind': 'SIMIND', 'opengate': 'OpenGate',
    'segmentation': 'Segmentation', 'synthetic_lesions': 'Synthetic Lesions',
    'pbpk_tac': 'PBPK TAC', 'spect_postprocess': 'SPECT Postprocess',
    'dosemap_postprocess': 'Dose Map Postprocess',
}
def fmt_stage_name(name):
    return _STAGE_DISPLAY.get(name.lower(), name.replace('_',' ').title())

def ordered_stage_names(names):
    order = {n:idx for idx,n in enumerate(globals().get('STAGE_ORDER',
        ['segmentation','synthetic_lesions','pbpk_tac','simind','opengate',
         'spect_postprocess','dosemap_postprocess']))}
    return sorted(names, key=lambda n: (order.get(n,10**6), n))

_folders = _resolve_out_folders()
for i, folder in enumerate(_folders):
    hits = sorted([os.path.join(folder,f) for f in os.listdir(folder)
                   if f.startswith('profiling_CT_') and f.endswith('.json')]) if os.path.isdir(folder) else []
    if not hits:
        print(f'{ct_label(i)}: no profiling file (run with --profile)'); continue
    with open(hits[0], encoding='utf-8') as fh:
        profiles[i] = json.load(fh)
    profile_summary_store[i] = summarize_profile(profiles[i])
    cpu_count = profiles[i].get('logical_cpu_count', 1)
    stages = [s['name'] for s in profiles[i].get('stages',[])]
    total_s = profiles[i].get('pipeline_total_elapsed_s', 0)
    print(f'\n{ct_label(i)}: {len(stages)} stages  total={total_s:.1f}s  logical_CPUs={cpu_count}')
    print(f'  stages: {stages}')

if not profiles:
    print('No profiling data — re-run with --profile to enable')


### 12a — Full Pipeline CPU/RAM Timeline

In [ ]:
if not globals().get('SHOW_PROFILE', False):
    print('12a Profile skipped — set SHOW_PROFILE = True in Setup')
elif not profiles:
    print('12a: no profiling data')
else:
    for i, prof in profiles.items():
        if not should_display(i): continue  # respect PLOT_PATIENTS
        stages    = prof.get('stages', [])
        cpu_count = prof.get('logical_cpu_count', 1)
        if not stages: continue

        colors = plt.get_cmap(plot_spec('profile','cmap','tab10'))(np.linspace(0,1,len(stages)))
        fig, (ax_cpu, ax_ram) = plt.subplots(1, 2, figsize=(20, 6))
        fig.suptitle(f'{ct_label(i)} — Pipeline Timeline  (CPU 100% = {cpu_count} logical core{"s" if cpu_count!=1 else ""})')

        offset = 0.0  # hours
        for s, col in zip(stages, colors):
            elapsed_hr = float(s.get('elapsed_s', 0.0)) / 3600.0
            t   = (np.asarray(s.get('sample_times_s', []), float) + offset * 3600) / 3600.0
            cpu = np.clip(np.asarray(s.get('cpu_pct_samples', []), float), 0.0, cpu_count * 100.0) / cpu_count
            ram = np.asarray(s.get('ram_mb_samples', []), float) / 1024.0
            name = fmt_stage_name(s.get('name', 'unknown'))
            if t.size:
                ax_cpu.plot(t, cpu, lw=1.4, color=col, label=name)
                ax_ram.plot(t, ram, lw=1.4, color=col, label=name)
            ax_cpu.axvspan(offset, offset + elapsed_hr, color=col, alpha=0.07)
            ax_ram.axvspan(offset, offset + elapsed_hr, color=col, alpha=0.07)
            ax_cpu.axvline(offset, color='k', lw=0.5, alpha=0.3)
            offset += elapsed_hr
        ax_cpu.axvline(offset, color='k', lw=0.5, alpha=0.3)

        ax_cpu.set_ylabel('CPU (% total capacity)')
        ax_cpu.set_xlabel('Pipeline elapsed time (hr)')
        ax_ram.set_ylabel('RAM (GB)')
        ax_ram.set_xlabel('Pipeline elapsed time (hr)')
        ax_cpu.set_ylim(0, 100)
        ax_ram.set_ylim(bottom=0)
        for ax in (ax_cpu, ax_ram):
            ax.legend(fontsize=9, loc='upper right')
        finish_plot(fig, f'ct_{i+1}_profile_timeline')


### 12b — Cross-Patient Stage CPU/RAM (mean ± std band)

In [ ]:
_STAGE_DISPLAY = {
    'simind': 'SIMIND', 'opengate': 'OpenGate',
    'segmentation': 'Segmentation', 'synthetic_lesions': 'Synthetic Lesions',
    'pbpk_tac': 'PBPK TAC', 'spect_postprocess': 'SPECT Postprocess',
    'dosemap_postprocess': 'Dose Map Postprocess',
}
def fmt_stage_name(name):
    return _STAGE_DISPLAY.get(name.lower(), name.replace('_',' ').title())

def ordered_stage_names(names):
    order = {n:idx for idx,n in enumerate(globals().get('STAGE_ORDER',
        ['segmentation','synthetic_lesions','pbpk_tac','simind','opengate',
         'spect_postprocess','dosemap_postprocess']))}
    return sorted(names, key=lambda n: (order.get(n,10**6), n))

def profile_stage_by_name(prof, stage_name):
    matches = [s for s in prof.get('stages',[]) if s.get('name')==stage_name]
    return matches[0] if matches else None

def normalized_stage_series(stage, cpu_count, metric, grid):
    elapsed = float(stage.get('elapsed_s',0.0))
    t = np.asarray(stage.get('sample_times_s',[]),float)
    if metric=='cpu':
        raw = np.asarray(stage.get('cpu_pct_samples',[]),float)
        y = np.clip(raw, 0.0, cpu_count * 100.0) / max(float(cpu_count),1.0)
    elif metric=='ram': y = np.asarray(stage.get('ram_mb_samples',[]),float)/1024.0
    else: raise ValueError(metric)
    mask = np.isfinite(t) & np.isfinite(y)
    t, y = t[mask], y[mask]
    if y.size == 0: return np.full_like(grid, np.nan, float)
    if t.size == 0 or elapsed <= 0: return np.full_like(grid, float(np.nanmean(y)), float)
    rel = np.clip(t/elapsed, 0.0, 1.0)
    order = np.argsort(rel); rel, y = rel[order], y[order]
    ru, ui = np.unique(rel, return_index=True); yu = y[ui]
    if ru.size == 1: return np.full_like(grid, float(yu[0]), float)
    return np.interp(grid, ru, yu, left=yu[0], right=yu[-1])

if not globals().get('SHOW_PROFILE', False):
    print('12b skipped')
elif not profiles:
    print('12b: no profiling data')
else:
    phase_filter  = globals().get('PROFILE_PHASE','')
    patient_idxs  = sorted(profiles.keys())

    def get_stages(prof, pf):
        stages = prof.get('stages',[])
        return [x for x in stages if pf.lower() in x.get('name','').lower()] if pf else stages

    all_stage_names = ordered_stage_names({
        s['name'] for prof in profiles.values() for s in get_stages(prof, phase_filter)
    })

    if not all_stage_names:
        print(f'No stages match PROFILE_PHASE={phase_filter!r}')
    else:
        grid   = np.linspace(0.0, 1.0, 101)
        x_pct  = grid * 100.0
        n_pts  = len(patient_idxs)
        # One color per patient — same palette as 12a
        patient_colors = plt.get_cmap('tab10')(np.linspace(0, 1, max(n_pts, 1)))

        # ── One figure per stage: CPU left, RAM right ─────────────────────────
        for stage_name in all_stage_names:
            fig, (ax_cpu, ax_ram) = plt.subplots(1, 2, figsize=(18, 5))
            fig.suptitle(f'Stage: {fmt_stage_name(stage_name)} — All Patients',
                         fontsize=16, fontweight='bold')

            cpu_stack, ram_stack = [], []
            for pidx, col in zip(patient_idxs, patient_colors):
                prof  = profiles[pidx]
                stage = profile_stage_by_name(prof, stage_name)
                if stage is None: continue
                cpu_count = prof.get('logical_cpu_count', 1)
                lbl = ct_label(pidx)
                c_ser = normalized_stage_series(stage, cpu_count, 'cpu', grid)
                r_ser = normalized_stage_series(stage, cpu_count, 'ram', grid)
                cpu_stack.append(c_ser); ram_stack.append(r_ser)
                ax_cpu.plot(x_pct, c_ser, lw=1.2, color=col, alpha=0.55, label=lbl)
                ax_ram.plot(x_pct, r_ser, lw=1.2, color=col, alpha=0.55, label=lbl)

            # Mean ± SD band over patients
            for ax, stack, ylabel in [
                    (ax_cpu, cpu_stack, 'CPU (% total capacity)'),
                    (ax_ram, ram_stack, 'RAM (GB)')]:
                if len(stack) >= 1:
                    arr = np.array(stack, float)
                    mean = np.nanmean(arr, axis=0)
                    std  = np.nanstd(arr, axis=0, ddof=1)
                    ax.plot(x_pct, mean, lw=2.8, color='black', label='Mean', zorder=5)
                    ax.fill_between(x_pct, mean - std, mean + std,
                                    color='black', alpha=0.18, label='±1 SD', zorder=4)
                ax.set_xlabel('Stage progress (%)')
                ax.set_ylabel(ylabel)
                ax.set_ylim(0, 100) if ax is ax_cpu else ax.set_ylim(bottom=0)
                ax.spines['top'].set_visible(False)
                ax.spines['right'].set_visible(False)

            # Shared legend (patients + mean) to the right of figure
            handles, labels = ax_cpu.get_legend_handles_labels()
            fig.legend(handles, labels, loc='center right',
                       fontsize=9, framealpha=0.9,
                       bbox_to_anchor=(1.01, 0.5),
                       bbox_transform=fig.transFigure)
            fig.tight_layout(rect=[0, 0, 0.88, 1])
            finish_plot(fig, f'profile_12b_{stage_name}')


### 12c — Stage Runtime, CPU, and RAM Summary

In [ ]:
if not globals().get('SHOW_PROFILE', False):
    print('12c skipped')
elif not profile_summary_store:
    print('12c: no profiling data')
else:
    all_names = ordered_stage_names({row['name'] for rows in profile_summary_store.values() for row in rows})
    patient_idxs = sorted(profile_summary_store.keys())
    x = np.arange(len(all_names))
    metric_defs = [
        ('elapsed_s',            'Avg Runtime (hr)',        'Runtime'),
        ('mean_cpu_capacity_pct','Mean CPU (% capacity)',   'Mean CPU'),
        ('mean_ram_gb',          'Mean RAM (GB)',            'Mean RAM'),
    ]
    bar_colors = ['slateblue', 'steelblue', 'seagreen']

    # Aggregate mean, std, median per metric per stage
    agg = {m: {'mean':[],'std':[],'med':[],'n':[]} for m,_,_ in metric_defs}
    for stage_name in all_names:
        for metric, _, _ in metric_defs:
            vals = []
            for pidx in patient_idxs:
                lu = {row['name']:row for row in profile_summary_store[pidx]}
                if stage_name in lu:
                    vals.append(lu[stage_name].get(metric, np.nan))
            arr = np.asarray(vals, float)
            _val = float(np.nanmean(arr)) if arr.size else np.nan
            if metric == 'elapsed_s': _val = _val / 3600.0
            agg[metric]['mean'].append(_val)
            _std = float(np.nanstd(arr, ddof=1)) if arr.size else np.nan
            if metric == 'elapsed_s': _std = _std / 3600.0
            agg[metric]['std'].append(_std)
            _med = float(np.nanmedian(arr)) if arr.size else np.nan
            if metric == 'elapsed_s': _med = _med / 3600.0
            agg[metric]['med'].append(_med)
            agg[metric]['n'].append(int(np.sum(np.isfinite(arr))))

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), squeeze=False)
    axes = axes.ravel()
    cpu_count_title = profiles[patient_idxs[0]].get('logical_cpu_count',1) if patient_idxs else '?'
    fig.suptitle(f'Stage Runtime / CPU / RAM  (CPU 100% = {cpu_count_title} cores, n={len(patient_idxs)} patients)')

    for ax, (metric, ylabel, title), color in zip(axes, metric_defs, bar_colors):
        means = np.asarray(agg[metric]['mean'], float)
        stds  = np.asarray(agg[metric]['std'],  float)
        meds  = np.asarray(agg[metric]['med'],  float)
        ax.bar(x, means, yerr=stds, capsize=4, color=color, alpha=0.8, label='Mean ± Std')
        ax.scatter(x, meds, marker='D', color='k', s=45, zorder=5, label='Median')
        ax.set_xticks(x)
        ax.set_xticklabels([fmt_stage_name(n) for n in all_names], rotation=30, ha='right')
        ax.set_ylabel(ylabel); ax.set_title(title)
        ax.legend(fontsize=11)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        for xi, yi, si, n in zip(x, means, stds, agg[metric]['n']):
            top = (yi + si) if np.isfinite(si) else yi
            if np.isfinite(yi): ax.text(xi, top if np.isfinite(top) else yi,
                                        f'n={n}', ha='center', va='bottom', fontsize=9,
                                        color='0.25', fontweight='bold')
    finish_plot(fig, 'profile_stage_summary')

    print('\nRaw stage profile per patient:')
    print(f"  {'CT':>4s} {'stage':22s} {'elapsed_hr':>11s} {'mean_CPU%':>10s} {'mean_RAM_GB':>12s}")
    for pidx in patient_idxs:
        lu = {row['name']:row for row in profile_summary_store[pidx]}
        for sn in all_names:
            if sn not in lu: continue
            row = lu[sn]
            print(f'  {pidx+1:4d} {fmt_stage_name(sn):22s} {row["elapsed_s"]/3600:11.4f} {fmt_num(row["mean_cpu_capacity_pct"],10,1)} {fmt_num(row["mean_ram_gb"],12,3)}')


### 12d — Resolution Scaling

In [ ]:
if not globals().get('SHOW_RESOLUTION_SCALING', False):
    print('12d skipped — set SHOW_RESOLUTION_SCALING = True in Setup')
elif not profile_summary_store:
    print('12d: no profiling data')
else:
    import math as _math

    def load_stage_metadata(folder, stage_name):
        path = os.path.join(folder, 'pipeline_metadata', f'{stage_name}.json')
        return load_json(path) if os.path.exists(path) else {}

    def finite_array(values):
        arr = np.asarray(values, float)
        return arr[np.isfinite(arr)]

    def geom_mean(values):
        arr = finite_array(values); arr = arr[arr>0]
        if arr.size == 0: return np.nan
        return float(np.prod(arr)**(1.0/arr.size))

    def cube_equivalent_edge_mm(spacing_xyz_mm):
        return geom_mean(spacing_xyz_mm)

    def fmt_spacing(values):
        arr = finite_array(values)
        return ' x '.join(f'{v:.2f}' for v in arr) if arr.size else 'n/a'

    def pct_of_baseline(value, baseline):
        return value/baseline*100.0 if np.isfinite(value) and np.isfinite(baseline) and baseline>0 else np.nan

    def grid_voxels_from_size(size_values):
        vals = [int(v) for v in (size_values or []) if int(v)>0]
        return int(np.prod(vals)) if vals else np.nan

    def stage_metric_row(patient_idx, stage_name):
        lu = {row['name']:row for row in profile_summary_store.get(patient_idx,[])}
        return lu.get(stage_name, {})

    def simind_resolution_info(patient_idx, folder):
        meta = simind_meta_store.get(patient_idx) or load_stage_metadata(folder,'simind_simulation_stage')
        spacing_zyx_mm = np.asarray(meta.get('arr_px_spacing_cm',[]),float)*10.0
        spacing_xyz_mm = spacing_zyx_mm[::-1] if finite_array(spacing_zyx_mm).size==3 else spacing_zyx_mm
        shape = meta.get('arr_shape_new',[])
        grid_voxels = meta.get('total_num_voxels') or grid_voxels_from_size(shape)
        return {'spacing_mm':spacing_xyz_mm,'spacing_text':fmt_spacing(spacing_xyz_mm),
                'cube_edge_mm':cube_equivalent_edge_mm(spacing_xyz_mm),
                'grid_voxels':float(grid_voxels) if np.isfinite(float(grid_voxels) if grid_voxels else np.nan) else np.nan}

    def opengate_resolution_info(folder):
        meta = load_stage_metadata(folder,'opengate_simulation_stage')
        grid = meta.get('dose_output_grid',{})
        spacing_mm = np.asarray(grid.get('spacing_xyz_mm') or meta.get('downsampling',{}).get('xyz_spacing_mm') or [],float)
        size_xyz = grid.get('size_xyz',[])
        grid_voxels = grid_voxels_from_size(size_xyz)
        return {'spacing_mm':spacing_mm,'spacing_text':fmt_spacing(spacing_mm),
                'cube_edge_mm':cube_equivalent_edge_mm(spacing_mm),
                'grid_voxels':float(grid_voxels) if np.isfinite(float(grid_voxels) if grid_voxels else np.nan) else np.nan}

    _folders = _resolve_out_folders()
    resolution_scaling_rows = []
    for patient_idx, folder in enumerate(_folders):
        if patient_idx not in profile_summary_store: continue
        for stage_name, info in [('simind',simind_resolution_info(patient_idx,folder)),
                                  ('opengate',opengate_resolution_info(folder))]:
            metrics = stage_metric_row(patient_idx, stage_name)
            if not metrics: continue
            resolution_scaling_rows.append({
                'patient_idx': patient_idx,
                'ct_label': f'CT {patient_idx+1}',
                'stage': stage_name,
                **info,
                'elapsed_s': float(metrics.get('elapsed_s',np.nan)),
                'peak_ram_gb': float(metrics.get('peak_ram_gb',np.nan)),
                'mean_cpu_pct': float(metrics.get('mean_cpu_capacity_pct',np.nan)),
            })

    if not resolution_scaling_rows:
        print('No resolution scaling data')
    else:
        for stage_name in ('simind','opengate'):
            rows = [r for r in resolution_scaling_rows if r['stage']==stage_name]
            if not rows: continue
            edges = [r['cube_edge_mm'] for r in rows]
            times = [r['elapsed_s'] for r in rows]
            rams  = [r['peak_ram_gb'] for r in rows]

            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))
            fig.suptitle(f'{stage_name.upper()} — Resolution Scaling')
            ax1.scatter(edges, times, s=85, color='steelblue')
            for r in rows: ax1.annotate(r['ct_label'],(r['cube_edge_mm'],r['elapsed_s']),fontsize=10,xytext=(0,5),textcoords='offset points',ha='center')
            ax1.set_xlabel('Cube-equivalent voxel edge (mm)'); ax1.set_ylabel('Elapsed time (s)'); ax1.set_title('Runtime')
            ax2.scatter(edges, rams, s=85, color='orangered')
            for r in rows: ax2.annotate(r['ct_label'],(r['cube_edge_mm'],r['peak_ram_gb']),fontsize=10,xytext=(0,5),textcoords='offset points',ha='center')
            ax2.set_xlabel('Cube-equivalent voxel edge (mm)'); ax2.set_ylabel('Peak RAM (GB)'); ax2.set_title('Peak RAM')
            for ax in (ax1,ax2): ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            finish_plot(fig, f'resolution_scaling_{stage_name}')


## 13 — Bug-fix Validation

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# VALIDATION — PBPK compartment aliasing, remaining_body mask/reference, and
# total_body (whole-volume SPECT vs whole-body PBPK total)
# ═══════════════════════════════════════════════════════════════════════════
import pandas as pd
from IPython.display import display

print('='*78)
print('Bug 1 — PBPK compartment aliasing (pbpk_total sanity check)')
print('='*78)
for i in sorted(pbpk_store.keys()):
    n_raw  = len(pbpk_store[i]) - 1
    n_dist = len(PBPK_COMPARTMENTS[i])
    print(f'  {ct_label(i):8s} raw keys={n_raw:3d}  distinct compartments={n_dist:3d}  '
          f'pbpk_total(1hr)={pbpk_total(i, 1.0):8.1f} MBq')
print(f'\nMean pbpk_total(1hr) across {len(pbpk_store)} patients: '
      f'{np.mean([pbpk_total(i, 1.0) for i in pbpk_store]):.1f} MBq  (expect ~4500 MBq)')

print()
print('='*78)
print('Bug 2 — remaining_body: before (Rest TAC alone) vs after (pbpk_total - SIMIND organs)')
print('='*78)
_rows = []
for i in sorted(diff_store.keys()):
    d = diff_store[i].get('remaining_body')
    if not d or not len(d['frame_hr']): continue
    t_hr, y_s = d['frame_hr'], d['y_spect']
    y_p_old = np.interp(t_hr, pbpk_store[i]['time_hr'], pbpk_store[i]['remaining_body'])
    y_p_new = d['y_pbpk']
    for fr, ys, ypo, ypn in zip(t_hr, y_s, y_p_old, y_p_new):
        _rows.append({
            'Patient': ct_label(i), 'Frame (hr)': round(float(fr), 1),
            'SPECT (MBq)': round(float(ys), 1),
            'PBPK before (MBq)': round(float(ypo), 1), '% Err before': round(float(pct(ys, ypo)), 1),
            'PBPK after (MBq)': round(float(ypn), 1),  '% Err after': round(float(pct(ys, ypn)), 1),
        })
_df_rb = pd.DataFrame(_rows)
display(_df_rb.style.set_caption('remaining_body — before/after % error (all patients, all frames)'))
print(f"Mean % error before: {_df_rb['% Err before'].mean():+.1f}%   "
      f"after: {_df_rb['% Err after'].mean():+.1f}%")

print()
print('='*78)
print('Bug 3 — total_body: whole-volume SPECT vs whole-body PBPK total')
print('='*78)
_rows_tb = []
for i in sorted(diff_store.keys()):
    tb = diff_store[i].get('total_body')
    if not tb or not len(tb['frame_hr']): continue
    for fr, ys, yp, pe in zip(tb['frame_hr'], tb['y_spect'], tb['y_pbpk'], tb['pct_err']):
        _rows_tb.append({'patient': i, 't_hr': float(fr), 'y_spect': float(ys),
                         'y_pbpk': float(yp), 'pct_err': float(pe)})
_df_tb = pd.DataFrame(_rows_tb)
_tb_summary = _df_tb.groupby('t_hr').agg(
    n=('patient', 'count'), mean_pct_err=('pct_err', 'mean'), sd_pct_err=('pct_err', 'std')).round(2)
display(_tb_summary)
_t0, _t1 = _df_tb.t_hr.min(), _df_tb.t_hr.max()
print(f"Whole-body % error at {_t0:.0f} hr: "
      f"{_df_tb[_df_tb.t_hr==_t0].pct_err.mean():+.1f} ± {_df_tb[_df_tb.t_hr==_t0].pct_err.std(ddof=1):.1f} %"
      f"  (expect ~ -5%)")
print(f"Whole-body % error at {_t1:.0f} hr: "
      f"{_df_tb[_df_tb.t_hr==_t1].pct_err.mean():+.1f} ± {_df_tb[_df_tb.t_hr==_t1].pct_err.std(ddof=1):.1f} %"
      f"  (expect ~ +112%)")

print()
print('='*78)
print('Whole-body AUC recovery coefficient (RC = SPECT AUC / PBPK AUC, total_body)')
print('='*78)
_rc_tb = np.array([
    auc_store[i]['total_body']['spect_auc_primary'] / auc_store[i]['total_body']['pbpk_auc']
    for i in sorted(auc_store.keys())
    if 'total_body' in auc_store.get(i, {}) and auc_store[i]['total_body']['pbpk_auc'] > 0
])
print(f'RC = {_rc_tb.mean():.3f} ± {_rc_tb.std(ddof=1):.3f}  (n={len(_rc_tb)}, expect ~1.07 ± 0.12)')


## 14 — Publication Figures
Figures and tables formatted for the manuscript. This section does not modify or replace the exploratory figures above (Sections 5–13), which serve a different purpose — browsing every patient and every frame during analysis. Everything here is pulled from the stores those sections already built (`pbpk_store`, `diff_store`, `auc_store`, `masks_recon`, `profile_summary_store`, ...); nothing is recomputed with different logic and no numbers are hard-coded.

### 14a — Publication Style Module

In [ ]:
from contextlib import contextmanager

# ── Config ────────────────────────────────────────────────────────────────────
PAPER_FIG_DIR      = Path(OUTPUT_ROOT) / 'figures' / 'paper'
PAPER_DPI          = 300
PAPER_SAVE         = False          # set True to write PAPER_FORMATS to PAPER_FIG_DIR
PAPER_FORMATS      = ('png', 'pdf')
PAPER_DEMO_PATIENT = 11             # CT number (not internal index) for single-patient panels
PAPER_PCTERR_YLIM  = 150            # % error clip for Figure 3B (late-frame lesion curves blow up)

def patient_index_for_ct(ct_number):
    """Map a CT number (e.g. 11 for 'CT 11') to its internal index into OUT_FOLDERS."""
    for idx, folder in enumerate(_resolve_out_folders()):
        if ct_index_from_name(os.path.basename(folder)) == ct_number:
            return idx
    raise ValueError(f'CT {ct_number} not found among output folders')

# ── Colorblind-safe categorical palette (Okabe-Ito), one fixed color per ROI ───
# Explicit per-ROI assignment (not a cycling index) so the 10 ROIs that appear
# together in Figure 3 never collide — black is reserved exclusively for
# total_body, which every figure already draws as a distinguished thick line.
OKABE_ITO = ['#E69F00', '#56B4E9', '#009E73', '#F0E442',
             '#0072B2', '#D55E00', '#CC79A7', '#999999']

PAPER_ROI_COLORS = {
    'kidney':                     OKABE_ITO[0],
    'liver_no_lesion':            OKABE_ITO[1],
    'liver_with_lesion':          OKABE_ITO[2],
    'lungs':                      OKABE_ITO[3],
    'salivary_glands':            OKABE_ITO[4],
    'remaining_body':             OKABE_ITO[5],
    'synthetic_lesion_tumor1':    OKABE_ITO[6],
    'synthetic_lesion_tumor2':    OKABE_ITO[7],
    'synthetic_lesion_tumorrest': '#A65628',
    'total_body':                 '#000000',
}

def paper_roi_color(roi):
    return PAPER_ROI_COLORS.get(str(roi).lower(), '#999999')

# ── rcParams for every publication figure ───────────────────────────────────────
PAPER_RC = {
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 8,
    'axes.labelsize': 9,
    'axes.titlesize': 9,
    'axes.linewidth': 1.0,
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'lines.linewidth': 1.2,
    'lines.markersize': 4,
    'legend.frameon': False,
    'legend.fontsize': 7,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'savefig.dpi': PAPER_DPI,
    'savefig.bbox': 'tight',
}

@contextmanager
def paper_style():
    """rcParams context for a publication figure. Medical-image panels still need
    an explicit black facecolor set on the figure/axes after creation — this only
    governs fonts, spines, and line/legend defaults."""
    with plt.rc_context(PAPER_RC):
        yield

def panel_label(ax, letter, color='black'):
    """Bold uppercase panel letter just outside the axes' top-left corner."""
    ax.text(-0.14, 1.08, letter, transform=ax.transAxes,
            fontsize=11, fontweight='bold', color=color, va='bottom', ha='left')

def paper_tumor_label(label):
    """'Tumor1'/'Tumor2'/'TumorRest' are just numbered PyCNO slots, not distinct
    biological classes — display the third slot as 'Tumor 3' rather than 'Tumor
    Rest', which implies it means something different from the first two."""
    return label.replace('Tumor Rest', 'Tumor 3').replace('TumorRest', 'Tumor 3')

def black_figure(fig, *axes):
    """Whole-canvas black background for figures made entirely of medical-image
    panels — matches the exploratory SPECT/dose figures elsewhere in this notebook,
    instead of leaving a white gutter around each black axes."""
    fig.patch.set_facecolor('black')
    for ax in axes:
        ax.set_facecolor('black')

def add_scalebar(ax, im, label, color='white'):
    """Intensity colorbar with explicit units, required on every SPECT/dose panel.
    Defaults to white text since every panel that needs one sits on this
    notebook's black medical-image background."""
    cb = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    cb.set_label(label, fontsize=8, color=color)
    cb.ax.tick_params(labelsize=7, colors=color)
    cb.outline.set_edgecolor('0.5' if color == 'white' else 'black')
    return cb

def save_paper_fig(fig, name):
    """No-op unless PAPER_SAVE is True — this notebook never writes files by default."""
    if not PAPER_SAVE:
        return
    PAPER_FIG_DIR.mkdir(parents=True, exist_ok=True)
    for ext in PAPER_FORMATS:
        out_path = PAPER_FIG_DIR / f'{name}.{ext}'
        fig.savefig(out_path, dpi=PAPER_DPI, bbox_inches='tight')
        print('Saved:', out_path)

print(f'Publication style module loaded. Demo patient: CT {PAPER_DEMO_PATIENT} '
      f'(index {patient_index_for_ct(PAPER_DEMO_PATIENT)}).  PAPER_SAVE={PAPER_SAVE}')


### 14b — Figure 2: Framework Outputs
Four-panel overview of the output chain for one representative patient: (A) segmentation + synthetic lesion, (B) SIMIND photopeak projection, (C) SPECT, coronal MIP, across all frames, (D) total absorbed dose.

In [ ]:
import glob as _glob

_demo_i      = patient_index_for_ct(PAPER_DEMO_PATIENT)
_demo_folder = _resolve_out_folders()[_demo_i]
_demo_label  = ct_label(_demo_i)

# ── Load every image up front so panel sizes can be chosen from their true ─────
# physical aspect ratios, instead of letting mismatched aspects inside equal
# axes boxes make some panels render much smaller than others.

# A / B: CT + segmentation + synthetic lesion (same grid, same aspect)
seg_img = sitk.ReadImage(os.path.join(_demo_folder, 'digital_twin', 'digital_twin.nii.gz'))
ct_img  = sitk.ReadImage(os.path.join(_demo_folder, 'digital_twin', 'ct.nii.gz'))
ct_res  = sitk.Resample(ct_img, seg_img, sitk.Transform(), sitk.sitkLinear, 0.0)
seg_arr = sitk.GetArrayFromImage(seg_img)[::-1]
ct_arr  = sitk.GetArrayFromImage(ct_res)[::-1]
sp_x, sp_y, sp_z = seg_img.GetSpacing()
y_mid = seg_arr.shape[1] // 2
ct_sl, seg_sl = ct_arr[:, y_mid, :], seg_arr[:, y_mid, :]
aspect_ab = (ct_sl.shape[0] * sp_z) / (ct_sl.shape[1] * sp_x)

lesion_path = os.path.join(_demo_folder, 'digital_twin', 'synthetic_lesions_stage',
                            'synthetic_lesions_all_lesions_labels.nii.gz')
les_sl = None
if os.path.exists(lesion_path):
    les_img = sitk.ReadImage(lesion_path)
    les_res = sitk.Resample(les_img, seg_img, sitk.Transform(), sitk.sitkNearestNeighbor, 0.0)
    les_sl  = sitk.GetArrayFromImage(les_res)[::-1][:, y_mid, :]

# C: SIMIND projections, all 3 energy windows, single angle, earliest frame
pp_dir = os.path.join(_demo_folder, 'post_processing', 'spect_postprocess')
_window_labels = {'w1': 'Lower Scatter', 'w2': 'Photopeak', 'w3': 'Upper Scatter'}
proj_frames = {}
for window in _window_labels:
    for p in _glob.glob(os.path.join(pp_dir, f'*_tot_{window}.nii.gz')):
        stem_hr = os.path.basename(p).replace(f'_tot_{window}.nii.gz', '').rsplit('_', 1)[-1]
        try:
            proj_frames.setdefault(float(stem_hr), {})[window] = p
        except ValueError:
            continue
c_angle = 0
c_frame_hr = imgs_c = c_vmax = aspect_c = None
if proj_frames:
    c_frame_hr = min(proj_frames)
    imgs_c = {w: np.rot90(sitk.GetArrayFromImage(sitk.ReadImage(p))[c_angle])
              for w, p in proj_frames[c_frame_hr].items()}
    _pos = np.concatenate([im[im > 0].ravel() for im in imgs_c.values() if np.any(im > 0)])
    c_vmax = float(np.percentile(_pos, 99.5)) if _pos.size else 1.0
    _any = next(iter(imgs_c.values()))
    aspect_c = _any.shape[0] / _any.shape[1]  # SIMIND projection pixels are isotropic

# D: reconstructed SPECT, coronal MIP, all frames, shared scale
frame_labels = sorted(recon_imgs_sitk[_demo_i].keys(), key=float)
mips_d = {lbl: np.max(sitk.GetArrayFromImage(recon_imgs_sitk[_demo_i][lbl])[::-1], axis=1)
          for lbl in frame_labels}
_pos_d = np.concatenate([m[m > 0].ravel() for m in mips_d.values() if np.any(m > 0)])
d_vmax = float(np.percentile(_pos_d, 99.5)) if _pos_d.size else 1.0
ref_sp_d = recon_arr_ref[_demo_i][1]
aspect_d = med_aspect(ref_sp_d[2], ref_sp_d[0])

# E / F: total dose + dose uncertainty (same grid, same aspect)
dose_path = os.path.join(_demo_folder, 'post_processing', 'dosemap_postprocess_total_dose.nii.gz')
unc_path  = os.path.join(_demo_folder, 'simulations', 'opengate_simulation_dose_sum_unc.nii.gz')
dose_sl = mask_sl_ef = unc_sl = e_vmax = f_vmax = aspect_ef = None
if os.path.exists(dose_path):
    dose_img = sitk.ReadImage(dose_path)
    dose_arr = sitk.GetArrayFromImage(dose_img)[::-1]
    sp_ef = dose_img.GetSpacing()
    body_mask = dose_arr > 0
    e_vmax = float(np.percentile(dose_arr[body_mask], 99.0)) if np.any(body_mask) else 1.0
    dose_sl   = dose_arr[:, dose_arr.shape[1] // 2, :]
    mask_sl_ef = body_mask[:, body_mask.shape[1] // 2, :]
    aspect_ef  = (dose_sl.shape[0] * sp_ef[2]) / (dose_sl.shape[1] * sp_ef[0])
    if os.path.exists(unc_path):
        unc_arr = sitk.GetArrayFromImage(sitk.ReadImage(unc_path))[::-1]
        unc_sl  = unc_arr[:, unc_arr.shape[1] // 2, :]
        f_vmax  = globals().get('DOSE_UNC_VMAX_PCT', 10) / 100.0

with paper_style():
    fig = plt.figure(figsize=(8.5, 13.0))
    # Row 2 (C, D) needs roughly 3x the height of a single-image row, since D
    # packs a 3x4 grid of frames into it; C and D share that row height with
    # equal column widths, so a proportionally tall row keeps every panel's
    # apparent image size consistent instead of one shrinking inside its box.
    gs_outer = fig.add_gridspec(3, 2, height_ratios=[1.0, 2.8, 1.0], hspace=0.28, wspace=0.35)
    ax_a = fig.add_subplot(gs_outer[0, 0])
    ax_b = fig.add_subplot(gs_outer[0, 1])
    gs_c = gs_outer[1, 0].subgridspec(1, 3, wspace=0.12)
    axes_c = [fig.add_subplot(gs_c[0, k]) for k in range(3)]
    gs_d = gs_outer[1, 1].subgridspec(3, 4, hspace=0.06, wspace=0.04)
    axes_d = [fig.add_subplot(gs_d[r, c]) for r in range(3) for c in range(4)]
    ax_e = fig.add_subplot(gs_outer[2, 0])
    ax_f = fig.add_subplot(gs_outer[2, 1])
    black_figure(fig, ax_a, ax_b, ax_e, ax_f, *axes_c, *axes_d)

    # ── A: original CT ───────────────────────────────────────────────────────
    vmin_ct, vmax_ct = CT_WINDOW
    ax_a.imshow(ct_sl, cmap='gray', vmin=vmin_ct, vmax=vmax_ct, aspect=aspect_ab, interpolation='bilinear')
    ax_a.axis('off')
    ax_a.set_title('Original CT', color='white', fontsize=9)
    panel_label(ax_a, 'A', color='white')

    # ── B: segmentation + synthetic lesion ──────────────────────────────────
    ax_b.imshow(ct_sl, cmap='gray', vmin=vmin_ct, vmax=vmax_ct, aspect=aspect_ab, interpolation='bilinear')
    cmap_seg = plt.get_cmap('tab20')
    seg_rgba = cmap_seg(seg_sl.astype(float) / max(20, int(seg_arr.max())))
    seg_rgba[..., 3] = np.where(seg_sl > 0, 0.40, 0.0)
    ax_b.imshow(seg_rgba, aspect=aspect_ab, interpolation='nearest')
    if les_sl is not None and np.any(les_sl > 0):
        ax_b.contour(les_sl > 0, levels=[0.5], colors='cyan', linewidths=1.2)
    ax_b.axis('off')
    ax_b.set_title('Segmentation + Synthetic Lesion', color='white', fontsize=9)
    panel_label(ax_b, 'B', color='white')

    # ── C: SIMIND projections, 3 energy windows ─────────────────────────────
    if imgs_c is not None:
        im_c = None
        for ax, w in zip(axes_c, ('w1', 'w2', 'w3')):
            im_c = ax.imshow(imgs_c[w], cmap='hot', vmin=0, vmax=c_vmax, aspect=aspect_c, interpolation='bilinear')
            ax.axis('off')
            ax.set_title(_window_labels[w], color='white', fontsize=7.5)
        cb_c = fig.colorbar(im_c, ax=axes_c, fraction=0.025, pad=0.02, shrink=0.9)
        cb_c.set_label('Counts', color='white', fontsize=8)
        cb_c.ax.tick_params(labelsize=7, colors='white')
        cb_c.outline.set_edgecolor('0.5')
    else:
        axes_c[1].text(0.5, 0.5, 'no projection data', color='white', ha='center', va='center',
                        transform=axes_c[1].transAxes)
        for ax in axes_c:
            ax.axis('off')
    _c_bbox = axes_c[0].get_position()
    fig.text((_c_bbox.x0 + gs_outer[1, 0].get_position(fig).x1) / 2 - 0.02, _c_bbox.y1 + 0.008,
              f'SIMIND Projections — Angle {c_angle}' + (f', {c_frame_hr:g} h' if c_frame_hr else ''),
              ha='center', va='bottom', color='white', fontsize=8, fontweight='bold')
    panel_label(axes_c[0], 'C', color='white')

    # ── D: SPECT MIP, all frames, shared scale ──────────────────────────────
    norm_d = mcolors.PowerNorm(gamma=0.5, vmin=0, vmax=d_vmax)
    last_im_d = None
    for ax, lbl in zip(axes_d, frame_labels):
        last_im_d = ax.imshow(mips_d[lbl], cmap='hot', norm=norm_d, aspect=aspect_d, interpolation='bilinear')
        ax.axis('off')
        ax.text(0.03, 0.92, f'{float(lbl):g} h', transform=ax.transAxes, color='white', fontsize=6.5, va='top')
    for ax in axes_d[len(frame_labels):]:
        ax.set_visible(False)
    _d_bbox = axes_d[0].get_position()
    fig.text(0.5, _d_bbox.y1 + 0.008, 'SPECT MIP', ha='center', va='bottom',
              color='white', fontsize=8, fontweight='bold')
    panel_label(axes_d[0], 'D', color='white')
    if last_im_d is not None:
        cb_d = fig.colorbar(last_im_d, ax=axes_d, fraction=0.02, pad=0.01, shrink=0.85)
        cb_d.set_label('MBq/mL', color='white', fontsize=8)
        cb_d.ax.tick_params(labelsize=7, colors='white')
        cb_d.outline.set_edgecolor('0.5')

    # ── E: total absorbed dose ───────────────────────────────────────────────
    if dose_sl is not None:
        cmap_dose = plt.get_cmap('plasma').copy()
        cmap_dose.set_bad('black')
        im_e = ax_e.imshow(np.where(mask_sl_ef, dose_sl, np.nan), cmap=cmap_dose, vmin=0, vmax=e_vmax,
                            aspect=aspect_ef, interpolation='bilinear')
        add_scalebar(ax_e, im_e, 'Dose (Gy)')
        ax_e.set_title('Total Absorbed Dose', color='white', fontsize=9)
    else:
        ax_e.text(0.5, 0.5, 'no dose map', color='white', ha='center', va='center', transform=ax_e.transAxes)
    ax_e.axis('off')
    panel_label(ax_e, 'E', color='white')

    # ── F: total absorbed dose uncertainty ──────────────────────────────────
    if unc_sl is not None:
        cmap_unc = plt.get_cmap('RdYlGn_r').copy()
        cmap_unc.set_bad('black')
        im_f = ax_f.imshow(np.where(mask_sl_ef, unc_sl, np.nan), cmap=cmap_unc, vmin=0, vmax=f_vmax,
                            aspect=aspect_ef, interpolation='bilinear')
        cb_f = add_scalebar(ax_f, im_f, f'Relative Uncertainty (0–{int(f_vmax*100)}%)')
        cb_f.ax.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda v, _: f'{v*100:.0f}%'))
        ax_f.set_title('Total Absorbed Dose Uncertainty', color='white', fontsize=9)
    else:
        ax_f.text(0.5, 0.5, 'no uncertainty map', color='white', ha='center', va='center', transform=ax_f.transAxes)
    ax_f.axis('off')
    panel_label(ax_f, 'F', color='white')

    fig.suptitle(f'Framework Outputs — {_demo_label}', color='white', fontsize=11, fontweight='bold', y=0.995)
    save_paper_fig(fig, 'figure2_framework_outputs')
    plt.show()

print(f'Figure 2 demo patient: {_demo_label}')
print(f'  Panel C window/level (SIMIND counts, shared across windows): (0, {c_vmax})' if c_vmax else '  Panel C: no data')
print(f'  Panel C angle index: {c_angle}, frame: {c_frame_hr} h' if c_frame_hr else '')
print(f'  Panel D window/level (SPECT, MBq/mL), shared across all {len(frame_labels)} frames: (0, {d_vmax})')
print(f'  Panel D frame times shown (h): {[float(l) for l in frame_labels]}')
print(f'  Panel E window/level (dose, Gy): (0, {e_vmax})' if e_vmax else '  Panel E: no data')
print(f'  Panel F window/level (uncertainty, %): (0, {f_vmax*100 if f_vmax else None})' if f_vmax else '  Panel F: no data')


### 14c — Figure 3: Cohort Activity Recovery
Mean difference and mean percent error (SPECT vs. PBPK) across the cohort, using the corrected `total_body` and `remaining_body` definitions from `diff_store`.

In [ ]:
if 'comparison1_summary_store' not in globals() or not comparison1_summary_store:
    print('Figure 3: run 9b first (needs comparison1_summary_store)')
else:
    _rois_f3 = sorted(comparison1_summary_store.keys(), key=lambda r: (r != 'total_body', r))

    with paper_style():
        fig, (ax_diff, ax_err) = plt.subplots(1, 2, figsize=(7.0, 3.2))

        for roi in _rois_f3:
            s = comparison1_summary_store[roi]
            is_total = roi == 'total_body'
            col = paper_roi_color(roi)
            lw = 2.2 if is_total else 1.2
            col = 'black' if is_total else col
            label = _roi_display_label_9b(roi) if '_roi_display_label_9b' in globals() else roi.replace('_', ' ').title()
            label = paper_tumor_label(label)

            ax_diff.plot(s['frame_hr'], s['mean_diff_mbq'], color=col, lw=lw, label=label, zorder=5 if is_total else 3)
            ax_diff.fill_between(s['frame_hr'], s['mean_diff_mbq'] - s['std_diff_mbq'],
                                  s['mean_diff_mbq'] + s['std_diff_mbq'], color=col, alpha=0.15, linewidth=0)

            ax_err.plot(s['frame_hr'], s['mean_pct_err'], color=col, lw=lw, label=label, zorder=5 if is_total else 3)
            ax_err.fill_between(s['frame_hr'], s['mean_pct_err'] - s['std_pct_err'],
                                 s['mean_pct_err'] + s['std_pct_err'], color=col, alpha=0.15, linewidth=0)

        for ax, ylabel, title in [(ax_diff, 'SPECT − PBPK (MBq)', 'Mean Difference'),
                                   (ax_err, '% Error', 'Mean Percent Error')]:
            ax.axhline(0, color='0.3', lw=0.8, ls='--')
            ax.set_xlabel('Time (hr)')
            ax.set_ylabel(ylabel)
            ax.set_title(title)

        ax_err.set_ylim(-PAPER_PCTERR_YLIM, PAPER_PCTERR_YLIM)
        panel_label(ax_diff, 'A')
        panel_label(ax_err, 'B')

        handles, labels = ax_diff.get_legend_handles_labels()
        fig.legend(handles, labels, loc='lower center', ncol=min(5, len(labels)), fontsize=6.5,
                   bbox_to_anchor=(0.5, -0.12), bbox_transform=fig.transFigure)
        fig.suptitle('Cohort activity recovery: total_body (whole SPECT volume) and '
                      'remaining_body (pbpk_total − SIMIND organs) use the corrected definitions',
                      fontsize=7, y=1.03, color='0.3')
        fig.tight_layout(rect=[0, 0.08, 1, 1])
        save_paper_fig(fig, 'figure3_cohort_activity_recovery')
        plt.show()

    print(f'Figure 3: {len(_rois_f3)} ROI(s) plotted, Total Body highlighted with a thicker black line.')
    print(f'Panel B (% error) clipped to ±{PAPER_PCTERR_YLIM}% — late-frame lesion curves exceed this range '
          f'(low absolute activity inflates % error); unclipped values remain in comparison1_summary_store.')


### 14d — Figure 4: Recovery Coefficient vs. Region Volume
Recovery falls off with region size across three orders of magnitude — the partial-volume-effect signature and the headline quantitative result of the paper.

In [ ]:
import pandas as pd
from IPython.display import display

# Trend regions: real organs with n>=5, spanning three orders of magnitude in volume.
PAPER_FIG4_INCLUDE = [
    'Synthetic Lesion: Tumor 3',
    'Synthetic Lesion: Tumor 1',
    'Salivary Glands',
    'Kidney',
    'Liver (no lesion)',
    'Lungs',
]
# Shown as their own reference markers, deliberately excluded from the trend/stats/fit.
PAPER_FIG4_REFERENCE = ['Remaining Body', 'Total Body']
PAPER_FIG4_EXCLUDE_REASONS = {
    'Liver (with lesion)': 'volume ties with Liver (no lesion) (~1608 mL both); its RC>1 is a '
                            'spill-in effect from the adjacent lesion (see 11d) — a different '
                            'mechanism from partial-volume-vs-size, would distort the trend',
    'Synthetic Lesion: Tumor 2': 'n=2, not reportable (11b requires n>=5 to report a mean RC)',
    'Remaining Body': 'not an organ — shown as its own reference marker instead',
    'Total Body': 'not an organ — shown as its own reference marker instead',
}

if '_vol_rows' not in globals():
    print('Figure 4: run 11c first (needs _vol_rows)')
else:
    _vol_by_roi = {r['ROI']: r for r in _vol_rows}
    # paper_roi_color() keys on the internal snake_case ROI name (e.g. 'liver_no_lesion'),
    # but _vol_rows is keyed on the formatted display label (e.g. 'Liver (no lesion)') —
    # this map recovers the internal name so colors actually match the shared ROI palette.
    _label_to_internal = {_auc_roi_label(r): r for r in _sorted_auc_rois_exp}

    def _fig4_row(label):
        row = _vol_by_roi.get(label)
        if row is None:
            raise KeyError(f'Figure 4: {label!r} not found in _vol_rows')
        return row

    # ── Table — every candidate region, whether it's in the trend, and why not ──
    _table_rows = []
    for label in PAPER_FIG4_INCLUDE + PAPER_FIG4_REFERENCE + list(PAPER_FIG4_EXCLUDE_REASONS):
        row = _fig4_row(label)
        status = ('trend' if label in PAPER_FIG4_INCLUDE else
                  'reference' if label in PAPER_FIG4_REFERENCE else 'excluded')
        _table_rows.append({
            'Region': paper_tumor_label(label).replace('Synthetic Lesion: ', ''),
            'n': row['n'],
            'Volume (mL)': f"{row['Mean Vol (mL)']:.1f} ± {row['SD Vol (mL)']:.1f}",
            'RC': row['Mean RC'],
            'Status': status,
            'Note': PAPER_FIG4_EXCLUDE_REASONS.get(label, ''),
        })
    PAPER_TABLE_FIG4 = pd.DataFrame(_table_rows)
    print('=== Figure 4 data — every candidate region, step by step ===')
    print(PAPER_TABLE_FIG4.to_string(index=False))
    display(PAPER_TABLE_FIG4.style.set_caption(
        'Figure 4 data: trend regions, reference markers (not in trend), and exclusions'))

    _fig4_pts = []
    for label in PAPER_FIG4_INCLUDE:
        row = _fig4_row(label)
        try:
            mean_rc, sd_rc = float(row['Mean RC']), float(row['SD RC'])
        except (TypeError, ValueError) as e:
            raise ValueError(f'Figure 4: {label!r} has no reported RC ({row["Mean RC"]!r}) — cannot plot') from e
        _fig4_pts.append({'label': label, 'internal': _label_to_internal[label], 'n': row['n'],
                           'vol_mean': row['Mean Vol (mL)'], 'vol_sd': row['SD Vol (mL)'],
                           'rc_mean': mean_rc, 'rc_sd': sd_rc})
    _fig4_df = pd.DataFrame(_fig4_pts).sort_values('vol_mean').reset_index(drop=True)

    _ref_pts = {}
    for label in PAPER_FIG4_REFERENCE:
        row = _fig4_row(label)
        _ref_pts[label] = (row['Mean Vol (mL)'], float(row['Mean RC']))

    # ── Monotonicity: stop rather than plot a trend that isn't actually there ──
    _rc_by_vol = _fig4_df['rc_mean'].to_numpy()
    if not np.all(np.diff(_rc_by_vol) > 0):
        raise AssertionError(
            f'Figure 4: recovery coefficient does not increase monotonically with volume — '
            f'{list(zip(_fig4_df.label, _fig4_df.vol_mean.round(1), _fig4_df.rc_mean))}. '
            f'Stopping rather than plotting a trend that is not there.')

    # ── Spearman rho + EXACT permutation p-value (n=6 -> 720 orderings; scipy's ─
    # asymptotic p underflows at rho=1). No fitted curve is drawn: extrapolated, a
    # log-linear fit predicts RC > 1 (physically impossible — recovery cannot
    # exceed the source), and the whole-body point already falls below where such
    # a fit would put it. The monotonic ranking is the result; the line adds
    # nothing and invites a reader to over-interpret an extrapolation. ─────────
    from itertools import permutations
    from scipy.stats import spearmanr

    rho_obs, _ = spearmanr(_fig4_df['vol_mean'], _fig4_df['rc_mean'])
    n_perm = n_extreme = 0
    for perm in permutations(_fig4_df['rc_mean'].to_numpy()):
        rho_p, _ = spearmanr(_fig4_df['vol_mean'], np.array(perm))
        n_perm += 1
        n_extreme += abs(rho_p) >= abs(rho_obs) - 1e-12
    p_exact = n_extreme / n_perm

    print(f'\nSpearman rho (volume vs RC, n={len(_fig4_df)} regions): {rho_obs:.3f}')
    print(f'Exact permutation p-value (brute force over all {n_perm} orderings): {p_exact:.6f}')

    # Hand-placed label offsets — Tumor 3 / Tumor 1 / Salivary Glands cluster
    # tightly at 10-30 mL and need leader lines pulling the text clear of the
    # cluster rather than crowding it.
    _fig4_label_pos = {
        'Synthetic Lesion: Tumor 3': dict(xytext=(-30, -22), ha='right',  va='top',    leader=True),
        'Synthetic Lesion: Tumor 1': dict(xytext=(26, -26),  ha='left',   va='top',    leader=True),
        'Salivary Glands':           dict(xytext=(8, 24),    ha='left',   va='bottom', leader=True),
        'Kidney':                    dict(xytext=(0, 10),    ha='center', va='bottom', leader=False),
        'Liver (no lesion)':         dict(xytext=(-9, 9),    ha='right',  va='bottom', leader=False),
        'Lungs':                     dict(xytext=(10, -6),   ha='left',   va='top',    leader=False),
    }

    with paper_style():
        fig, ax = plt.subplots(figsize=(3.9, 3.7))
        for _, row in _fig4_df.iterrows():
            col = paper_roi_color(row.internal)
            ax.errorbar(row.vol_mean, row.rc_mean, xerr=row.vol_sd, yerr=row.rc_sd,
                        fmt='o', ms=5, color=col, ecolor=col, elinewidth=1.0, capsize=2, zorder=4)
            pos = _fig4_label_pos[row.label]
            arrowprops = dict(arrowstyle='-', color='0.4', lw=0.6, shrinkA=0, shrinkB=3) if pos['leader'] else None
            ax.annotate(paper_tumor_label(row.label).replace('Synthetic Lesion: ', ''),
                        xy=(row.vol_mean, row.rc_mean), xytext=pos['xytext'], textcoords='offset points',
                        fontsize=6.5, ha=pos['ha'], va=pos['va'], arrowprops=arrowprops)

        _wb_vol, _wb_rc = _ref_pts['Total Body']
        _rb_vol, _rb_rc = _ref_pts['Remaining Body']
        ax.scatter([_wb_vol], [_wb_rc], marker='*', s=140, color='black', zorder=5, label='Whole body')
        ax.annotate('whole body', xy=(_wb_vol, _wb_rc), xytext=(0, -11),
                    textcoords='offset points', fontsize=6.5, ha='center', va='top')
        ax.scatter([_rb_vol], [_rb_rc], marker='D', s=55, color='0.35', zorder=5, label='Remaining body')
        ax.annotate('remaining body', xy=(_rb_vol, _rb_rc), xytext=(0, 9),
                    textcoords='offset points', fontsize=6.5, ha='center', va='bottom')

        ax.axhline(1.0, color='k', lw=0.8, ls=':', zorder=1)
        ax.set_xscale('log')
        ax.set_xlabel('Cohort mean region volume (mL)')
        ax.set_ylabel('Recovery coefficient (matched-trapezoid)')
        ax.text(0.98, (1.0 - ax.get_ylim()[0]) / (ax.get_ylim()[1] - ax.get_ylim()[0]) + 0.015,
                'complete recovery', transform=ax.transAxes, fontsize=6, ha='right', va='bottom', color='0.4')
        ax.legend(loc='lower right', fontsize=6.5)
        fig.tight_layout()
        save_paper_fig(fig, 'figure4_rc_vs_volume')
        plt.show()

    print(f'\nFigure 4: {len(_fig4_df)} trend regions, spanning '
          f'{_fig4_df.vol_mean.min():.1f}–{_fig4_df.vol_mean.max():.1f} mL '
          f'({_fig4_df.vol_mean.max()/_fig4_df.vol_mean.min():.0f}x). '
          f'Reference markers (not in trend): remaining body ({_rb_vol:.0f} mL, RC={_rb_rc:.3f}), '
          f'whole body ({_wb_vol:.0f} mL, RC={_wb_rc:.3f}).')


### 14e — Figure 5: Computational Cost
Execution timeline for the demo patient alongside mean per-stage cost across the cohort.

In [ ]:
if not profiles or not profile_summary_store:
    print('Figure 5: no profiling data (run pipeline with --profile)')
else:
    _demo_i = patient_index_for_ct(PAPER_DEMO_PATIENT)
    _demo_prof = profiles.get(_demo_i)
    if _demo_prof is None:
        _demo_i = sorted(profiles.keys())[0]
        _demo_prof = profiles[_demo_i]
        print(f'Figure 5: no profiling data for CT {PAPER_DEMO_PATIENT}, using {ct_label(_demo_i)} instead')

    _stage_names = ordered_stage_names({row['name'] for rows in profile_summary_store.values() for row in rows})
    _metric_defs = [('elapsed_s', 'Runtime (hr)', 3600.0), ('mean_cpu_capacity_pct', 'CPU (% capacity)', 1.0),
                    ('mean_ram_gb', 'RAM (GB)', 1.0)]
    _stage_abbrev = {'Segmentation': 'Segm.', 'Synthetic Lesions': 'Syn. Les.', 'PBPK TAC': 'PBPK',
                      'SIMIND': 'SIMIND', 'OpenGate': 'OpenGate', 'SPECT Postprocess': 'SPECT PP',
                      'Dose Map Postprocess': 'Dose PP'}

    def _thin(n, max_pts=120):
        """Plot at most max_pts samples per stage — raw per-second sampling over a
        multi-hour stage otherwise renders as a solid block, not a readable line."""
        return np.arange(n) if n <= max_pts else np.unique(np.linspace(0, n - 1, max_pts).astype(int))

    with paper_style():
        fig = plt.figure(figsize=(11.7, 3.4))
        gs = fig.add_gridspec(1, 2, width_ratios=[1.4, 1.8], wspace=0.4)
        ax_a = fig.add_subplot(gs[0, 0])
        gs_b = gs[0, 1].subgridspec(1, 3, wspace=0.85)
        axes_b = [fig.add_subplot(gs_b[0, k]) for k in range(3)]

        # ── A: demo-patient execution timeline (wall-clock, CPU, RAM) ──────────
        cpu_count = _demo_prof.get('logical_cpu_count', 1)
        ax_ram = ax_a.twinx()
        offset = 0.0
        for s in _demo_prof.get('stages', []):
            elapsed_hr = float(s.get('elapsed_s', 0.0)) / 3600.0
            t   = (np.asarray(s.get('sample_times_s', []), float) + offset * 3600) / 3600.0
            cpu = np.clip(np.asarray(s.get('cpu_pct_samples', []), float), 0.0, cpu_count * 100.0) / cpu_count
            ram = np.asarray(s.get('ram_mb_samples', []), float) / 1024.0
            idx = _thin(t.size)
            if idx.size:
                ax_a.plot(t[idx], cpu[idx], color=OKABE_ITO[0], lw=0.8, alpha=0.9)
                ax_ram.plot(t[idx], ram[idx], color=OKABE_ITO[4], lw=0.8, ls='--', alpha=0.9)
            ax_a.axvline(offset, color='0.6', lw=0.5)
            offset += elapsed_hr
        ax_a.axvline(offset, color='0.6', lw=0.5)
        ax_a.set_ylim(0, 100)
        ax_a.set_xlabel('Pipeline elapsed time (hr)')
        ax_a.set_ylabel('CPU (% capacity)', color=OKABE_ITO[0])
        ax_a.tick_params(axis='y', labelcolor=OKABE_ITO[0])
        ax_ram.set_ylabel('RAM (GB)', color=OKABE_ITO[4])
        ax_ram.tick_params(axis='y', labelcolor=OKABE_ITO[4])
        ax_a.set_title(f'{ct_label(_demo_i)} Execution Timeline', fontsize=8)
        panel_label(ax_a, 'A')

        # ── B: mean per-stage cost across the cohort, mean ± SD ────────────────
        x = np.arange(len(_stage_names))
        for ax, (metric, ylabel, div) in zip(axes_b, _metric_defs):
            means, stds = [], []
            for stage in _stage_names:
                vals = [row[metric] / div for rows in profile_summary_store.values()
                        for row in rows if row['name'] == stage]
                vals = np.asarray(vals, float)
                means.append(np.nanmean(vals) if vals.size else np.nan)
                stds.append(np.nanstd(vals, ddof=1) if vals.size > 1 else np.nan)
            ax.bar(x, means, yerr=stds, capsize=2, color=OKABE_ITO[1], alpha=0.85)
            ax.set_xticks(x)
            ax.set_xticklabels([_stage_abbrev.get(fmt_stage_name(n), fmt_stage_name(n)) for n in _stage_names],
                                rotation=90, fontsize=5.5)
            ax.set_ylabel(ylabel, fontsize=7)
        axes_b[1].set_title(f'Mean Per-Stage Cost, {len(profile_summary_store)} Patients', fontsize=8)
        panel_label(axes_b[0], 'B')

        save_paper_fig(fig, 'figure5_computational_cost')
        plt.show()

    print(f'Figure 5 demo patient: {ct_label(_demo_i)}, '
          f'total runtime {_demo_prof.get("pipeline_total_elapsed_s", np.nan)/3600:.3f} hr, '
          f'{cpu_count} logical CPUs.')
    print(f'Cohort mean per-stage cost computed over {len(profile_summary_store)} patients, '
          f'stages: {[fmt_stage_name(s) for s in _stage_names]}')


### 14f — Cross-Reference Table
Every number that appears in the manuscript, printed here as the authoritative source of truth for writing the text — and cross-checked against the values already established in Sections 11 and 13.

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option('display.width', 120)

# ── Table A — per-ROI summary (manuscript Table 1) ──────────────────────────────
_table_a_rows = []
for roi in _sorted_auc_rois_exp:
    is_liver_sub = roi in ('liver_no_lesion', 'liver_with_lesion')
    source_roi   = 'liver' if is_liver_sub else roi
    contributing = (_liver_no_pts if roi == 'liver_no_lesion'
                    else _liver_yes_pts if roi == 'liver_with_lesion'
                    else patient_idxs)
    lbl = _auc_roi_label(roi)

    vols, rcs, diffs = [], [], []
    for pidx in contributing:
        a = auc_store.get(pidx, {}).get(source_roi)
        if a is None or not (np.isfinite(a['pbpk_auc']) and a['pbpk_auc'] > 0): continue
        rcs.append(a['spect_auc_primary'] / a['pbpk_auc'])
        diffs.append(a['spect_auc_primary'] - a['pbpk_auc'])
        vol = (_body_outline_vol(pidx) if source_roi == 'total_body'
               else (int(m.sum()) * recon_vox_ml[pidx] if (m := masks_recon.get(pidx, {}).get(source_roi)) is not None else np.nan))
        if np.isfinite(vol): vols.append(vol)

    n = len(rcs)
    bias = float(np.mean(diffs)) if diffs else np.nan
    sd_bias = float(np.std(diffs, ddof=1)) if len(diffs) > 1 else np.nan
    loa_lo, loa_hi = (bias - 1.96 * sd_bias, bias + 1.96 * sd_bias) if np.isfinite(sd_bias) else (np.nan, np.nan)

    _table_a_rows.append({
        'ROI': paper_tumor_label(lbl), 'n': n,
        'Volume (mL)': f'{np.mean(vols):.1f} ± {np.std(vols, ddof=1):.1f}' if len(vols) > 1 else 'N/A',
        'Recovery coefficient': f'{np.mean(rcs):.3f} ± {np.std(rcs, ddof=1):.3f}' if n >= 5 else '— (n<5)',
        'BA bias (MBq·hr)': f'{bias:.1f}' if np.isfinite(bias) else 'N/A',
        'BA LoA lower': f'{loa_lo:.1f}' if (np.isfinite(loa_lo) and n >= 10) else '(n<10)',
        'BA LoA upper': f'{loa_hi:.1f}' if (np.isfinite(loa_hi) and n >= 10) else '(n<10)',
    })

PAPER_TABLE_A = pd.DataFrame(_table_a_rows)
print('=== Table A — Per-ROI Summary (manuscript Table 1) ===')
print(PAPER_TABLE_A.to_string(index=False))
display(PAPER_TABLE_A.style.set_caption('Table A — Per-ROI Summary'))

# ── Table B — whole-body activity recovery by frame ─────────────────────────────
_tb_rows = []
for i in sorted(diff_store.keys()):
    tb = diff_store[i].get('total_body')
    if not tb or not len(tb['frame_hr']): continue
    for fr, pe in zip(tb['frame_hr'], tb['pct_err']):
        _tb_rows.append({'t_hr': float(fr), 'pct_err': float(pe)})
_tb_df = pd.DataFrame(_tb_rows)
_tb_summary = _tb_df.groupby('t_hr').agg(
    n=('pct_err', 'count'), mean_pct_err=('pct_err', 'mean'), sd_pct_err=('pct_err', 'std')).round(1)

_wb_rc_vals = [auc_store[i]['total_body']['spect_auc_primary'] / auc_store[i]['total_body']['pbpk_auc']
               for i in auc_store if 'total_body' in auc_store[i] and auc_store[i]['total_body']['pbpk_auc'] > 0]
_wb_rc_mean, _wb_rc_sd = float(np.mean(_wb_rc_vals)), float(np.std(_wb_rc_vals, ddof=1))

PAPER_TABLE_B = _tb_summary.reset_index().rename(columns={
    't_hr': 'Time (hr)', 'mean_pct_err': 'Mean % error', 'sd_pct_err': 'SD % error', 'n': 'n'})
print('\n=== Table B — Whole-Body Activity Recovery by Frame ===')
print(PAPER_TABLE_B.to_string(index=False))
print(f'Whole-body cumulated-activity recovery coefficient: {_wb_rc_mean:.3f} ± {_wb_rc_sd:.3f}  (n={len(_wb_rc_vals)})')
display(PAPER_TABLE_B.style.set_caption('Table B — Whole-Body Activity Recovery by Frame'))

# ── Table C — per-stage computational cost (manuscript Table 2) ─────────────────
_stage_names_c = ordered_stage_names({row['name'] for rows in profile_summary_store.values() for row in rows})
_table_c_rows = []
for stage in _stage_names_c:
    rt  = np.array([row['elapsed_s'] / 3600.0 for rows in profile_summary_store.values() for row in rows if row['name'] == stage])
    cpu = np.array([row['mean_cpu_capacity_pct'] for rows in profile_summary_store.values() for row in rows if row['name'] == stage])
    ram = np.array([row['peak_ram_gb'] for rows in profile_summary_store.values() for row in rows if row['name'] == stage])
    _table_c_rows.append({
        'Stage': fmt_stage_name(stage), 'n': rt.size,
        'Runtime (hr)': f'{rt.mean():.3f} ± {rt.std(ddof=1):.3f}' if rt.size > 1 else 'N/A',
        'CPU (%)': f'{cpu.mean():.1f} ± {cpu.std(ddof=1):.1f}' if cpu.size > 1 else 'N/A',
        'Peak RAM (GB)': f'{ram.mean():.2f} ± {ram.std(ddof=1):.2f}' if ram.size > 1 else 'N/A',
    })
PAPER_TABLE_C = pd.DataFrame(_table_c_rows)
print('\n=== Table C — Per-Stage Computational Cost (manuscript Table 2) ===')
print(PAPER_TABLE_C.to_string(index=False))
display(PAPER_TABLE_C.style.set_caption('Table C — Per-Stage Computational Cost'))

# ── Table D — cohort characteristics ─────────────────────────────────────────────
_n_patients   = len(pbpk_store)
_n_lesion_pts = sum(1 for i in configs if (configs[i].get('phase_1', {})
                    .get('synthetic_lesions_stage', {}).get('specs') or {}))
_n_liver_lesion_pts = len(_liver_yes_pts)
_frames_hr = sorted({float(f) for d in diff_store.values() for r in d.values() for f in r['frame_hr']})
_pbpk_1hr = np.array([pbpk_total(i, 1.0) for i in pbpk_store])

PAPER_TABLE_D = pd.DataFrame([{
    'n patients': _n_patients,
    'n with synthetic lesions': _n_lesion_pts,
    'n with liver lesions': _n_liver_lesion_pts,
    'Imaging time points (hr)': ', '.join(f'{t:g}' for t in _frames_hr),
    'PBPK whole-body activity @1hr (MBq)': f'{_pbpk_1hr.mean():.1f} ± {_pbpk_1hr.std(ddof=1):.1f}',
}])
print('\n=== Table D — Cohort Characteristics ===')
print(PAPER_TABLE_D.to_string(index=False))
display(PAPER_TABLE_D.style.set_caption('Table D — Cohort Characteristics'))

# ── Cross-check against the manuscript's stated target values ───────────────────
# Every value below was independently re-derived from auc_store/diff_store in this
# cell; if any of them drift, something upstream changed and the manuscript text
# would silently go stale — so this asserts rather than reports quietly.
_targets = {
    'Kidney': (0.758, 0.038), 'Lungs': (0.905, 0.044), 'Liver (no lesion)': (0.880, 0.015),
    'Salivary Glands': (0.539, 0.116), 'Remaining Body': (1.132, 0.090), 'Total Body': (0.993, 0.022),
}
_rc_lookup = {r['ROI']: r['Recovery coefficient'] for r in _table_a_rows}
for roi, (t_mean, t_sd) in _targets.items():
    val = _rc_lookup.get(roi, '')
    m = re.match(r'([\d.]+) ± ([\d.]+)', val)
    if not m:
        raise AssertionError(f'Table A / target check: {roi!r} has no numeric RC ({val!r}) to compare against target.')
    got_mean, got_sd = float(m.group(1)), float(m.group(2))
    if abs(got_mean - t_mean) > 0.005 or abs(got_sd - t_sd) > 0.005:
        raise AssertionError(
            f'Table A / target check: {roi} RC = {got_mean:.3f} ± {got_sd:.3f}, expected {t_mean:.3f} ± {t_sd:.3f}. '
            f'Stopping rather than adjusting anything — investigate upstream before trusting this table.')

_err_1hr = PAPER_TABLE_B.loc[PAPER_TABLE_B['Time (hr)'] == PAPER_TABLE_B['Time (hr)'].min(), 'Mean % error'].iloc[0]
_err_168hr = PAPER_TABLE_B.loc[PAPER_TABLE_B['Time (hr)'] == PAPER_TABLE_B['Time (hr)'].max(), 'Mean % error'].iloc[0]
assert abs(_err_1hr - (-6.7)) <= 0.5, f'Whole-body % error @1hr = {_err_1hr}, expected -6.7'
assert abs(_err_168hr - 27.1) <= 0.5, f'Whole-body % error @168hr = {_err_168hr}, expected +27.1'

print('\nAll values cross-checked against the manuscript target numbers — no drift detected.')
